## Faster R-CNN · inference experiments

### Experiment 1: Soft-NMS

We evaluate the trained Faster R-CNN `lr_scheduler` model again, replacing standard NMS with Gaussian Soft-NMS at the final detection stage.

The goal is to check whether Soft-NMS improves COCO mAP@[.50:.95], especially AP_small, on the validation folds.

In [ ]:
%pip -q install pycocotools plotly

In [ ]:
import os, io, sys, json, time, math, base64, hashlib, types
from concurrent.futures import ThreadPoolExecutor
from contextlib import redirect_stdout
from pathlib import Path
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from PIL import Image
import plotly.graph_objects as go
import plotly.io as pio
from plotly.subplots import make_subplots
import torchvision
from torchvision.ops import boxes as box_ops, box_iou
from torchvision.models.detection import fasterrcnn_resnet50_fpn, FasterRCNN_ResNet50_FPN_Weights
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from pycocotools.coco import COCO
from pycocotools.cocoeval import COCOeval
from IPython.display import display

In [ ]:
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
pio.renderers.default = 'colab' if IN_COLAB else 'plotly_mimetype+notebook_connected'

ROOT = Path(os.environ.get('UCU_HW2_ROOT', '/content/drive/MyDrive/ucu_hw2' if IN_COLAB else Path('../..').resolve()))
DATA = ROOT / 'datasets/multiple-object-video-detection-hw-2-ucu-ai-2026'
FOLDS_CSV = ROOT / 'shared/sequence_folds.csv'
RUNS = ROOT / 'faster_rcnn/lr_scheduler'
OUT = ROOT / 'faster_rcnn/inference_experiments/soft_nms'

In [ ]:
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
AMP = DEVICE.type == 'cuda'
MAX_DETS = 100
EVAL_BATCH = 4
SIGMAS = [0.3, 0.5, 0.7]
EXTRA_SIGMAS = [0.1, 0.2]
ALL_SIGMAS = sorted(SIGMAS + EXTRA_SIGMAS)
SOFT_MIN_SCORE = 1e-3
SPEED_IMAGES, SPEED_WARMUP = 60, 10
SEED = 0

In [ ]:
for path in [DATA / 'train/annotations_coco.json', FOLDS_CSV, RUNS]:
    assert path.exists(), f'Missing {path}: check ROOT and DATA'
assert OUT.resolve() != RUNS.resolve()
OUT.mkdir(parents=True, exist_ok=True)
print('torch:', torch.__version__, '| torchvision:', torchvision.__version__, '| device:', DEVICE,
      torch.cuda.get_device_name() if DEVICE.type == 'cuda' else '')

torch: 2.11.0+cu128 | torchvision: 0.26.0+cu128 | device: cuda NVIDIA GeForce RTX 5070


### Dataset and checkpoint loading
For each fold, we use the same validation frames from `sequence_folds.csv` and process them the same way as in `faster_rcnn.ipynb`.

In [ ]:
with open(DATA / 'train/annotations_coco.json') as f:
    coco_train = json.load(f)
folds = pd.read_csv(FOLDS_CSV)
CATS = {int(category['id']): category['name'] for category in coco_train['categories']}
CAT_IDS = sorted(CATS)
LABEL_TO_CAT = {i + 1: cat_id for i, cat_id in enumerate(CAT_IDS)}
N_CLASSES = len(CATS) + 1
FOLDS = sorted(int(fold) for fold in folds.Fold.unique())
FOLDS_SHA = hashlib.sha256(FOLDS_CSV.read_bytes()).hexdigest()[:12]
KEYS = ['mAP', 'AP50', 'AP75', 'APs', 'APm', 'APl', 'AR']
assert folds.groupby('video').Fold.nunique().max() == 1, 'a video is split across folds'

In [ ]:
def fold_images(fold):
    ids = set(folds.loc[folds.Fold == fold, 'id'].astype(int))
    return [image for image in coco_train['images'] if int(image['id']) in ids]

def make_gt(coco, ids):
    ids = set(map(int, ids))
    anns = []
    for annotation in coco['annotations']:
        if int(annotation['image_id']) not in ids:
            continue
        x, y, w, h = map(float, annotation['bbox'])
        if w <= 0 or h <= 0:
            continue
        item = dict(annotation)
        item['area'] = float(annotation.get('area', w*h))
        item['iscrowd'] = int(annotation.get('iscrowd', 0))
        anns.append(item)
    for j, annotation in enumerate(anns):
        annotation['id'] = j + 1
    gt = COCO()
    gt.dataset = dict(info={}, licenses=[],
                      images=[image for image in coco['images'] if int(image['id']) in ids],
                      annotations=anns, categories=coco['categories'])
    with redirect_stdout(io.StringIO()):
        gt.createIndex()
    return gt

In [ ]:
def coco_metrics(preds, gt):
    detections = []
    for image_id, pred in preds:
        for box, score, label in zip(pred['boxes'].tolist(), pred['scores'].tolist(),
                                     pred['labels'].tolist()):
            x1, y1, x2, y2 = box
            if x2 <= x1 or y2 <= y1:
                continue
            detections.append(dict(image_id=image_id, category_id=LABEL_TO_CAT[int(label)],
                                   bbox=[x1, y1, x2-x1, y2-y1], score=float(score)))
    with redirect_stdout(io.StringIO()):
        evaluation = COCOeval(gt, gt.loadRes(detections), 'bbox')
        evaluation.params.imgIds = sorted(gt.getImgIds())
        evaluation.params.maxDets = [1, 10, MAX_DETS]
        evaluation.evaluate()
        evaluation.accumulate()
        evaluation.summarize()
    precision = evaluation.eval['precision'][:, :, :, 0, -1]
    stats = evaluation.stats
    return dict(mAP=float(precision[precision > -1].mean()), AP50=float(stats[1]), AP75=float(stats[2]),
                APs=float(stats[3]),
                APm=float(stats[4]), APl=float(stats[5]), AR=float(stats[8]))

In [ ]:
def cfg_id(run_config):
    stable = {k: value for k, value in run_config.items() if k != 'save_every'}
    return hashlib.sha256(json.dumps(stable, sort_keys=True).encode()).hexdigest()

def build_model(run_config):
    model = fasterrcnn_resnet50_fpn(
        weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT,
        trainable_backbone_layers=run_config['trainable_layers'],
        min_size=run_config['min_size'], max_size=run_config['max_size'],
        rpn_pre_nms_top_n_train=run_config['rpn_pre_train'],
        rpn_post_nms_top_n_train=run_config['rpn_post_train'],
        rpn_pre_nms_top_n_test=run_config['rpn_pre_test'],
        rpn_post_nms_top_n_test=run_config['rpn_post_test'],
        rpn_batch_size_per_image=run_config['rpn_batch'],
        box_batch_size_per_image=run_config['roi_batch'], box_positive_fraction=run_config['roi_pos_frac'],
        box_detections_per_img=run_config['det_per_img'], box_score_thresh=run_config['score_thr'],
        box_nms_thresh=run_config['nms_thr'])
    model.roi_heads.box_predictor = FastRCNNPredictor(model.roi_heads.box_predictor.cls_score.in_features,
                                                      N_CLASSES)
    return model

In [ ]:
def load_model(fold):
    folder = RUNS / f'fold{fold}'
    run_config = json.loads((folder / 'config.json').read_text())
    checkpoint = torch.load(folder / 'best.pth', map_location='cpu', weights_only=False)
    assert (checkpoint['cfg_id'] == cfg_id(run_config) and run_config['fold'] == fold
            and run_config['folds'] == FOLDS_SHA)
    model = build_model(run_config)
    model.load_state_dict(checkpoint['model'])
    return model.to(DEVICE).eval(), run_config, checkpoint

def sha256(path):
    digest = hashlib.sha256()
    with open(path, 'rb') as file:
        for block in iter(lambda: file.read(1 << 24), b''):
            digest.update(block)
    return digest.hexdigest()

In [ ]:
stored = {k: json.loads((RUNS / f'fold{k}/result.json').read_text()) for k in FOLDS}
checkpoints = pd.DataFrame([dict(fold=k, val_frames=len(fold_images(k)), best_epoch=result['best_epoch'],
                                 stored_mAP=result['metrics']['mAP'], stored_APs=result['metrics']['APs'],
                                 det_per_img=result['cfg']['det_per_img'],
                                 score_thr=result['cfg']['score_thr'],
                                 nms_thr=result['cfg']['nms_thr'],
                                 rpn_post_test=result['cfg']['rpn_post_test'])
                            for k, result in stored.items()]).set_index('fold')
assert (checkpoints.val_frames == [stored[k]['val_frames'] for k in FOLDS]).all()
assert checkpoints[['det_per_img', 'score_thr', 'nms_thr', 'rpn_post_test']].nunique().max() == 1
INFERENCE = {k: stored[0]['cfg'][k] for k in ['min_size', 'max_size', 'rpn_pre_test', 'rpn_post_test',
                                              'det_per_img', 'score_thr', 'nms_thr']}
print('fold file:', FOLDS_SHA, '| classes:', N_CLASSES - 1, '| inference settings:', INFERENCE)
display(checkpoints.round(4))

fold file: 2f1ef1c758e5 | classes: 8 | inference settings: {'min_size': 720, 'max_size': 1280, 'rpn_pre_test': 1000, 'rpn_post_test': 1000, 'det_per_img': 500, 'score_thr': 0.01, 'nms_thr': 0.5}


,val_frames,best_epoch,stored_mAP,stored_APs,det_per_img,score_thr,nms_thr,rpn_post_test
fold,,,,,,,,
0,981,5,0.2377,0.1030,500,0.01,0.5,1000
1,979,5,0.2208,0.0951,500,0.01,0.5,1000
2,963,4,0.2380,0.0876,500,0.01,0.5,1000
3,974,4,0.2015,0.0412,500,0.01,0.5,1000
4,971,4,0.1571,0.0516,500,0.01,0.5,1000


### Baseline inference


In [ ]:
def read_images(images):
    arrays = []
    for image in images:
        with Image.open(DATA / 'train' / image['file_name']) as picture:
            arrays.append(np.array(picture.convert('RGB')))
    return arrays

def to_tensor(rgb):
    return torch.from_numpy(rgb).permute(2, 0, 1).contiguous().float().div_(255)

def batches(images, batch_size=EVAL_BATCH):
    chunks = [images[i:i + batch_size] for i in range(0, len(images), batch_size)]
    with ThreadPoolExecutor(1) as pool:
        pending = pool.submit(read_images, chunks[0])
        for k, chunk in enumerate(chunks):
            arrays = pending.result()
            if k + 1 < len(chunks):
                pending = pool.submit(read_images, chunks[k + 1])
            yield chunk, [to_tensor(array) for array in arrays]

In [ ]:
@torch.inference_mode()
def predict(model, images):
    model.eval()
    rows = []
    for chunk, image_tensors in batches(images):
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            preds = model([image.to(DEVICE, non_blocking=True) for image in image_tensors])
        rows += [(int(image['id']), {k: value.cpu() for k, value in pred.items()})
                 for image, pred in zip(chunk, preds)]
    return rows

def same_predictions(preds_a, preds_b):
    return len(preds_a) == len(preds_b) and all(
        id_a == id_b and all(torch.equal(pred_a[k], pred_b[k]) for k in ('boxes', 'scores', 'labels'))
                                    for (id_a, pred_a), (id_b, pred_b) in zip(preds_a, preds_b))

### 4. Soft-NMS implementation

**Gaussian Soft-NMS**

Standard NMS removes boxes of the same class when their IoU is above 0.5, while Soft-NMS reduces their scores instead using the formula:

$$s_i \leftarrow s_i \cdot \exp\left(-\frac{\mathrm{IoU}(M, b_i)^2}{\sigma}\right)$$

For this experiment, we only change the final NMS and keep the rest of the model unchanged. We also check whether standard NMS gives exactly the same results as the original model.

To avoid running the model separately for each σ value, we save the boxes before NMS and use them for all tests, so each fold only needs to be processed once.

In [ ]:
def postprocess_with(final_nms):
    def postprocess_detections(self, class_logits, box_regression, proposals, image_shapes):
        device = class_logits.device
        num_classes = class_logits.shape[-1]
        boxes_per_image = [boxes_in_image.shape[0] for boxes_in_image in proposals]
        pred_boxes = self.box_coder.decode(box_regression, proposals)
        pred_scores = F.softmax(class_logits, -1)
        all_boxes, all_scores, all_labels = [], [], []
        for boxes, scores, image_shape in zip(pred_boxes.split(boxes_per_image, 0),
                                              pred_scores.split(boxes_per_image, 0), image_shapes):
            boxes = box_ops.clip_boxes_to_image(boxes, image_shape)
            labels = torch.arange(num_classes, device=device).view(1, -1).expand_as(scores)
            boxes, scores, labels = boxes[:, 1:], scores[:, 1:], labels[:, 1:]
            boxes, scores, labels = boxes.reshape(-1, 4), scores.reshape(-1), labels.reshape(-1)
            inds = torch.where(scores > self.score_thresh)[0]
            boxes, scores, labels = boxes[inds], scores[inds], labels[inds]
            keep = box_ops.remove_small_boxes(boxes, min_size=1e-2)
            boxes, scores, labels = boxes[keep], scores[keep], labels[keep]
            keep, scores = final_nms(boxes, scores, labels, self)
            all_boxes.append(boxes[keep])
            all_scores.append(scores)
            all_labels.append(labels[keep])
        return all_boxes, all_scores, all_labels
    return postprocess_detections

In [ ]:
def use_final_nms(model, final_nms=None):
    heads = model.roi_heads
    if final_nms is None:
        heads.__dict__.pop('postprocess_detections', None)
    else:
        heads.postprocess_detections = types.MethodType(postprocess_with(final_nms), heads)

def standard_nms(boxes, scores, labels, heads):
    keep = box_ops.batched_nms(boxes, scores, labels, heads.nms_thresh)[:heads.detections_per_img]
    return keep, scores[keep]

In [ ]:
def class_iou(boxes, labels):
    boxes = boxes.float()
    return box_iou(boxes, boxes) * (labels[:, None] == labels[None, :])

def soft_nms(boxes, scores, labels, sigma, max_dets, iou=None):
    if iou is None:
        iou = class_iou(boxes, labels)
    decay = torch.exp(-iou.square() / sigma)
    current = scores.float().clone()
    n_steps = min(len(current), max_dets)
    keep = torch.empty(n_steps, dtype=torch.long, device=current.device)
    final = torch.empty(n_steps, device=current.device)
    for i in range(n_steps):
        j = current.argmax()
        keep[i], final[i] = j, current[j]
        current *= decay[j]
        current[j] = -1.
    valid = final >= SOFT_MIN_SCORE
    return keep[valid], final[valid]

In [ ]:
def soft_final_nms(sigma):
    return lambda boxes, scores, labels, heads: soft_nms(boxes, scores, labels, sigma,
                                                         heads.detections_per_img)

def capture(store):
    def keep_all(boxes, scores, labels, heads):
        store.append((boxes.cpu(), scores.cpu(), labels.cpu()))
        return torch.arange(len(scores), device=scores.device), scores
    return keep_all

### Five-fold Evaluation

For each fold, we use its checkpoint `lr_scheduler/best.pth` and the corresponding validation frames to compare the results of standard NMS and Soft-NMS.

First, we evaluate the baseline and compare its metrics with the saved `result.json`, then we save the boxes before NMS and check whether they produce exactly the same results as the original model.

Next, we test Soft-NMS with different σ values on the same boxes using autocast, and verify the implementation by comparing the results with Soft-NMS applied directly inside the model.

We measure the speed of each variant on the same 60 frames with batch size 1 and autocast, including the full forward pass and NMS.

In [ ]:
SETTINGS = dict(experiment='lr_scheduler', method='gaussian_soft_nms', sigmas=SIGMAS,
                soft_min_score=SOFT_MIN_SCORE,
                coco_max_dets=MAX_DETS, eval_batch=EVAL_BATCH, amp=AMP, speed_images=SPEED_IMAGES,
                speed_warmup=SPEED_WARMUP, seed=SEED, folds_sha=FOLDS_SHA, inference=INFERENCE)
SETTINGS_ID = hashlib.sha256(json.dumps(SETTINGS, sort_keys=True).encode()).hexdigest()[:16]

def save_json(obj, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = path.with_suffix('.tmp')
    tmp_path.write_text(json.dumps(obj, indent=2))
    os.replace(tmp_path, path)

In [ ]:
def postprocess_cached(candidates, outputs, heads, sigmas=SIGMAS):
    baseline_rows, softs = [], {sigma: [] for sigma in sigmas}
    with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=AMP):
        for (image_id, output), (boxes, scores, labels) in zip(outputs, candidates):
            boxes, scores, labels = boxes.to(DEVICE), scores.to(DEVICE), labels.to(DEVICE)
            variants = [(baseline_rows, standard_nms(boxes, scores, labels, heads))]
            iou = class_iou(boxes, labels)
            variants += [(softs[sigma], soft_nms(boxes, scores, labels, sigma, heads.detections_per_img, iou))
                         for sigma in sigmas]
            for rows, (keep, new_scores) in variants:
                keep = keep.cpu()
                rows.append((image_id, dict(boxes=output['boxes'][keep], scores=new_scores.cpu(),
                                            labels=output['labels'][keep])))
    return baseline_rows, softs

In [ ]:
@torch.inference_mode()
def measure_speed(model, images):
    times = []
    for i, image in enumerate(images):
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        start = time.perf_counter()
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            model([image])
        if DEVICE.type == 'cuda':
            torch.cuda.synchronize()
        if i >= SPEED_WARMUP:
            times.append(time.perf_counter() - start)
    return float(np.mean(times) * 1000)

In [ ]:
def evaluate_fold(fold):
    started = time.time()
    model, run_config, checkpoint = load_model(fold)
    images = fold_images(fold)
    gt = make_gt(coco_train, [image['id'] for image in images])

    baseline = predict(model, images)
    candidates = []
    use_final_nms(model, capture(candidates))
    outputs = predict(model, images)
    use_final_nms(model)
    cached_base, softs = postprocess_cached(candidates, outputs, model.roi_heads)
    assert same_predictions(baseline, cached_base), 'patched post-processing differs from TorchVision'
    for sigma in SIGMAS:
        use_final_nms(model, soft_final_nms(sigma))
        assert same_predictions(predict(model, images[:EVAL_BATCH]), softs[sigma][:EVAL_BATCH])
    use_final_nms(model)

    metrics = {'baseline': coco_metrics(baseline, gt)}
    metrics.update({f'soft_nms_s{sigma}': coco_metrics(softs[sigma], gt) for sigma in SIGMAS})

    rng = np.random.default_rng(SEED + fold)
    picks = [images[i] for i in rng.choice(len(images), SPEED_IMAGES + SPEED_WARMUP, replace=False)]
    speed_images = [to_tensor(array).to(DEVICE) for array in read_images(picks)]
    speed = {'baseline': measure_speed(model, speed_images)}
    for sigma in SIGMAS:
        use_final_nms(model, soft_final_nms(sigma))
        speed[f'soft_nms_s{sigma}'] = measure_speed(model, speed_images)
    use_final_nms(model)

    n_candidates = np.array([len(candidate[1]) for candidate in candidates])
    n_kept = {name: float(np.mean([len(pred['scores']) for _, pred in rows]))
              for name, rows in [('baseline', baseline)] + [(f'soft_nms_s{sigma}', softs[sigma])
                                                            for sigma in SIGMAS]}
    rows = [dict(fold=fold, variant=name, method=name.split('_s')[0],
                 sigma=None if name == 'baseline' else float(name.split('_s')[1]),
                 **variant_metrics, ms_per_img=speed[name], fps=1000 / speed[name],
                 detections_per_img=n_kept[name])
            for name, variant_metrics in metrics.items()]
    result = dict(fold=fold, settings_id=SETTINGS_ID,
                  checkpoint=str((RUNS / f'fold{fold}/best.pth').relative_to(ROOT)),
                  checkpoint_sha256=sha256(RUNS / f'fold{fold}/best.pth'), best_epoch=checkpoint['epoch'],
                  val_frames=len(images),
                  candidates_per_img=dict(mean=float(n_candidates.mean()), max=int(n_candidates.max())),
                  stored_baseline={k: stored[fold]['metrics'][k] for k in KEYS},
                  baseline_max_abs_diff_vs_stored=max(abs(metrics['baseline'][k] - stored[fold]['metrics'][k])
                                                      for k in KEYS),
                  checks=dict(patched_standard_nms_identical=True,
                              forward_soft_nms_identical_first_batch=True),
                  rows=rows, runtime_s=time.time() - started)
    del model, candidates, outputs, speed_images
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return result

In [ ]:
fold_results = {}
for fold in FOLDS:
    path = OUT / 'folds' / f'fold{fold}.json'
    if path.exists():
        old = json.loads(path.read_text())
        if (old['settings_id'] == SETTINGS_ID
            and old['checkpoint_sha256'] == sha256(RUNS / f'fold{fold}/best.pth')):
            fold_results[fold] = old
    if fold not in fold_results:
        fold_results[fold] = evaluate_fold(fold)
        save_json(fold_results[fold], path)
    record = fold_results[fold]
    m = {row['variant']: row for row in record['rows']}
    print(f"fold {fold}: baseline mAP {m['baseline']['mAP']:.4f} (stored "
          f"{record['stored_baseline']['mAP']:.4f}) | " +
          ' | '.join(f"σ={sigma}: {m[f'soft_nms_s{sigma}']['mAP']:.4f}"
                     for sigma in SIGMAS) + f" | {record['runtime_s'] / 60:.1f} min")

fold 0: baseline mAP 0.2377 (stored 0.2377) | σ=0.3: 0.2407 | σ=0.5: 0.2382 | σ=0.7: 0.2345 | 4.5 min
fold 1: baseline mAP 0.2208 (stored 0.2208) | σ=0.3: 0.2246 | σ=0.5: 0.2202 | σ=0.7: 0.2158 | 5.6 min
fold 2: baseline mAP 0.2380 (stored 0.2380) | σ=0.3: 0.2425 | σ=0.5: 0.2398 | σ=0.7: 0.2359 | 7.0 min


fold 3: baseline mAP 0.2015 (stored 0.2015) | σ=0.3: 0.2044 | σ=0.5: 0.2027 | σ=0.7: 0.2000 | 6.6 min
fold 4: baseline mAP 0.1571 (stored 0.1571) | σ=0.3: 0.1596 | σ=0.5: 0.1581 | σ=0.7: 0.1562 | 6.5 min


### Extension: σ = 0.1 and 0.2

In the previous experiment, the smallest value, σ = 0.3, gave the best result, while mAP decreased as σ increased, so we test two additional smaller values.

In [ ]:
EXTENSION_SETTINGS = dict(SETTINGS, sigmas=EXTRA_SIGMAS)
EXTENSION_ID = hashlib.sha256(json.dumps(EXTENSION_SETTINGS, sort_keys=True).encode()).hexdigest()[:16]

In [ ]:
def evaluate_extension(fold):
    started = time.time()
    model, run_config, checkpoint = load_model(fold)
    images = fold_images(fold)
    gt = make_gt(coco_train, [image['id'] for image in images])

    first_batch = predict(model, images[:EVAL_BATCH])
    candidates = []
    use_final_nms(model, capture(candidates))
    outputs = predict(model, images)
    use_final_nms(model)
    n_candidates = np.array([len(candidate[1]) for candidate in candidates])
    assert float(n_candidates.mean()) == fold_results[fold]['candidates_per_img']['mean']
    cached_base, softs = postprocess_cached(candidates, outputs, model.roi_heads, EXTRA_SIGMAS)
    assert same_predictions(first_batch, cached_base[:EVAL_BATCH])
    for sigma in EXTRA_SIGMAS:
        use_final_nms(model, soft_final_nms(sigma))
        assert same_predictions(predict(model, images[:EVAL_BATCH]), softs[sigma][:EVAL_BATCH])
    use_final_nms(model)

    metrics = {f'soft_nms_s{sigma}': coco_metrics(softs[sigma], gt) for sigma in EXTRA_SIGMAS}

    rng = np.random.default_rng(SEED + fold)
    picks = [images[i] for i in rng.choice(len(images), SPEED_IMAGES + SPEED_WARMUP, replace=False)]
    speed_images = [to_tensor(array).to(DEVICE) for array in read_images(picks)]
    reference = measure_speed(model, speed_images)
    speed = {}
    for sigma in EXTRA_SIGMAS:
        use_final_nms(model, soft_final_nms(sigma))
        speed[f'soft_nms_s{sigma}'] = measure_speed(model, speed_images)
    use_final_nms(model)

    rows = [dict(fold=fold, variant=name, method='soft_nms', sigma=float(name.split('_s')[1]),
                 **variant_metrics,
                 ms_per_img=speed[name], fps=1000 / speed[name],
                 detections_per_img=float(np.mean([len(pred['scores'])
                                                   for _, pred in softs[float(name.split('_s')[1])]])))
            for name, variant_metrics in metrics.items()]
    first_run = {row['variant']: row for row in fold_results[fold]['rows']}['baseline']['ms_per_img']
    result = dict(fold=fold, settings_id=EXTENSION_ID,
                  checkpoint=str((RUNS / f'fold{fold}/best.pth').relative_to(ROOT)),
                  checkpoint_sha256=sha256(RUNS / f'fold{fold}/best.pth'), best_epoch=checkpoint['epoch'],
                  val_frames=len(images),
                  candidates_per_img=dict(mean=float(n_candidates.mean()), max=int(n_candidates.max())),
                  checks=dict(candidates_same_as_first_run=True, standard_nms_identical_first_batch=True,
                              forward_soft_nms_identical_first_batch=True),
                  speed_reference=dict(standard_nms_ms_now=reference, standard_nms_ms_first_run=first_run),
                  rows=rows, runtime_s=time.time() - started)
    del model, candidates, outputs, speed_images
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return result

In [ ]:
extension_results = {}
for fold in FOLDS:
    path = OUT / 'folds' / f'fold{fold}_extra_sigmas.json'
    if path.exists():
        old = json.loads(path.read_text())
        if (old['settings_id'] == EXTENSION_ID
            and old['checkpoint_sha256'] == fold_results[fold]['checkpoint_sha256']):
            extension_results[fold] = old
    if fold not in extension_results:
        extension_results[fold] = evaluate_extension(fold)
        save_json(extension_results[fold], path)
    record = extension_results[fold]
    m = {row['variant']: row for row in record['rows']}
    print(f"fold {fold}: " + ' | '.join(f"σ={sigma}: {m[f'soft_nms_s{sigma}']['mAP']:.4f}"
                                        for sigma in EXTRA_SIGMAS) +
          f" | standard NMS {record['speed_reference']['standard_nms_ms_now']:.1f} ms now, "
          f"{record['speed_reference']['standard_nms_ms_first_run']:.1f} ms in the first run | "
          f"{record['runtime_s'] / 60:.1f} min")

fold 0: σ=0.1: 0.2384 | σ=0.2: 0.2408 | standard NMS 26.9 ms now, 27.9 ms in the first run | 3.4 min


fold 1: σ=0.1: 0.2220 | σ=0.2: 0.2252 | standard NMS 26.9 ms now, 27.0 ms in the first run | 3.4 min


fold 2: σ=0.1: 0.2396 | σ=0.2: 0.2426 | standard NMS 27.2 ms now, 26.4 ms in the first run | 4.0 min


fold 3: σ=0.1: 0.2012 | σ=0.2: 0.2040 | standard NMS 27.5 ms now, 26.3 ms in the first run | 4.0 min


fold 4: σ=0.1: 0.1576 | σ=0.2: 0.1595 | standard NMS 26.5 ms now, 25.2 ms in the first run | 3.8 min



### Results and visualizations


In [ ]:
fold_df = pd.DataFrame([row for k in FOLDS for row in fold_results[k]['rows'] + extension_results[k]['rows']])
fold_df.to_csv(OUT / 'soft_nms_fold_results.csv', index=False)
VARIANTS = ['baseline'] + [f'soft_nms_s{sigma}' for sigma in ALL_SIGMAS]
COLS = KEYS + ['ms_per_img', 'fps', 'detections_per_img']

stats = fold_df.groupby('variant')[COLS].agg(['mean', 'std']).loc[VARIANTS]
summary = pd.DataFrame(index=VARIANTS)
summary['sigma'] = [None] + ALL_SIGMAS
for col in COLS:
    summary[col + '_mean'], summary[col + '_std'] = stats[col, 'mean'], stats[col, 'std']
baseline_rows = fold_df[fold_df.variant == 'baseline'].set_index('fold')
for variant in VARIANTS:
    diff = (fold_df[fold_df.variant == variant].set_index('fold')[['mAP',
                                                                   'APs']] - baseline_rows[['mAP', 'APs']])
    summary.loc[variant,
                'delta_mAP_mean'], summary.loc[variant, 'delta_mAP_std'] = diff.mAP.mean(), diff.mAP.std()
    summary.loc[variant,
                'delta_APs_mean'], summary.loc[variant, 'delta_APs_std'] = diff.APs.mean(), diff.APs.std()
    summary.loc[variant, 'folds_mAP_improved'] = int((diff.mAP > 0).sum())
summary.index.name = 'variant'
summary.to_csv(OUT / 'soft_nms_summary.csv')

In [ ]:
BEST = summary.drop('baseline').mAP_mean.idxmax()
BEST_SIGMA = float(summary.loc[BEST, 'sigma'])
table = pd.DataFrame({col: summary[col + '_mean'].map('{:.4f}'.format) + ' ± '
                      + summary[col + '_std'].map('{:.4f}'.format)
                      for col in KEYS + ['ms_per_img']})
table['ΔmAP vs baseline'] = (summary.delta_mAP_mean.map('{:+.4f}'.format) + ' ± '
                             + summary.delta_mAP_std.map('{:.4f}'.format))
table['ΔAP_small'] = (summary.delta_APs_mean.map('{:+.4f}'.format) + ' ± '
                      + summary.delta_APs_std.map('{:.4f}'.format))
display(table)
display(fold_df.pivot(index='fold', columns='variant', values='mAP')[VARIANTS].round(4))
print('Best σ by mean CV mAP:', BEST_SIGMA)

checks = pd.DataFrame([dict(
    fold=k, baseline_vs_stored_max_abs_diff=result['baseline_max_abs_diff_vs_stored'],
    **result['checks'], candidates_mean=result['candidates_per_img']['mean'],
    candidates_max=result['candidates_per_img']['max'],
    runtime_min=result['runtime_s'] / 60,
    **{'extension_' + col: value
       for col, value in extension_results[k]['checks'].items()},
    standard_nms_ms_first_run=extension_results[k]['speed_reference']['standard_nms_ms_first_run'],
    standard_nms_ms_extension=extension_results[k]['speed_reference']['standard_nms_ms_now'],
    extension_runtime_min=extension_results[k]['runtime_s'] / 60)
                       for k, result in fold_results.items()]).set_index('fold')
display(checks.round(6))

,mAP,AP50,AP75,APs,APm,APl,AR,ms_per_img,ΔmAP vs baseline,ΔAP_small
variant,,,,,,,,,,
baseline,0.2110 ± 0.0337,0.4137 ± 0.0489,0.1925 ± 0.0410,0.0757 ± 0.0276,0.1981 ± 0.0132,0.3856 ± 0.0507,0.3450 ± 0.0404,26.5588 ± 0.9877,+0.0000 ± 0.0000,+0.0000 ± 0.0000
soft_nms_s0.1,0.2117 ± 0.0340,0.4157 ± 0.0495,0.1930 ± 0.0415,0.0769 ± 0.0274,0.1990 ± 0.0135,0.3848 ± 0.0526,0.3504 ± 0.0402,95.6750 ± 6.0465,+0.0007 ± 0.0007,+0.0012 ± 0.0002
soft_nms_s0.2,0.2144 ± 0.0344,0.4176 ± 0.0496,0.1970 ± 0.0418,0.0784 ± 0.0282,0.2019 ± 0.0135,0.3896 ± 0.0519,0.3709 ± 0.0415,91.3553 ± 5.7607,+0.0034 ± 0.0010,+0.0027 ± 0.0009
soft_nms_s0.3,0.2144 ± 0.0342,0.4147 ± 0.0488,0.1985 ± 0.0420,0.0789 ± 0.0287,0.2020 ± 0.0130,0.3908 ± 0.0515,0.3802 ± 0.0421,86.8125 ± 7.0301,+0.0034 ± 0.0008,+0.0032 ± 0.0016
soft_nms_s0.5,0.2118 ± 0.0336,0.4063 ± 0.0474,0.1979 ± 0.0415,0.0782 ± 0.0288,0.1995 ± 0.0113,0.3889 ± 0.0517,0.3873 ± 0.0430,87.6332 ± 5.1938,+0.0008 ± 0.0009,+0.0025 ± 0.0020
soft_nms_s0.7,0.2085 ± 0.0328,0.3976 ± 0.0458,0.1961 ± 0.0406,0.0770 ± 0.0287,0.1965 ± 0.0101,0.3848 ± 0.0524,0.3900 ± 0.0438,86.6794 ± 9.2154,-0.0025 ± 0.0016,+0.0013 ± 0.0023


variant,baseline,soft_nms_s0.1,soft_nms_s0.2,soft_nms_s0.3,soft_nms_s0.5,soft_nms_s0.7
fold,,,,,,
0,0.2377,0.2384,0.2408,0.2407,0.2382,0.2345
1,0.2208,0.2220,0.2252,0.2246,0.2202,0.2158
2,0.2380,0.2396,0.2426,0.2425,0.2398,0.2359
3,0.2015,0.2012,0.2040,0.2044,0.2027,0.2000
4,0.1571,0.1576,0.1595,0.1596,0.1581,0.1562


Best σ by mean CV mAP: 0.2


,baseline_vs_stored_max_abs_diff,patched_standard_nms_identical,forward_soft_nms_identical_first_batch,candidates_mean,candidates_max,runtime_min,extension_candidates_same_as_first_run,extension_standard_nms_identical_first_batch,extension_forward_soft_nms_identical_first_batch,standard_nms_ms_first_run,standard_nms_ms_extension,extension_runtime_min
fold,,,,,,,,,,,,
0,0.0,True,True,586.285423,1048,4.538648,True,True,True,27.879173,26.864888,3.364881
1,0.0,True,True,543.364658,1035,5.637870,True,True,True,26.998278,26.886418,3.363997
2,0.0,True,True,729.692627,1234,7.042655,True,True,True,26.440222,27.234328,4.014106
3,0.0,True,True,716.241273,1184,6.619092,True,True,True,26.287813,27.492370,3.980964
4,0.0,True,True,748.372812,1264,6.459814,True,True,True,25.188453,26.498908,3.769459


In [ ]:
save_json(dict(SETTINGS, sigmas=ALL_SIGMAS, runs=dict(
    first=dict(sigmas=SIGMAS, settings_id=SETTINGS_ID),
    extension=dict(sigmas=EXTRA_SIGMAS, settings_id=EXTENSION_ID)),
               settings_id=SETTINGS_ID, selection='highest mean 5-fold CV mAP@[.50:.95]',
               evaluation='COCOeval bbox, original annotations incl. iscrowd, maxDets=[1,10,100], held-out '
                          'fold of each checkpoint',
               nms_replaced='RoIHeads.postprocess_detections: batched_nms(IoU nms_thr) + top det_per_img',
               unchanged='RPN NMS, score_thr, box decoding, clipping, small-box removal, det_per_img, '
                         'preprocessing',
               soft_nms='s_i <- s_i * exp(-IoU(M, b_i)^2 / sigma), same class only; drop s_i < '
                        'soft_min_score',
               speed='batch 1, autocast, images on GPU before timing, full forward pass incl. NMS; '
                     'sigma 0.1 and 0.2 timed in a later session (see standard_nms_ms_extension in the '
                     'checks table)',
               checkpoints={k: dict(path=result['checkpoint'], sha256=result['checkpoint_sha256'])
                            for k, result in fold_results.items()},
               environment=dict(torch=torch.__version__, torchvision=torchvision.__version__,
                                device=torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu')),
          OUT / 'soft_nms_config.json')
best_rows = fold_df[fold_df.variant == BEST].set_index('fold')
save_json(dict(variant=BEST, sigma=BEST_SIGMA, selected_by='mean 5-fold CV mAP@[.50:.95]',
               cv_mean={col: float(summary.loc[BEST, col + '_mean']) for col in COLS},
               cv_std={col: float(summary.loc[BEST, col + '_std']) for col in COLS},
               baseline_cv_mean={col: float(summary.loc['baseline', col + '_mean']) for col in COLS},
               delta_vs_baseline=dict(mAP_mean=float(summary.loc[BEST, 'delta_mAP_mean']),
                                      mAP_std=float(summary.loc[BEST, 'delta_mAP_std']),
                                      APs_mean=float(summary.loc[BEST, 'delta_APs_mean']),
                                      APs_std=float(summary.loc[BEST, 'delta_APs_std']),
                                      folds_mAP_improved=int(summary.loc[BEST, 'folds_mAP_improved'])),
               folds={int(k): {col: float(best_rows.loc[k, col]) for col in KEYS + ['ms_per_img']}
                      for k in best_rows.index}),
          OUT / 'soft_nms_best_config.json')

In [ ]:
PLOTS = OUT / 'plots'
PLOTS.mkdir(exist_ok=True)
NAMES = {'baseline': 'Standard NMS', **{f'soft_nms_s{sigma}': f'Soft-NMS σ={sigma}' for sigma in ALL_SIGMAS}}
COLORS = {'baseline': '#4c78a8', **dict(zip([f'soft_nms_s{sigma}' for sigma in ALL_SIGMAS],
                                            ['#9d755d', '#54a24b', '#f58518', '#e45756', '#72b7b2']))}

def save_plot(fig, name):
    fig.update_layout(template='plotly_white', font=dict(size=13))
    fig.write_html(PLOTS / f'{name}.html', include_plotlyjs='directory', div_id=name)
    fig.show()

In [ ]:
fig = go.Figure()
for variant in ['baseline', BEST]:
    subset = fold_df[fold_df.variant == variant]
    fig.add_bar(x=[f'fold {k}' for k in subset.fold] + ['mean'], y=list(subset.mAP) + [subset.mAP.mean()],
                name=NAMES[variant],
                marker_color=COLORS[variant], error_y=dict(array=[0] * len(subset) + [subset.mAP.std()]))
fig.update_layout(title=f'COCO mAP@[.50:.95] per fold: standard NMS vs Soft-NMS σ={BEST_SIGMA} (CV held-out '
                        f'folds)',
                  yaxis_title='mAP', barmode='group')
save_plot(fig, 'map_baseline_vs_soft_nms')

fig = go.Figure()
for variant in VARIANTS:
    subset = fold_df[fold_df.variant == variant]
    fig.add_scatter(x=subset.fold, y=subset.APs, mode='lines+markers', name=NAMES[variant],
                    line=dict(color=COLORS[variant]))
fig.update_layout(title='AP_small per fold (CV held-out folds)', xaxis=dict(title='fold', dtick=1),
                  yaxis_title='AP_small')
save_plot(fig, 'ap_small_by_fold')

In [ ]:
subset = summary.drop('baseline')
fig = go.Figure(go.Scatter(x=subset.sigma, y=subset.mAP_mean, error_y=dict(array=subset.mAP_std),
                           mode='lines+markers',
                           name='Soft-NMS', line=dict(color=COLORS[BEST])))
fig.add_hline(y=summary.loc['baseline', 'mAP_mean'], line_dash='dash', line_color=COLORS['baseline'],
              annotation_text=f"standard NMS {summary.loc['baseline', 'mAP_mean']:.4f}")
fig.update_layout(title='Mean 5-fold CV mAP for each σ (error bars: std over folds)',
                  xaxis=dict(title='σ', tickvals=ALL_SIGMAS),
                  yaxis_title='mAP@[.50:.95]')
save_plot(fig, 'map_by_sigma')

fig = go.Figure(go.Bar(x=[NAMES[variant] for variant in VARIANTS], y=summary.ms_per_img_mean,
                       error_y=dict(array=summary.ms_per_img_std),
                       marker_color=[COLORS[variant] for variant in VARIANTS],
                       text=[f'{ms:.1f} ms · {fps:.1f} FPS'
                             for ms, fps in zip(summary.ms_per_img_mean, summary.fps_mean)]))
fig.update_layout(title=f'Inference time per image (batch 1, AMP, {DEVICE.type}; mean ± std over 5 fold '
                        f'models)',
                  yaxis_title='ms / image')
save_plot(fig, 'inference_time')

In [ ]:
VIS = OUT / 'visualizations'
VIS.mkdir(exist_ok=True)
SHOW_THR = .3
rep_fold = int(baseline_rows.mAP.sort_values().index[len(baseline_rows) // 2])
images = fold_images(rep_fold)
n_objects = (pd.Series([annotation['image_id'] for annotation in coco_train['annotations']
                        if not annotation.get('iscrowd', 0)]).value_counts())
crowded = max(images, key=lambda image_info: n_objects.get(image_info['id'], 0))
random_frame = images[int(np.random.default_rng(SEED).integers(len(images)))]

In [ ]:
def box_trace(boxes, color, name, text=None):
    xs, ys, hover = [], [], []
    for i, (x1, y1, x2, y2) in enumerate(boxes):
        xs += [x1, x2, x2, x1, x1, None]
        ys += [y1, y1, y2, y2, y1, None]
        hover += [text[i] if text else name] * 5 + [None]
    return go.Scatter(x=xs, y=ys, mode='lines', line=dict(color=color, width=1.2), name=name,
                      text=hover, hoverinfo='text', showlegend=False)

In [ ]:
def prediction_figure(rgb, image_info, preds, title):
    h, w = rgb.shape[:2]
    anns = [annotation for annotation in coco_train['annotations']
            if annotation['image_id'] == image_info['id']]
    gt_boxes = [[annotation['bbox'][0], annotation['bbox'][1], annotation['bbox'][0] + annotation['bbox'][2],
                 annotation['bbox'][1]
                 + annotation['bbox'][3]] for annotation in anns if not annotation.get('iscrowd', 0)]
    ignored = [[
        annotation['bbox'][0], annotation['bbox'][1], annotation['bbox'][0] + annotation['bbox'][2],
        annotation['bbox'][1] + annotation['bbox'][3]] for annotation in anns if annotation.get('iscrowd', 0)]
    small = Image.fromarray(rgb)
    small.thumbnail((1280, 1280))
    buffer = io.BytesIO()
    small.save(buffer, 'JPEG', quality=85)
    source = 'data:image/jpeg;base64,' + base64.b64encode(buffer.getvalue()).decode()
    names = list(preds)
    fig = make_subplots(1, len(names), horizontal_spacing=.01,
                        subplot_titles=[
                            f"{name}: {int((pred['scores'] >= SHOW_THR).sum())} boxes with score ≥ {SHOW_THR}"
                            for name, pred in preds.items()])
    for col, name in enumerate(names, 1):
        pred = preds[name]
        keep = pred['scores'] >= SHOW_THR
        text = [f"{CATS[LABEL_TO_CAT[int(label)]]} {score:.2f}"
                for label, score in zip(pred['labels'][keep].tolist(), pred['scores'][keep].tolist())]
        fig.add_trace(box_trace(gt_boxes, 'lime', 'ground truth'), 1, col)
        fig.add_trace(box_trace(ignored, 'yellow', 'ignored region'), 1, col)
        fig.add_trace(box_trace(pred['boxes'][keep].tolist(), 'red', 'prediction', text), 1, col)
        fig.add_layout_image(source=source, xref=f'x{col}', yref=f'y{col}', x=0, y=0, sizex=w, sizey=h,
                             xanchor='left', yanchor='top', sizing='stretch', layer='below')
    fig.update_xaxes(range=[0, w], visible=False)
    fig.update_yaxes(range=[h, 0], visible=False)
    fig.update_layout(title=title + ' · green = ground truth, red = prediction, yellow = ignored region',
                      width=1500, height=int(750 * h / w) + 120, margin=dict(l=10, r=10, t=90, b=10),
                      template='plotly_white')
    return fig

In [ ]:
model, run_config, checkpoint = load_model(rep_fold)
saved = []
for label, image_info in [('most_objects', crowded), ('random', random_frame)]:
    rgb = read_images([image_info])[0]
    image = to_tensor(rgb).to(DEVICE)
    preds = {}
    for name, final_nms in [('Standard NMS', None), (f'Soft-NMS σ={BEST_SIGMA}', soft_final_nms(BEST_SIGMA))]:
        use_final_nms(model, final_nms)
        with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=AMP):
            preds[name] = {k: value.cpu() for k, value in model([image])[0].items()}
    use_final_nms(model)
    fig = prediction_figure(rgb, image_info, preds,
                            f"Fold {rep_fold} held-out frame {image_info['file_name']} "
                            f"({n_objects.get(image_info['id'], 0)} objects)")
    path = VIS / f"fold{rep_fold}_{label}_{image_info['id']}.html"
    fig.write_html(path, include_plotlyjs='directory')
    saved.append(path.name)
del model
print('Saved to', VIS, ':', saved)

Saved to E:\ucu_hw2\faster_rcnn\inference_experiments\soft_nms\visualizations : ['fold1_most_objects_1128.html', 'fold1_random_18517.html']


### Conclusion

Gaussian Soft-NMS with a small σ improves the trained `lr_scheduler` Faster R-CNN:

| | mAP | ΔmAP vs standard NMS | ΔAP_small | folds with higher mAP | ms / image |
|---|---|---|---|---|---|
| Standard NMS | 0.2110 ± 0.0337 | | | | 26.6 |
| σ = 0.1 | 0.2117 ± 0.0340 | +0.0007 ± 0.0007 | +0.0012 | 4 / 5 | 95.7 |
| **σ = 0.2** (selected) | **0.2144 ± 0.0344** | **+0.0034 ± 0.0010** | +0.0027 | 5 / 5 | 91.4 |
| σ = 0.3 | 0.2144 ± 0.0342 | +0.0034 ± 0.0008 | +0.0032 | 5 / 5 | 86.8 |
| σ = 0.5 | 0.2118 ± 0.0336 | +0.0008 ± 0.0009 | +0.0025 | 4 / 5 | 87.6 |
| σ = 0.7 | 0.2085 ± 0.0328 | −0.0025 ± 0.0016 | +0.0013 | 0 / 5 | 86.7 |

* σ = 0.2 has the highest mean CV mAP, but σ = 0.3 is practically equal. Both improve mAP on all five folds; σ = 0.3 gives slightly more AP_small and recall.
* mAP peaks between σ = 0.2 and 0.3. A smaller σ suppresses overlaps almost like standard NMS (σ = 0.1 gains little); a larger σ keeps too many duplicates, AP50 falls and σ = 0.7 is worse than standard NMS.
* The improvement is small (+0.003 mAP, about 1.6%), while the sequential Python implementation of Soft-NMS makes inference about 3.4 times slower.

## Experiment №2 SAHI

**Slicing Aided Hyper Inference (SAHI)**

In this experiment, we split each validation image into smaller parts, run Faster R-CNN on each part, and then combine all predictions into one image. Since each part is resized to 720 px, small objects appear larger, and we hope this will help the model detect them better.

The goal is to check whether SAHI improves COCO mAP@[.50:.95] and AP_small compared to the standard model without Soft-NMS.

We run the experiment in three stages, testing each configuration on all 5 folds:

1. Compare slice sizes of 512, 640, 768, and 1024 px with an overlap of 0.2.
2. For the best slice size, test overlaps of 0.1, 0.2, 0.3, and 0.4.
3. For the best combination, compare NMS and GREEDYNMM, and check whether an additional full-image pass improves the results.

In [ ]:
%pip -q install sahi

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import sahi
from sahi.slicing import get_slice_bboxes
from sahi.prediction import ObjectPrediction
from sahi.postprocess.combine import batched_nms, NMSPostprocess, GreedyNMMPostprocess
import sahi.predict as sahi_predict

In [ ]:
SAHI_OUT = ROOT / 'faster_rcnn/inference_experiments/sahi'
SLICE_SIZES = [512, 640, 768, 1024]
OVERLAPS = [0.1, 0.2, 0.3, 0.4]
STAGE1_OVERLAP = 0.2
SLICE_BATCH = 8
MATCH = {'nms': ('IOU', 0.5), 'greedynmm': ('IOS', 0.5)}
DET_PER_IMG = INFERENCE['det_per_img']
SAHI_SETTINGS = dict(sahi=sahi.__version__, torch=torch.__version__, torchvision=torchvision.__version__,
                     slice_batch=SLICE_BATCH, match=MATCH, det_per_img=DET_PER_IMG, amp=AMP,
                     coco_max_dets=MAX_DETS,
                     speed_images=SPEED_IMAGES, speed_warmup=SPEED_WARMUP, seed=SEED, folds_sha=FOLDS_SHA,
                     inference=INFERENCE)
SAHI_ID = hashlib.sha256(json.dumps(SAHI_SETTINGS, sort_keys=True).encode()).hexdigest()[:16]
for subfolder in ['folds', 'cache', 'plots', 'visualizations']:
    (SAHI_OUT / subfolder).mkdir(parents=True, exist_ok=True)
print('sahi', sahi.__version__, '| torch', torch.__version__, '| torchvision', torchvision.__version__,
      '| SAHI low-confidence switch below', sahi_predict.LOW_MODEL_CONFIDENCE)

sahi 0.14.0 | torch 2.11.0+cu128 | torchvision 0.26.0+cu128 | SAHI low-confidence switch below 0.1


### Image sizes and resizing

The model resizes each image so that its shorter side is 720 px and its longer side does not exceed 1280 px, which means full frames are downscaled while smaller slices can be upscaled.

For example, a 512 px slice is upscaled by about 1.41 times, making small objects easier for the model to detect. The `zoom` value shows how much larger an object appears compared to processing the full frame.

With 1024 px slices, the scaling effect is much smaller, and for some images there is no difference at all (`zoom = 1.0`), because SAHI clips slices that are larger than the image itself.

In [ ]:
def input_scale(h, w):
    return min(INFERENCE['min_size'] / min(h, w), INFERENCE['max_size'] / max(h, w))

frames = pd.DataFrame([image_info for k in FOLDS for image_info in fold_images(k)])
sizes = frames.groupby(['width', 'height']).size().rename('frames').reset_index()
sizes['full_scale'] = [input_scale(h, w) for w, h in zip(sizes.width, sizes.height)]
for s in SLICE_SIZES:
    windows = [get_slice_bboxes(h, w, s, s, False, STAGE1_OVERLAP, STAGE1_OVERLAP)
               for w, h in zip(sizes.width, sizes.height)]
    for (w, h), frame_windows in zip(zip(sizes.width, sizes.height), windows):
        assert min(x0 for x0, _, _, _ in frame_windows) == 0 and max(x1 for _, _, x1, _ in frame_windows) == w
        assert min(y0 for _, y0, _, _ in frame_windows) == 0 and max(y1 for _, _, _, y1 in frame_windows) == h
        assert all(0 <= x0 < x1 <= w and 0 <= y0 < y1 <= h for x0, y0, x1, y1 in frame_windows)
    sizes[f'slices_{s}'] = [len(frame_windows) for frame_windows in windows]
    sizes[f'zoom_{s}'] = [input_scale(min(s, h), min(s, w)) / full_scale
                          for w, h, full_scale in zip(sizes.width, sizes.height, sizes.full_scale)]
display(sizes.round(2))
weights = sizes.frames / sizes.frames.sum()
print('Mean slices per frame (overlap 0.2):',
      {size: round(float((sizes[f'slices_{size}'] * weights).sum()), 1) for size in SLICE_SIZES})

,width,height,frames,full_scale,slices_512,zoom_512,slices_640,zoom_640,slices_768,zoom_768,slices_1024,zoom_1024
0,1344,756,1134,0.95,8,1.48,6,1.18,2,1.00,2,1.00
1,1360,765,193,0.94,8,1.49,6,1.20,2,1.00,2,1.00
2,1904,1071,2593,0.67,15,2.09,8,1.67,6,1.39,6,1.05
3,1920,1080,145,0.67,15,2.11,8,1.69,6,1.41,6,1.05
4,2688,1512,707,0.48,28,2.95,15,2.36,15,1.97,8,1.48
5,2720,1530,96,0.47,28,2.99,18,2.39,15,1.99,8,1.49


Mean slices per frame (overlap 0.2): {512: 15.2, 640: 8.7, 768: 6.4, 1024: 5.2}


### SAHI inference and merging

We use SAHI's get_slice_bboxes function to split images into overlapping slices of a selected size, adjusting slices near the edges so they fit completely within the frame.

Each slice is processed by the original Faster R-CNN model with standard NMS, a score threshold of 0.01, and a maximum of 500 detections. The detected boxes are then shifted back to the coordinates of the full image.

To merge boxes from the same image, we use two methods: NMS with IoU 0.5, which removes duplicates, and GREEDYNMM with IoS 0.5, which merges boxes of the same class. After merging, we keep up to 500 detections with the highest scores.

In the first two stages, we test NMS without an additional full-image pass, while in the third stage, we compare it with GREEDYNMM and separately check whether an additional full-image pass improves the results.


In [ ]:
def slice_predict(model, image, size, overlap):
    h, w = image.shape[-2:]
    windows = get_slice_bboxes(h, w, size, size, False, overlap, overlap)
    boxes, scores, labels = [], [], []
    with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=AMP):
        for i in range(0, len(windows), SLICE_BATCH):
            chunk = windows[i:i + SLICE_BATCH]
            preds = model([image[:, y0:y1, x0:x1] for x0, y0, x1, y1 in chunk])
            for (x0, y0, _, _), pred in zip(chunk, preds):
                boxes.append(pred['boxes'].float() + torch.tensor([x0, y0, x0, y0], dtype=torch.float32,
                                                                  device=DEVICE))
                scores.append(pred['scores'].float())
                labels.append(pred['labels'])
    return torch.cat(
        boxes).cpu().numpy(), torch.cat(scores).cpu().numpy(), torch.cat(labels).cpu().numpy(), len(windows)

In [ ]:
def merge(boxes, scores, labels, method, h, w):
    metric, threshold = MATCH[method]
    if len(scores) > 1 and method == 'nms':
        keep = batched_nms(np.c_[boxes, scores, labels], metric, threshold)
        boxes, scores, labels = boxes[keep], scores[keep], labels[keep]
    elif len(scores) > 1:
        preds = [ObjectPrediction(bbox=box.tolist(), category_id=int(label), category_name=str(int(label)),
                                  score=float(score),
                                  shift_amount=[0, 0], full_shape=[h, w]) for box, score, label in zip(boxes,
                                  scores, labels)]
        merged = GreedyNMMPostprocess(threshold, metric, class_agnostic=False)(preds)
        boxes = np.array([pred.bbox.to_xyxy() for pred in merged], dtype=np.float32).reshape(-1, 4)
        scores = np.array([pred.score.value for pred in merged], dtype=np.float32)
        labels = np.array([pred.category.id for pred in merged], dtype=np.int64)
    top = np.argsort(-scores, kind='stable')[:DET_PER_IMG]
    return boxes[top], scores[top], labels[top]

def as_prediction(boxes, scores, labels):
    return dict(boxes=boxes, scores=scores, labels=labels)

In [ ]:
rng = np.random.default_rng(SEED)
top_left = rng.uniform(0, 900, (400, 2))
toy_boxes = np.c_[top_left, top_left + rng.uniform(5, 80, (400, 2))].astype(np.float32)
toy_scores, toy_labels = rng.uniform(.01, 1, 400).astype(np.float32), rng.integers(1, 9, 400)
objects = [ObjectPrediction(bbox=box.tolist(), category_id=int(label), category_name=str(int(label)),
                            score=float(score),
                            shift_amount=[0, 0],
                            full_shape=[1080, 1920]) for box, score, label in zip(toy_boxes, toy_scores,
                            toy_labels)]
reference = sorted((round(pred.score.value, 6), pred.category.id)
                   for pred in NMSPostprocess(0.5, 'IOU', class_agnostic=False)(objects))
assert (reference == sorted(
    (round(float(score), 6), int(label))
    for score, label in zip(*merge(toy_boxes, toy_scores, toy_labels, 'nms', 1080, 1920)[1:])))
print('NMS check passed.')

NMS check passed.


### Evaluation, caching and timing


In [ ]:
def config_name(size, overlap, method, full):
    return f's{size}_o{overlap}_{method}' + ('_full' if full else '')

def write_json(obj, path):
    tmp_path = path.with_suffix('.tmp')
    tmp_path.write_text(json.dumps(obj, indent=2))
    os.replace(tmp_path, path)

def save_cache(path, rows):
    tmp_path = path.with_name(path.stem + '.tmp.npz')
    np.savez(tmp_path, ids=np.array([row[0] for row in rows]), counts=np.array([len(row[2]) for row in rows]),
             slices=np.array([row[4] if len(row) > 4 else 1 for row in rows]),
             boxes=np.concatenate([row[1] for row in rows]).astype(np.float32),
             scores=np.concatenate([row[2] for row in rows]),
             labels=np.concatenate([row[3] for row in rows]))
    os.replace(tmp_path, path)

def load_cache(path):
    data = np.load(path)
    ends = np.cumsum(data['counts'])
    starts = ends - data['counts']
    boxes, scores, labels = data['boxes'], data['scores'], data['labels']
    return [(int(i), boxes[start:end], scores[start:end], labels[start:end], int(n_slices))
            for i, start, end, n_slices in zip(data['ids'], starts, ends, data['slices'])]

In [ ]:
def result_path(name, fold):
    return SAHI_OUT / 'folds' / f'{name}_fold{fold}.json'

def is_done(name, fold, checkpoint_sha):
    path = result_path(name, fold)
    if not path.exists():
        return False
    old = json.loads(path.read_text())
    return old['settings_id'] == SAHI_ID and old['checkpoint_sha256'] == checkpoint_sha

def speed_frames(fold, images):
    rng = np.random.default_rng(SEED + fold)
    picks = [images[i] for i in rng.choice(len(images), SPEED_IMAGES + SPEED_WARMUP, replace=False)]
    return [to_tensor(array).to(DEVICE) for array in read_images(picks)]

In [ ]:
@torch.inference_mode()
def run_frame(model, image, size=None, overlap=None, method=None, full=False):
    if size is None:
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            return model([image])
    boxes, scores, labels, _ = slice_predict(model, image, size, overlap)
    if full:
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            pred = model([image])[0]
        boxes = np.concatenate([boxes, pred['boxes'].float().cpu().numpy()])
        scores = np.concatenate([scores, pred['scores'].float().cpu().numpy()])
        labels = np.concatenate([labels, pred['labels'].cpu().numpy()])
    return merge(boxes, scores, labels, method, *image.shape[-2:])

def result_row(name, fold, metrics, preds, extra):
    return dict(config=name, fold=fold, **metrics,
                detections_per_img=float(np.mean([len(pred['scores']) for _, pred in preds])), **extra)

In [ ]:
def run_baseline(fold):
    checkpoint_hash = sha256(RUNS / f'fold{fold}/best.pth')
    if is_done('baseline', fold, checkpoint_hash):
        return
    model, run_config, checkpoint = load_model(fold)
    images = fold_images(fold)
    preds = predict(model, images)
    metrics = coco_metrics(preds, make_gt(coco_train, [image_info['id'] for image_info in images]))
    diff = max(abs(metrics[k] - stored[fold]['metrics'][k]) for k in KEYS)
    assert diff == 0, f'baseline differs from result.json by {diff}'
    save_cache(SAHI_OUT / 'cache' / f'full_fold{fold}.npz',
               [(image_id, pred['boxes'].float().numpy(), pred['scores'].numpy(), pred['labels'].numpy())
                for image_id, pred in preds])
    row = result_row('baseline', fold, metrics, preds,
                     dict(stage='baseline', slice_size=None, overlap=None, merge='model NMS', full_image=True,
                          slices_per_img=1, candidates_per_img=None))
    write_json(dict(settings_id=SAHI_ID, checkpoint_sha256=checkpoint_hash,
                    baseline_max_abs_diff_vs_stored=diff, row=row), result_path('baseline', fold))
    del model
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

In [ ]:
def run_sahi(fold, stage, size, overlap, variants):
    checkpoint_hash = sha256(RUNS / f'fold{fold}/best.pth')
    todo = [(method, with_full) for method, with_full in variants
            if not is_done(config_name(size, overlap, method, with_full), fold, checkpoint_hash)]
    if not todo:
        return
    started = time.time()
    model, run_config, checkpoint = load_model(fold)
    images = fold_images(fold)
    dims = {int(image_info['id']): (image_info['height'], image_info['width']) for image_info in images}
    gt = make_gt(coco_train, list(dims))
    path = SAHI_OUT / 'cache' / f'slices{size}_overlap{overlap}_fold{fold}.npz'
    if not path.exists():
        rows = []
        for chunk, image_tensors in batches(images, 1):
            rows.append((int(chunk[0]['id']),
                         *slice_predict(model, image_tensors[0].to(DEVICE), size, overlap)))
        save_cache(path, rows)
    candidates = load_cache(path)
    full = load_cache(SAHI_OUT / 'cache' / f'full_fold{fold}.npz')
    assert ([candidate[0] for candidate in candidates] == [
        full_row[0] for full_row in full] == [int(image_info['id']) for image_info in images])
    for method, with_full in todo:
        name = config_name(size, overlap, method, with_full)
        preds, n_candidates = [], []
        for (image_id, boxes, scores, labels,
             _), (_, full_boxes, full_scores, full_labels, _) in zip(candidates, full):
            if with_full:
                boxes, scores, labels = np.concatenate([boxes, full_boxes]), np.concatenate([
                    scores, full_scores]), np.concatenate([labels, full_labels])
            n_candidates.append(len(scores))
            h, w = dims[image_id]
            merged_boxes, merged_scores, merged_labels = merge(boxes, scores, labels, method, h, w)
            assert (len(merged_boxes) == 0 or (merged_boxes[:, :2].min() >= 0
                                               and (merged_boxes[:, 2] <= w).all()
                                               and (merged_boxes[:, 3] <= h).all()))
            preds.append((image_id, as_prediction(merged_boxes, merged_scores, merged_labels)))
        metrics = coco_metrics(preds, gt)
        row = result_row(name, fold, metrics, preds,
                         dict(stage=stage, slice_size=size, overlap=overlap, merge=method,
                              full_image=with_full,
                              slices_per_img=float(np.mean([candidate[4]
                                                            for candidate in candidates])) + with_full,
                              candidates_per_img=float(np.mean(n_candidates))))
        write_json(dict(settings_id=SAHI_ID, checkpoint_sha256=checkpoint_hash, row=row,
                        runtime_s=time.time() - started), result_path(name, fold))
        print(f"  {name} fold {fold}: mAP {metrics['mAP']:.4f} | AP_small {metrics['APs']:.4f}")
    del model
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

In [ ]:
def sahi_results():
    rows = pd.DataFrame([json.loads(path.read_text())['row']
                         for path in sorted((SAHI_OUT / 'folds').glob('*_fold*.json'))])
    rows = rows.drop(columns=['ms_per_img', 'fps'], errors='ignore')
    speed = [dict(config=name, fold=record['fold'], ms_per_img=ms)
             for path in sorted((SAHI_OUT / 'speed').glob('fold*.json'))
             for record in [json.loads(path.read_text())]
             for name, ms in record['ms_per_img'].items()]
    if speed:
        rows = rows.merge(pd.DataFrame(speed), on=['config', 'fold'], how='left')
        rows['fps'] = 1000 / rows.ms_per_img
    return rows

def sahi_summary(fold_table):
    cols = [col for col in KEYS + ['ms_per_img', 'fps', 'detections_per_img'] if col in fold_table]
    assert (fold_table.groupby('config').fold.nunique()
            == len(FOLDS)).all(), 'a configuration is missing folds'
    stats = fold_table.groupby('config')[cols].agg(['mean', 'std'])
    summary = fold_table.groupby('config')[['stage', 'slice_size', 'overlap', 'merge', 'full_image',
                                            'slices_per_img']].first()
    for col in cols:
        summary[col + '_mean'], summary[col + '_std'] = stats[col, 'mean'], stats[col, 'std']
    baseline_rows = fold_table[fold_table.config == 'baseline'].set_index('fold')
    for name, config_rows in fold_table.groupby('config'):
        diff = config_rows.set_index('fold')[['mAP', 'APs']] - baseline_rows[['mAP', 'APs']]
        summary.loc[name,
                    'delta_mAP_mean'], summary.loc[name, 'delta_mAP_std'] = diff.mAP.mean(), diff.mAP.std()
        summary.loc[name,
                    'delta_APs_mean'], summary.loc[name, 'delta_APs_std'] = diff.APs.mean(), diff.APs.std()
        summary.loc[name, 'folds_mAP_improved'] = int((diff.mAP > 0).sum())
    return summary

In [ ]:
def show_sahi_table(summary, names):
    selected = summary.loc[names]
    display(pd.DataFrame({col: selected[col + '_mean'].map('{:.4f}'.format) + ' ± '
                          + selected[col + '_std'].map('{:.4f}'.format) for col in ['mAP', 'APs', 'AR']}
                         ).assign(ms_per_img=selected.ms_per_img_mean.round(1)
                         if 'ms_per_img_mean' in selected else None, slices=selected.slices_per_img.round(1),
                                  delta_mAP=selected.delta_mAP_mean.map('{:+.4f}'.format),
                                  delta_APs=selected.delta_APs_mean.map('{:+.4f}'.format),
                                  folds_improved=selected.folds_mAP_improved.astype(int)))

In [ ]:
started = time.time()
for fold in FOLDS:
    run_baseline(fold)
base_df = sahi_results().query("config == 'baseline'")
print('Baseline (re-evaluated, equal to result.json): mAP', round(base_df.mAP.mean(), 4), '| AP_small',
      round(base_df.APs.mean(), 4),
      f'| {time.time() - started:.0f} s')

Baseline (re-evaluated, equal to result.json): mAP 0.211 | AP_small 0.0757 | 0 s


### Stage №1 slice size (overlap 0.2, NMS)



In [ ]:
for size in SLICE_SIZES:
    for fold in FOLDS:
        run_sahi(fold, 'slice_size', size, STAGE1_OVERLAP, [('nms', False)])
summary = sahi_summary(sahi_results())
stage1 = [config_name(size, STAGE1_OVERLAP, 'nms', False) for size in SLICE_SIZES]
show_sahi_table(summary, ['baseline'] + stage1)
BEST_SIZE = int(summary.loc[stage1, 'mAP_mean'].idxmax().split('_')[0][1:])
print('Best slice size by mean CV mAP:', BEST_SIZE)

,mAP,APs,AR,ms_per_img,slices,delta_mAP,delta_APs,folds_improved
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,25.2,1.0,+0.0000,+0.0000,0
s512_o0.2_nms,0.1690 ± 0.0226,0.0901 ± 0.0197,0.3377 ± 0.0278,256.3,17.8,-0.0420,+0.0144,0
s640_o0.2_nms,0.1997 ± 0.0299,0.1000 ± 0.0161,0.3686 ± 0.0356,146.9,9.7,-0.0113,+0.0243,1
s768_o0.2_nms,0.2086 ± 0.0324,0.0919 ± 0.0224,0.3737 ± 0.0412,110.5,8.1,-0.0024,+0.0162,3
s1024_o0.2_nms,0.2179 ± 0.0334,0.0907 ± 0.0235,0.3774 ± 0.0421,92.3,6.0,+0.0069,+0.0151,4


Best slice size by mean CV mAP: 1024


### Stage №2 overlap (best slice size, NMS)

In [ ]:
for overlap in OVERLAPS:
    for fold in FOLDS:
        run_sahi(fold, 'overlap', BEST_SIZE, overlap, [('nms', False)])
summary = sahi_summary(sahi_results())
stage2 = [config_name(BEST_SIZE, overlap, 'nms', False) for overlap in OVERLAPS]
show_sahi_table(summary, ['baseline'] + stage2)
BEST_OVERLAP = float(summary.loc[stage2, 'mAP_mean'].idxmax().split('_')[1][1:])
print('Best overlap by mean CV mAP:', BEST_OVERLAP)

,mAP,APs,AR,ms_per_img,slices,delta_mAP,delta_APs,folds_improved
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,25.2,1.0,+0.0000,+0.0000,0
s1024_o0.1_nms,0.2183 ± 0.0341,0.0915 ± 0.0239,0.3750 ± 0.0426,67.1,4.3,+0.0073,+0.0158,4
s1024_o0.2_nms,0.2179 ± 0.0334,0.0907 ± 0.0235,0.3774 ± 0.0421,92.3,6.0,+0.0069,+0.0151,4
s1024_o0.3_nms,0.2189 ± 0.0329,0.0926 ± 0.0231,0.3795 ± 0.0430,92.2,6.0,+0.0079,+0.0169,3
s1024_o0.4_nms,0.2181 ± 0.0332,0.0931 ± 0.0235,0.3781 ± 0.0422,92.2,6.0,+0.0071,+0.0174,3


Best overlap by mean CV mAP: 0.3


### Stage №3 merging method and full-image pass

In [ ]:
STAGE3 = [('nms', False), ('greedynmm', False), ('nms', True), ('greedynmm', True)]
for fold in FOLDS:
    run_sahi(fold, 'merging', BEST_SIZE, BEST_OVERLAP, STAGE3)
fold_df = sahi_results()
summary = sahi_summary(fold_df)
stage3 = [config_name(BEST_SIZE, BEST_OVERLAP, method, full) for method, full in STAGE3]
show_sahi_table(summary, ['baseline'] + stage3)

,mAP,APs,AR,ms_per_img,slices,delta_mAP,delta_APs,folds_improved
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,25.2,1.0,+0.0000,+0.0000,0
s1024_o0.3_nms,0.2189 ± 0.0329,0.0926 ± 0.0231,0.3795 ± 0.0430,92.2,6.0,+0.0079,+0.0169,3
s1024_o0.3_greedynmm,0.1678 ± 0.0292,0.0769 ± 0.0215,0.3221 ± 0.0395,102.5,6.0,-0.0432,+0.0012,0
s1024_o0.3_nms_full,0.2232 ± 0.0340,0.0934 ± 0.0219,0.3868 ± 0.0445,117.6,7.0,+0.0122,+0.0177,5
s1024_o0.3_greedynmm_full,0.1620 ± 0.0274,0.0736 ± 0.0184,0.3185 ± 0.0392,130.5,7.0,-0.0490,-0.0020,0


### Speed measurement

For each fold, we use its checkpoint and the same 60 validation frames as in Experiment 1, along with 10 additional frames for warm-up. We test all configurations one after another on the same frames using autocast to keep the comparison conditions as similar as possible.

For SAHI, we measure the total processing time per image, including slicing, model forward passes, an additional full-image pass (if used), and merging the results.

In [ ]:
(SAHI_OUT / 'speed').mkdir(exist_ok=True)
evaluated = sahi_results().groupby('config')[['slice_size', 'overlap', 'merge', 'full_image']].first()
SPEED_CONFIGS = (['baseline'] + [
    name for name in evaluated.index
    if evaluated.loc[name, 'slice_size'] in SLICE_SIZES])

def config_params(name):
    if name == 'baseline':
        return {}
    row = evaluated.loc[name]
    return dict(size=int(row.slice_size), overlap=float(row.overlap), method=row.merge,
                full=bool(row.full_image))

In [ ]:
def run_speed(fold, names=None, folder='speed'):
    names = names or SPEED_CONFIGS
    (SAHI_OUT / folder).mkdir(exist_ok=True)
    path = SAHI_OUT / folder / f'fold{fold}.json'
    checkpoint_hash = sha256(RUNS / f'fold{fold}/best.pth')
    if path.exists():
        old = json.loads(path.read_text())
        if (old['settings_id'] == SAHI_ID and old['checkpoint_sha256'] == checkpoint_hash
            and set(names) <= set(old['ms_per_img'])):
            return old
    model, run_config, checkpoint = load_model(fold)
    frames = speed_frames(fold, fold_images(fold))
    times = {name: [] for name in names}
    for i, image in enumerate(frames):
        for name in names:
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            start = time.perf_counter()
            run_frame(model, image, **config_params(name))
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            if i >= SPEED_WARMUP:
                times[name].append(time.perf_counter() - start)
    result = dict(settings_id=SAHI_ID, checkpoint_sha256=checkpoint_hash, fold=fold, frames=SPEED_IMAGES,
                  warmup=SPEED_WARMUP,
                  device=torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu',
                  ms_per_img={name: float(np.mean(durations) * 1000) for name, durations in times.items()})
    write_json(result, path)
    del model, frames
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return result

In [ ]:
speed_table = pd.DataFrame({k: run_speed(k)['ms_per_img'] for k in FOLDS}).loc[SPEED_CONFIGS]
speed_table['mean'], speed_table['std'] = speed_table[FOLDS].mean(axis=1), speed_table[FOLDS].std(axis=1)
display(speed_table.round(1).rename(columns=lambda col: f'fold {col}' if isinstance(col, int) else col))

,fold 0,fold 1,fold 2,fold 3,fold 4,mean,std
baseline,25.2,25.3,25.3,25.2,25.0,25.2,0.1
s1024_o0.1_nms,72.0,66.3,68.7,72.3,56.3,67.1,6.5
s1024_o0.2_nms,100.8,88.8,94.3,102.3,75.1,92.3,11.0
s1024_o0.3_greedynmm,109.1,98.0,105.6,113.7,85.8,102.5,10.9
s1024_o0.3_greedynmm_full,136.6,125.7,134.0,141.9,114.1,130.5,10.9
s1024_o0.3_nms,100.7,88.8,94.3,102.2,75.2,92.2,10.9
s1024_o0.3_nms_full,126.2,114.2,119.7,127.6,100.4,117.6,11.0
s1024_o0.4_nms,100.6,88.8,94.2,102.2,75.2,92.2,10.9
s512_o0.2_nms,277.9,256.3,266.2,284.1,197.0,256.3,34.8
s640_o0.2_nms,154.3,148.0,155.1,157.8,119.3,146.9,15.9


### Results

In [ ]:
fold_df = sahi_results()
summary = sahi_summary(fold_df)
order = (['baseline'] + stage1 + [name for name in stage2
                                  if name not in stage1] + [name for name in stage3 if name not in stage2])
summary = summary.loc[order]
summary.index.name = 'config'
fold_df = fold_df.set_index('config').loc[order].reset_index()
fold_df.to_csv(SAHI_OUT / 'sahi_fold_results.csv', index=False)
summary.to_csv(SAHI_OUT / 'sahi_summary.csv')
BEST = summary.drop('baseline').mAP_mean.idxmax()
best = summary.loc[BEST]
show_sahi_table(summary, order)
print('Best SAHI configuration:', BEST)

,mAP,APs,AR,ms_per_img,slices,delta_mAP,delta_APs,folds_improved
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,25.2,1.0,+0.0000,+0.0000,0
s512_o0.2_nms,0.1690 ± 0.0226,0.0901 ± 0.0197,0.3377 ± 0.0278,256.3,17.8,-0.0420,+0.0144,0
s640_o0.2_nms,0.1997 ± 0.0299,0.1000 ± 0.0161,0.3686 ± 0.0356,146.9,9.7,-0.0113,+0.0243,1
s768_o0.2_nms,0.2086 ± 0.0324,0.0919 ± 0.0224,0.3737 ± 0.0412,110.5,8.1,-0.0024,+0.0162,3
s1024_o0.2_nms,0.2179 ± 0.0334,0.0907 ± 0.0235,0.3774 ± 0.0421,92.3,6.0,+0.0069,+0.0151,4
s1024_o0.1_nms,0.2183 ± 0.0341,0.0915 ± 0.0239,0.3750 ± 0.0426,67.1,4.3,+0.0073,+0.0158,4
s1024_o0.3_nms,0.2189 ± 0.0329,0.0926 ± 0.0231,0.3795 ± 0.0430,92.2,6.0,+0.0079,+0.0169,3
s1024_o0.4_nms,0.2181 ± 0.0332,0.0931 ± 0.0235,0.3781 ± 0.0422,92.2,6.0,+0.0071,+0.0174,3
s1024_o0.3_greedynmm,0.1678 ± 0.0292,0.0769 ± 0.0215,0.3221 ± 0.0395,102.5,6.0,-0.0432,+0.0012,0


Best SAHI configuration: s1024_o0.3_nms_full


In [ ]:
write_json(dict(SAHI_SETTINGS, settings_id=SAHI_ID, slice_sizes=SLICE_SIZES, overlaps=OVERLAPS,
                stage1_overlap=STAGE1_OVERLAP,
                stage3=[config_name(BEST_SIZE, BEST_OVERLAP, method, full) for method, full in STAGE3],
                sahi_defaults=dict(postprocess_type='GREEDYNMM', postprocess_match_metric='IOS',
                                   postprocess_match_threshold=0.5,
                                   postprocess_class_agnostic=False, perform_standard_pred=True,
                                   low_confidence_switch=f'NMS/IOU when confidence < '
                                                         f'{sahi_predict.LOW_MODEL_CONFIDENCE}'),
                slicing='sahi.slicing.get_slice_bboxes, square slices, same overlap in x and y',
                merging=dict(nms='sahi batched_nms, IOU >= 0.5, class-aware',
                             greedynmm='sahi GreedyNMMPostprocess, IOS >= 0.5, class-aware'),
                evaluation='COCOeval bbox, original annotations incl. iscrowd, maxDets=[1,10,100], held-out '
                           'fold of each checkpoint; top 500 boxes per frame after merging',
                speed='batch 1 frame, autocast, frames on GPU before timing; total time per original frame '
                      'incl. slicing, all slice passes and merging',
                checkpoints={k: dict(path=f'faster_rcnn/lr_scheduler/fold{k}/best.pth',
                                     sha256=sha256(RUNS / f'fold{k}/best.pth'))
                             for k in FOLDS}),
           SAHI_OUT / 'sahi_config.json')
best_rows = fold_df[fold_df.config == BEST].set_index('fold')
write_json(dict(config=BEST, slice_size=int(best.slice_size), overlap=float(best.overlap), merge=best.merge,
                merge_metric=MATCH[best.merge][0], merge_threshold=MATCH[best.merge][1],
                full_image_pass=bool(best.full_image),
                selected_by='highest mean 5-fold CV mAP@[.50:.95]', best_slice_size_stage1=BEST_SIZE,
                best_overlap_stage2=BEST_OVERLAP,
                cv_mean={col: float(best[col + '_mean']) for col in KEYS + ['ms_per_img', 'fps']},
                cv_std={col: float(best[col + '_std']) for col in KEYS + ['ms_per_img', 'fps']},
                baseline_cv_mean={col: float(summary.loc['baseline', col + '_mean'])
                                  for col in KEYS + ['ms_per_img', 'fps']},
                delta_vs_baseline=dict(mAP_mean=float(best.delta_mAP_mean), mAP_std=float(best.delta_mAP_std),
                                       APs_mean=float(best.delta_APs_mean), APs_std=float(best.delta_APs_std),
                                       folds_mAP_improved=int(best.folds_mAP_improved)),
                folds={int(k): {col: float(best_rows.loc[k, col]) for col in KEYS + ['ms_per_img']}
                       for k in best_rows.index}),
           SAHI_OUT / 'sahi_best_config.json')

In [ ]:
SAHI_PLOTS = SAHI_OUT / 'plots'

def save_sahi_plot(fig, name):
    fig.update_layout(template='plotly_white', font=dict(size=13))
    fig.write_html(SAHI_PLOTS / f'{name}.html', include_plotlyjs='directory', div_id=name)
    fig.show()

def baseline_line(fig, key, **kwargs):
    fig.add_hline(y=summary.loc['baseline', key + '_mean'], line_dash='dash', line_color='#4c78a8',
                  annotation_text=f"standard NMS baseline {summary.loc['baseline', key + '_mean']:.4f}",
                  **kwargs)

In [ ]:
subset = summary.loc[stage1]
fig = make_subplots(1, 2, subplot_titles=['mAP@[.50:.95]', 'AP_small'])
for col, key in enumerate(['mAP', 'APs'], 1):
    fig.add_bar(x=[f'{size} px' for size in SLICE_SIZES], y=subset[key + '_mean'],
                error_y=dict(array=subset[key + '_std']), marker_color='#f58518',
                text=[f'{value:.4f}' for value in subset[key + '_mean']], showlegend=False, row=1, col=col)
    baseline_line(fig, key, row=1, col=col)
fig.update_layout(title='Stage 1: slice size (overlap 0.2, NMS); mean ± std over 5 CV folds')
save_sahi_plot(fig, 'map_by_slice_size')

subset = summary.loc[stage2]
fig = make_subplots(1, 2, subplot_titles=['mAP@[.50:.95]', 'AP_small'])
for col, key in enumerate(['mAP', 'APs'], 1):
    fig.add_scatter(x=OVERLAPS, y=subset[key + '_mean'], error_y=dict(array=subset[key + '_std']),
                    mode='lines+markers', line_color='#e45756',
                    text=[f'{ms:.0f} ms/frame, {n_slices:.1f} slices'
                          for ms, n_slices in zip(subset.ms_per_img_mean, subset.slices_per_img)],
                    hovertemplate='overlap %{x}: %{y:.4f}<br>%{text}<extra></extra>', showlegend=False, row=1,
                    col=col)
    baseline_line(fig, key, row=1, col=col)
fig.update_xaxes(title='overlap ratio', tickvals=OVERLAPS)
fig.update_layout(title=f'Stage 2: overlap ({BEST_SIZE} px slices, NMS); mean ± std over 5 CV folds')
save_sahi_plot(fig, 'map_aps_by_overlap')

In [ ]:
subset = summary.loc[['baseline'] + stage3]
labels = (['standard NMS baseline'] + [f"{method.upper()}{' + full image' if full else ''}"
                                       for method, full in STAGE3])
fig = go.Figure()
for key, name, color in [('mAP', 'mAP@[.50:.95]', '#4c78a8'), ('APs', 'AP_small', '#f58518'),
                         ('AR', 'AR@100', '#54a24b')]:
    fig.add_bar(x=labels, y=subset[key + '_mean'], error_y=dict(array=subset[key + '_std']), name=name,
                marker_color=color,
                text=[f'{value:.4f}' for value in subset[key + '_mean']])
fig.update_layout(title=f'Stage 3: merging method and full-image pass ({BEST_SIZE} px, overlap '
                        f'{BEST_OVERLAP}); mean ± std over 5 CV folds',
                  barmode='group')
save_sahi_plot(fig, 'merging_methods')

stage_colors = {'baseline': '#4c78a8', 'slice_size': '#f58518', 'overlap': '#e45756', 'merging': '#54a24b'}
fig = go.Figure()
for stage, subset in summary.groupby('stage', sort=False):
    fig.add_scatter(x=subset.ms_per_img_mean, y=subset.mAP_mean, mode='markers+text', name=stage,
                    text=list(subset.index), textposition='top center',
                    marker=dict(size=11, color=stage_colors[stage]),
                    error_y=dict(array=subset.mAP_std, thickness=1),
                    hovertemplate='%{text}<br>%{x:.0f} ms/frame<br>mAP %{y:.4f}<extra></extra>')
fig.update_layout(title='Latency per original frame vs mean CV mAP (all configurations)',
                  xaxis_title='ms / frame (batch 1, AMP)',
                  yaxis_title='mAP@[.50:.95]')
save_sahi_plot(fig, 'speed_vs_map')

fig = make_subplots(1, 2, subplot_titles=['mAP@[.50:.95]', 'AP_small'])
for config, name, color in [('baseline', 'standard NMS baseline', '#4c78a8'),
                            (BEST, f'SAHI {BEST}', '#e45756')]:
    subset = fold_df[fold_df.config == config]
    for col, key in enumerate(['mAP', 'APs'], 1):
        fig.add_bar(x=[f'fold {k}' for k in subset.fold] + ['mean'],
                    y=list(subset[key]) + [subset[key].mean()], name=name, marker_color=color,
                    error_y=dict(array=[0] * len(subset) + [subset[key].std()]), showlegend=col == 1, row=1,
                    col=col)
fig.update_layout(title=f'Baseline vs best SAHI configuration per fold (CV held-out folds)', barmode='group')
save_sahi_plot(fig, 'baseline_vs_best_sahi')

In [ ]:
def sahi_figure(rgb, image_info, panels, title, windows):
    h, w = rgb.shape[:2]
    anns = [annotation for annotation in coco_train['annotations']
            if annotation['image_id'] == image_info['id']]
    xyxy = lambda annotation: [annotation['bbox'][0], annotation['bbox'][1],
                               annotation['bbox'][0] + annotation['bbox'][2],
                               annotation['bbox'][1] + annotation['bbox'][3]]
    small = Image.fromarray(rgb)
    small.thumbnail((1280, 1280))
    buffer = io.BytesIO()
    small.save(buffer, 'JPEG', quality=85)
    source = 'data:image/jpeg;base64,' + base64.b64encode(buffer.getvalue()).decode()
    fig = make_subplots(1, len(panels), horizontal_spacing=.01,
                        subplot_titles=[f"{name}: {int((pred['scores'] >= .3).sum())} boxes with score ≥ 0.3"
                                        for name, pred in panels.items()])
    def outline(boxes, color, width=1.2, dash=None, text=None, name=''):
        xs, ys, hover = [], [], []
        for i, (x1, y1, x2, y2) in enumerate(boxes):
            xs += [x1, x2, x2, x1, x1, None]
            ys += [y1, y1, y2, y2, y1, None]
            hover += [text[i] if text else name] * 5 + [None]
        return go.Scatter(x=xs, y=ys, mode='lines', line=dict(color=color, width=width, dash=dash),
                          text=hover,
                          hoverinfo='text', showlegend=False)
    for col, (name, pred) in enumerate(panels.items(), 1):
        keep = pred['scores'] >= .3
        text = [f"{CATS[LABEL_TO_CAT[int(label)]]} {score:.2f}"
                for label, score in zip(pred['labels'][keep], pred['scores'][keep])]
        if name.startswith('SAHI'):
            fig.add_trace(outline(windows, 'white', .8, 'dot', name='slice'), 1, col)
        fig.add_trace(outline([xyxy(annotation) for annotation in anns if not annotation.get('iscrowd', 0)],
                              'lime', name='ground truth'), 1, col)
        fig.add_trace(outline([xyxy(annotation) for annotation in anns if annotation.get('iscrowd', 0)],
                              'yellow', name='ignored region'), 1, col)
        fig.add_trace(outline(pred['boxes'][keep].tolist(), 'red', text=text), 1, col)
        fig.add_layout_image(source=source, xref=f'x{col}', yref=f'y{col}', x=0, y=0, sizex=w, sizey=h,
                             xanchor='left', yanchor='top', sizing='stretch', layer='below')
    fig.update_xaxes(range=[0, w], visible=False)
    fig.update_yaxes(range=[h, 0], visible=False)
    fig.update_layout(
        title=title
        + ' · green = ground truth, red = prediction (score ≥ 0.3), yellow = ignored, white dots = slices',
                      width=1500, height=int(750 * h / w) + 120, margin=dict(l=10, r=10, t=90, b=10),
                      template='plotly_white')
    return fig

In [ ]:
base_folds = fold_df[fold_df.config == 'baseline'].set_index('fold').mAP
rep_fold = int(base_folds.sort_values().index[len(base_folds) // 2])
images = fold_images(rep_fold)
n_objects = (pd.Series([annotation['image_id'] for annotation in coco_train['annotations']
                        if not annotation.get('iscrowd', 0)]).value_counts())
examples = [('most_objects', max(images, key=lambda image_info: n_objects.get(image_info['id'], 0))),
            ('random', images[int(np.random.default_rng(SEED).integers(len(images)))])]
size, overlap, method, full = int(best.slice_size), float(best.overlap), best.merge, bool(best.full_image)
model, run_config, checkpoint = load_model(rep_fold)
saved = []
for label, image_info in examples:
    rgb = read_images([image_info])[0]
    image = to_tensor(rgb).to(DEVICE)
    with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=AMP):
        p = model([image])[0]
    base_pred = {k: value.float().cpu().numpy() if k != 'labels' else value.cpu().numpy()
                 for k, value in p.items()}
    b, s, l, _ = slice_predict(model, image, size, overlap)
    if full:
        b, s, l = np.concatenate([b, base_pred['boxes']]), np.concatenate([
            s, base_pred['scores']]), np.concatenate([l, base_pred['labels']])
    sahi_pred = as_prediction(*merge(b, s, l, method, *rgb.shape[:2]))
    windows = get_slice_bboxes(rgb.shape[0], rgb.shape[1], size, size, False, overlap, overlap)
    fig = sahi_figure(rgb, image_info, {'Standard NMS (full frame)': base_pred, f'SAHI {BEST}': sahi_pred},
                      f"Fold {rep_fold} held-out frame {image_info['file_name']} "
                      f"({n_objects.get(image_info['id'], 0)} objects)",
                      windows)
    path = SAHI_OUT / 'visualizations' / f"fold{rep_fold}_{label}_{image_info['id']}.html"
    fig.write_html(path, include_plotlyjs='directory')
    saved.append(path.name)
del model
print('Saved to', SAHI_OUT / 'visualizations', ':', saved)

Saved to E:\ucu_hw2\faster_rcnn\inference_experiments\sahi\visualizations : ['fold1_most_objects_1128.html', 'fold1_random_18517.html']


### Check 1280 px slices

In the first stage, the largest slice size of 1024 px gave the best result, so we also test 1280 px to see if this trend continues.

In [ ]:
CHECK_SIZE = 1280
for fold in FOLDS:
    run_sahi(fold, 'check', CHECK_SIZE, STAGE1_OVERLAP, [('nms', False)])
evaluated = sahi_results().groupby('config')[['slice_size', 'overlap', 'merge', 'full_image']].first()
check = ['baseline', config_name(1024, STAGE1_OVERLAP, 'nms', False),
         config_name(CHECK_SIZE, STAGE1_OVERLAP, 'nms', False)]
check_speed = pd.DataFrame([dict(config=name, fold=k, ms_per_img=ms) for k in FOLDS
                            for name, ms in run_speed(k, check, 'speed_check_1280')['ms_per_img'].items()])
check_df = (sahi_results().query('config in @check').drop(
    columns=['ms_per_img', 'fps'], errors='ignore').merge(check_speed, on=['config', 'fold']))
check_df['fps'] = 1000 / check_df.ms_per_img
check_summary = sahi_summary(check_df).loc[check]
ref = check_df[check_df.config == check[1]].set_index('fold').mAP
check_summary['delta_mAP_vs_1024'] = [(check_df[check_df.config == name].set_index('fold').mAP - ref).mean()
                                      for name in check]
check_df.to_csv(SAHI_OUT / 'sahi_check_1280_fold_results.csv', index=False)
check_summary.to_csv(SAHI_OUT / 'sahi_check_1280_summary.csv')
show_sahi_table(check_summary, check)
display(check_df.pivot(index='fold', columns='config', values='mAP')[check].round(4))
print(f"1280 px vs 1024 px: ΔmAP {check_summary.loc[check[2], 'delta_mAP_vs_1024']:+.4f}, "
      f"slices per frame {check_summary.loc[check[2], 'slices_per_img']:.1f} vs "
      f"{check_summary.loc[check[1], 'slices_per_img']:.1f}")

  s1280_o0.2_nms fold 0: mAP 0.2439 | AP_small 0.1085


  s1280_o0.2_nms fold 1: mAP 0.2239 | AP_small 0.1004


  s1280_o0.2_nms fold 2: mAP 0.2371 | AP_small 0.0936


  s1280_o0.2_nms fold 3: mAP 0.2119 | AP_small 0.0677


  s1280_o0.2_nms fold 4: mAP 0.1595 | AP_small 0.0529


,mAP,APs,AR,ms_per_img,slices,delta_mAP,delta_APs,folds_improved
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,26.8,1.0,+0.0000,+0.0000,0
s1024_o0.2_nms,0.2179 ± 0.0334,0.0907 ± 0.0235,0.3774 ± 0.0421,97.7,6.0,+0.0069,+0.0151,4
s1280_o0.2_nms,0.2153 ± 0.0335,0.0846 ± 0.0234,0.3676 ± 0.0436,57.1,3.2,+0.0042,+0.0089,4


config,baseline,s1024_o0.2_nms,s1280_o0.2_nms
fold,,,
0,0.2377,0.2525,0.2439
1,0.2208,0.2211,0.2239
2,0.2380,0.2351,0.2371
3,0.2015,0.2174,0.2119
4,0.1571,0.1636,0.1595


1280 px vs 1024 px: ΔmAP -0.0027, slices per frame 3.2 vs 6.0


## Experiment №3 Test-Time Augmentation (TTA)

In this experiment, we check whether we can improve Faster R-CNN performance by using several versions of the same image with different scales and flips, and then combining their predictions.

As the baseline, we use the original model with standard NMS (mAP 0.2110, AP_small 0.0757) and check its results again.

During training, the model saw horizontal flips but not vertical flips, so we also want to test whether both types help improve performance.

We run the experiment in four stages:

- Test horizontal and vertical flips without changing the scale.

- Compare scales from 0.5× to 2.0× and their combinations.

- Combine the best flip and scale settings.

- Compare NMS and Weighted Boxes Fusion for the best configurations.

In [ ]:
%pip -q install ensemble-boxes

In [ ]:
import subprocess
from importlib.metadata import version
from ensemble_boxes import weighted_boxes_fusion

In [ ]:
TTA_OUT = ROOT / 'faster_rcnn/inference_experiments/tta'
SCALES = [0.5, 0.75, 1.0, 1.25, 1.5, 2.0]
TTA_BATCH = {0.5: 4, 0.75: 4, 1.0: 4, 1.25: 4, 1.5: 2, 2.0: 1}
NMS_IOU = INFERENCE['nms_thr']
WBF_IOU = 0.55
DET_PER_IMG = INFERENCE['det_per_img']
TTA_SETTINGS = dict(torch=torch.__version__, torchvision=torchvision.__version__,
                    ensemble_boxes=version('ensemble-boxes'),
                    scales=SCALES, batch=TTA_BATCH, nms_iou=NMS_IOU, det_per_img=DET_PER_IMG, amp=AMP,
                    coco_max_dets=MAX_DETS,
                    wbf=dict(iou_thr=WBF_IOU, skip_box_thr=0.0, conf_type='avg', weights='equal'),
                    speed_images=SPEED_IMAGES, speed_warmup=SPEED_WARMUP, seed=SEED, folds_sha=FOLDS_SHA,
                    inference=INFERENCE)
TTA_ID = hashlib.sha256(json.dumps(TTA_SETTINGS, sort_keys=True).encode()).hexdigest()[:16]
for subfolder in ['views', 'folds', 'speed', 'plots', 'visualizations']:
    (TTA_OUT / subfolder).mkdir(parents=True, exist_ok=True)
CHECKPOINT_SHA = {k: sha256(RUNS / f'fold{k}/best.pth') for k in FOLDS}
print('torch', torch.__version__, '| torchvision', torchvision.__version__, '| ensemble-boxes',
      version('ensemble-boxes'))

torch 2.11.0+cu128 | torchvision 0.26.0+cu128 | ensemble-boxes 1.0.10


### Input resolution inside Faster R-CNN

Faster R-CNN automatically resizes each image using GeneralizedRCNNTransform, so simply resizing an image before passing it to the model does not help, because the model resizes it back to its standard size.

To test different scales, we change min_size and max_size inside the model for each forward pass by multiplying the original values of 720 and 1280 by the selected scale factor. We then restore the original settings without changing the model weights or any other parameters.

The model automatically maps the detected boxes back to the original image coordinates, so no additional processing is needed. The table below shows the image sizes for each scale to confirm that all configurations use different resolutions.

In [ ]:
def set_scale(model, scale):
    model.transform.min_size = (round(INFERENCE['min_size'] * scale),)
    model.transform.max_size = round(INFERENCE['max_size'] * scale)

In [ ]:
model, run_config, checkpoint = load_model(FOLDS[0])
assert (model.transform.min_size == (INFERENCE['min_size'],)
        and model.transform.max_size == INFERENCE['max_size'])
frame_sizes = (pd.DataFrame([image_info for k in FOLDS for image_info in fold_images(k)]).groupby([
    'width', 'height']).size().rename('frames'))
rows = []
with torch.inference_mode():
    for (w, h), n in frame_sizes.items():
        row = dict(frame=f'{w}×{h}', frames=n)
        for s in SCALES:
            set_scale(model, s)
            input_h, input_w = model.transform([torch.zeros(3, h, w, device=DEVICE)])[0].image_sizes[0]
            row[f'{s}×'] = f'{input_w}×{input_h}'
        rows.append(row)
    set_scale(model, 1.0)
    w, h = frame_sizes.idxmax()
    full = model.transform([torch.zeros(3, h, w, device=DEVICE)])[0].image_sizes[0]
    half = model.transform([torch.zeros(3, h // 2, w // 2, device=DEVICE)])[0].image_sizes[0]
del model
input_sizes = pd.DataFrame(rows).set_index('frame')
assert all(len(set(row[f'{scale}×'] for scale in SCALES)) == len(SCALES)
           for _, row in input_sizes.iterrows()), 'two scales give the same resolution'
input_sizes.to_csv(TTA_OUT / 'tta_input_sizes.csv')
display(input_sizes)
print(f'A {w}×{h} frame resized to 0.5× before the model is still processed at {half[1]}×{half[0]} '
      f'(without resizing: {full[1]}×{full[0]}), so external resizing is undone by the transform.')

,frames,0.5×,0.75×,1.0×,1.25×,1.5×,2.0×
frame,,,,,,,
1344×756,1134,640×360,960×540,1280×720,1600×900,1920×1080,2560×1440
1360×765,193,640×360,960×540,1280×720,1600×900,1920×1080,2560×1440
1904×1071,2593,640×360,960×540,1280×720,1600×900,1920×1080,2560×1440
1920×1080,145,640×360,960×540,1280×720,1600×900,1920×1080,2560×1440
2688×1512,707,640×360,960×540,1280×720,1600×900,1920×1080,2560×1440
2720×1530,96,640×360,960×540,1280×720,1600×900,1920×1080,2560×1440


A 1904×1071 frame resized to 0.5× before the model is still processed at 1280×719 (without resizing: 1280×720), so external resizing is undone by the transform.


### Views, box restoration and merging

We label each image view as flip@scale, where none@1.0 means the original image, h@1.0 means a horizontally flipped image, and none@1.5 means the image at 1.5× scale. We run the model with standard NMS settings for each view and save the predictions to avoid processing the same images again.

After flipping, we map the detected boxes back to the original image coordinates and check that this transformation is correct by comparing the horizontal flip results with the baseline.

To combine predictions, we test two methods: NMS with IoU 0.5, which removes duplicates, and Weighted Boxes Fusion (WBF) with IoU 0.55, which combines similar boxes from different views based on their coordinates and scores. In WBF, all views have equal weight, and boxes detected in only one view receive a lower score.

After merging, we keep a maximum of 500 detections per image.


In [ ]:
def view_name(flip, scale):
    return f'{flip}@{scale}'

def parse_view(view):
    flip, scale = view.split('@')
    return flip, float(scale)

def flip_image(image, flip):
    return image.flip(-1) if flip == 'h' else image.flip(-2) if flip == 'v' else image

def unflip(boxes, flip, h, w):
    x1, y1, x2, y2 = boxes.unbind(1)
    if flip == 'h':
        return torch.stack([w - x2, y1, w - x1, y2], 1)
    if flip == 'v':
        return torch.stack([x1, h - y2, x2, h - y1], 1)
    return boxes

box = torch.tensor([[10., 20., 30., 50.]])
assert unflip(torch.tensor([[170., 20., 190., 50.]]), 'h', 100, 200).equal(box)
assert unflip(torch.tensor([[10., 50., 30., 80.]]), 'v', 100, 200).equal(box)

In [ ]:
def write_json(obj, path):
    tmp_path = path.with_suffix('.tmp')
    tmp_path.write_text(json.dumps(obj, indent=2))
    os.replace(tmp_path, path)

def view_path(fold, view):
    return TTA_OUT / 'views' / f'fold{fold}_{view}.npz'

def oom_path(fold, view):
    return TTA_OUT / 'views' / f'fold{fold}_{view}.oom.json'

def save_view(path, rows):
    tmp_path = path.with_name(path.stem + '.tmp.npz')
    np.savez(tmp_path, ids=np.array([row[0] for row in rows]), counts=np.array([len(row[2]) for row in rows]),
             boxes=np.concatenate([row[1] for row in rows]).reshape(-1, 4).astype(np.float32),
             scores=np.concatenate([row[2] for row in rows]), labels=np.concatenate([row[3] for row in rows]))
    os.replace(tmp_path, path)

def load_view(path):
    data = np.load(path)
    ends = np.cumsum(data['counts'])
    starts = ends - data['counts']
    boxes, scores, labels = data['boxes'], data['scores'], data['labels']
    return [(int(i), boxes[start:end], scores[start:end], labels[start:end])
            for i, start, end in zip(data['ids'], starts, ends)]

In [ ]:
def compute_views(fold, views):
    todo = [view for view in views
            if not view_path(fold, view).exists() and not oom_path(fold, view).exists()]
    if not todo:
        return
    model, run_config, checkpoint = load_model(fold)
    images = fold_images(fold)
    for view in todo:
        flip, scale = parse_view(view)
        set_scale(model, scale)
        rows = []
        try:
            with torch.inference_mode():
                for chunk, image_tensors in batches(images, TTA_BATCH[scale]):
                    image_tensors = [image.to(DEVICE, non_blocking=True) for image in image_tensors]
                    with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                        preds = model([flip_image(image, flip) for image in image_tensors])
                    for info, image, pred in zip(chunk, image_tensors, preds):
                        boxes = unflip(pred['boxes'].float(), flip, *image.shape[-2:])
                        rows.append((int(info['id']), boxes.cpu().numpy(),
                                     pred['scores'].float().cpu().numpy(), pred['labels'].cpu().numpy()))
        except torch.cuda.OutOfMemoryError:
            write_json(dict(fold=fold, view=view, batch=TTA_BATCH[scale], error='CUDA out of memory'),
                       oom_path(fold, view))
            print(f'  fold {fold} {view}: CUDA out of memory, not evaluated')
            continue
        finally:
            set_scale(model, 1.0)
            if DEVICE.type == 'cuda':
                torch.cuda.empty_cache()
        save_view(view_path(fold, view), rows)
        print(f'  fold {fold} {view}: predicted {len(rows)} frames')
    del model

In [ ]:
def tta_merge(parts, method, h, w):
    if len(parts) == 1:
        return parts[0]
    if method == 'nms':
        boxes, scores, labels = (np.concatenate(x) for x in zip(*parts))
        keep = (box_ops.batched_nms(torch.from_numpy(boxes), torch.from_numpy(scores),
                                    torch.from_numpy(labels), NMS_IOU).numpy())
    else:
        size = np.array([w, h, w, h], dtype=np.float32)
        boxes, scores, labels = weighted_boxes_fusion([part[0] / size for part in parts],
                                                      [part[1] for part in parts],
                                                      [part[2] for part in parts],
                                        iou_thr=WBF_IOU, skip_box_thr=0.0, conf_type='avg')
        boxes, scores, labels = (
            boxes * size).astype(np.float32), scores.astype(np.float32), labels.astype(np.int64)
        keep = np.argsort(-scores, kind='stable')
    keep = keep[:DET_PER_IMG]
    return boxes[keep], scores[keep], labels[keep]

In [ ]:
CONFIGS = {}

def add_config(name, views, method='nms', stage=''):
    CONFIGS[name] = (list(views), method, stage)
    return name

gt_cache = {}
def fold_gt(fold):
    if fold not in gt_cache:
        gt_cache[fold] = make_gt(coco_train, [image_info['id'] for image_info in fold_images(fold)])
    return gt_cache[fold]

In [ ]:
def evaluate_config(name, fold):
    views, method, stage = CONFIGS[name]
    merge_label = method if len(views) > 1 else 'none'
    path = TTA_OUT / 'folds' / f'{name}_fold{fold}.json'
    if path.exists():
        old = json.loads(path.read_text())
        if (old['settings_id'] == TTA_ID and old['checkpoint_sha256'] == CHECKPOINT_SHA[fold]
                and old['row']['views'] == '+'.join(views) and old['row']['merge'] == merge_label):
            return 'saved'
    missing = [view for view in views if not view_path(fold, view).exists()]
    if missing:
        return 'missing views: ' + ', '.join(missing)
    images = fold_images(fold)
    data = [load_view(view_path(fold, view)) for view in views]
    assert all([row[0] for row in view_rows] == [int(image_info['id']) for image_info in images]
               for view_rows in data)
    preds = []
    for k, image_info in enumerate(images):
        h, w = image_info['height'], image_info['width']
        boxes, scores, labels = tta_merge([view_rows[k][1:] for view_rows in data], method, h, w)
        assert (
            len(boxes) == 0
            or (boxes.min() >= -.5 and boxes[:, [0, 2]].max() <= w + .5 and boxes[:, [1, 3]].max() <= h + .5))
        preds.append((int(image_info['id']), dict(boxes=boxes, scores=scores, labels=labels)))
    metrics = coco_metrics(preds, fold_gt(fold))
    row = dict(config=name, stage=stage, views='+'.join(views), n_views=len(views), merge=merge_label,
               fold=fold, **metrics,
               detections_per_img=float(np.mean([len(pred['scores']) for _, pred in preds])))
    extra = {}
    if name == 'baseline':
        extra['baseline_max_abs_diff_vs_stored'] = max(abs(metrics[k] - stored[fold]['metrics'][k])
                                                       for k in KEYS)
        assert extra['baseline_max_abs_diff_vs_stored'] == 0, 'baseline differs from result.json'
    write_json(dict(settings_id=TTA_ID, checkpoint_sha256=CHECKPOINT_SHA[fold], row=row, **extra), path)
    return 'evaluated'

In [ ]:
def run_stage(names):
    views = list(dict.fromkeys(view for name in names for view in CONFIGS[name][0]))
    status = {}
    for fold in FOLDS:
        compute_views(fold, views)
        for name in names:
            status[name, fold] = evaluate_config(name, fold)
    for (name, fold), state in status.items():
        if state.startswith('missing'):
            print(f'NOT EVALUATED: {name} fold {fold} ({state})')

In [ ]:
def tta_results():
    rows = pd.DataFrame([json.loads(path.read_text())['row']
                         for path in sorted((TTA_OUT / 'folds').glob('*_fold*.json'))])
    rows = rows[rows.config.isin(CONFIGS)]
    speed = [dict(config=name, fold=record['fold'], ms_per_img=ms)
             for path in sorted((TTA_OUT / 'speed').glob('fold*.json'))
             for record in [json.loads(path.read_text())] for name, ms in record['ms_per_img'].items()]
    if speed:
        rows = rows.merge(pd.DataFrame(speed), on=['config', 'fold'], how='left')
        rows['fps'] = 1000 / rows.ms_per_img
    return rows

def tta_summary(fold_table):
    complete = fold_table.groupby('config').fold.nunique() == len(FOLDS)
    fold_table = fold_table[fold_table.config.isin(complete[complete].index)]
    cols = [col for col in KEYS + ['ms_per_img', 'fps', 'detections_per_img'] if col in fold_table]
    stats = fold_table.groupby('config')[cols].agg(['mean', 'std'])
    summary = fold_table.groupby('config')[['stage', 'views', 'n_views', 'merge']].first()
    for col in cols:
        summary[col + '_mean'], summary[col + '_std'] = stats[col, 'mean'], stats[col, 'std']
    baseline_rows = fold_table[fold_table.config == 'baseline'].set_index('fold')
    for name, config_rows in fold_table.groupby('config'):
        diff = config_rows.set_index('fold')[['mAP', 'APs']] - baseline_rows[['mAP', 'APs']]
        summary.loc[name,
                    'delta_mAP_mean'], summary.loc[name, 'delta_mAP_std'] = diff.mAP.mean(), diff.mAP.std()
        summary.loc[name,
                    'delta_APs_mean'], summary.loc[name, 'delta_APs_std'] = diff.APs.mean(), diff.APs.std()
        summary.loc[name, 'folds_mAP_improved'] = int((diff.mAP > 0).sum())
    return summary

In [ ]:
def tta_show(summary, names):
    selected = summary.loc[[name for name in names if name in summary.index]]
    table = pd.DataFrame({col: selected[col + '_mean'].map('{:.4f}'.format) + ' ± '
                          + selected[col + '_std'].map('{:.4f}'.format) for col in ['mAP', 'APs', 'AR']})
    table['views'], table['merge'] = selected.views, selected['merge']
    if 'ms_per_img_mean' in selected:
        table['ms_per_img'] = selected.ms_per_img_mean.round(1)
    table['delta_mAP'] = selected.delta_mAP_mean.map('{:+.4f}'.format)
    table['delta_APs'] = selected.delta_APs_mean.map('{:+.4f}'.format)
    table['folds_improved'] = selected.folds_mAP_improved.astype(int)
    display(table)

### Experiment  flips

In this experiment, we compare the baseline with combinations of the original image and its horizontal, vertical, or both flips, merging the predictions using NMS.

We also test flip_h_only and flip_v_only separately to check whether the boxes are correctly mapped back to the original image coordinates, but these variants are not considered when selecting the best configuration.

In [ ]:
add_config('baseline', ['none@1.0'], stage='baseline')
add_config('flip_h_only', ['h@1.0'], stage='check')
add_config('flip_v_only', ['v@1.0'], stage='check')
FLIPS = {'flip_h': ['none', 'h'], 'flip_v': ['none', 'v'], 'flip_hv': ['none', 'h', 'v']}
for name, flips in FLIPS.items():
    add_config(name, [view_name(flip, 1.0) for flip in flips], stage='A_flip')
run_stage(['baseline', 'flip_h_only', 'flip_v_only', *FLIPS])
summary = tta_summary(tta_results())
print('Baseline (re-evaluated, equal to result.json): mAP', round(summary.loc['baseline', 'mAP_mean'], 4),
      '| AP_small', round(summary.loc['baseline', 'APs_mean'], 4))
tta_show(summary, ['baseline', 'flip_h_only', 'flip_v_only', *FLIPS])
BEST_FLIP = summary.loc[['baseline', *FLIPS], 'mAP_mean'].idxmax()
FLIP_SET = FLIPS.get(BEST_FLIP, ['none'])
print('Best flip configuration:', BEST_FLIP, FLIP_SET)

Baseline (re-evaluated, equal to result.json): mAP 0.211 | AP_small 0.0757


,mAP,APs,AR,views,merge,delta_mAP,delta_APs,folds_improved
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,none@1.0,none,+0.0000,+0.0000,0
flip_h_only,0.2077 ± 0.0326,0.0765 ± 0.0280,0.3453 ± 0.0430,h@1.0,none,-0.0033,+0.0008,0
flip_v_only,0.0540 ± 0.0075,0.0238 ± 0.0063,0.1475 ± 0.0153,v@1.0,none,-0.1570,-0.0519,0
flip_h,0.2157 ± 0.0340,0.0804 ± 0.0286,0.3631 ± 0.0425,none@1.0+h@1.0,nms,+0.0047,+0.0048,5
flip_v,0.2014 ± 0.0298,0.0736 ± 0.0238,0.3511 ± 0.0345,none@1.0+v@1.0,nms,-0.0096,-0.0020,0
flip_hv,0.2085 ± 0.0308,0.0788 ± 0.0260,0.3678 ± 0.0371,none@1.0+h@1.0+v@1.0,nms,-0.0026,+0.0031,1


Best flip configuration: flip_h ['none', 'h']


### Experiment scales

First, we test each scale separately, with 1.0× as the baseline, then combine 1.0× with each other scale, and finally test 1.0× together with the two additional scales that gave the highest mean CV mAP in the previous tests. We use NMS to merge predictions in all cases.

In [ ]:
SINGLES = [add_config(f'scale_{scale}', [view_name('none', scale)], stage='B_scale') for scale in SCALES
           if scale != 1.0]
PAIRS = {scale: add_config(f'ms_1.0+{scale}', ['none@1.0', view_name('none', scale)], stage='B_multiscale')
         for scale in [0.75, 1.25, 1.5, 2.0]}
run_stage(SINGLES + list(PAIRS.values()))
summary = tta_summary(tta_results())
tta_show(summary, ['baseline'] + SINGLES)
ranked = sorted([scale for scale, name in PAIRS.items() if name in summary.index],
                key=lambda scale: -summary.loc[PAIRS[scale], 'mAP_mean'])
TOP2_SCALES = sorted(ranked[:2])
TRIPLE = add_config('ms_1.0+' + '+'.join(map(str, TOP2_SCALES)),
                    ['none@1.0'] + [view_name('none', scale) for scale in TOP2_SCALES], stage='B_multiscale')
run_stage([TRIPLE])
summary = tta_summary(tta_results())
MULTISCALE = [name for name in list(PAIRS.values()) + [TRIPLE] if name in summary.index]
tta_show(summary, ['baseline'] + MULTISCALE)
BEST_MULTISCALE, SECOND_MULTISCALE = summary.loc[MULTISCALE,
                                                 'mAP_mean'].sort_values(ascending=False).index[:2]
print('Two best additional scales:', TOP2_SCALES, '| best multi-scale:', BEST_MULTISCALE, '| second:',
      SECOND_MULTISCALE)

,mAP,APs,AR,views,merge,delta_mAP,delta_APs,folds_improved
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,none@1.0,none,+0.0000,+0.0000,0
scale_0.5,0.0992 ± 0.0190,0.0168 ± 0.0091,0.1859 ± 0.0287,none@0.5,none,-0.1118,-0.0589,0
scale_0.75,0.1703 ± 0.0257,0.0486 ± 0.0204,0.2904 ± 0.0337,none@0.75,none,-0.0407,-0.0270,0
scale_1.25,0.2228 ± 0.0356,0.0911 ± 0.0262,0.3663 ± 0.0402,none@1.25,none,+0.0118,+0.0154,5
scale_1.5,0.2222 ± 0.0356,0.0959 ± 0.0221,0.3692 ± 0.0376,none@1.5,none,+0.0112,+0.0202,5
scale_2.0,0.2031 ± 0.0362,0.0842 ± 0.0184,0.3565 ± 0.0378,none@2.0,none,-0.0079,+0.0085,2


,mAP,APs,AR,views,merge,delta_mAP,delta_APs,folds_improved
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,none@1.0,none,+0.0000,+0.0000,0
ms_1.0+0.75,0.2093 ± 0.0321,0.0747 ± 0.0274,0.3534 ± 0.0400,none@1.0+none@0.75,nms,-0.0017,-0.0010,2
ms_1.0+1.25,0.2282 ± 0.0357,0.0909 ± 0.0261,0.3802 ± 0.0402,none@1.0+none@1.25,nms,+0.0172,+0.0152,5
ms_1.0+1.5,0.2319 ± 0.0368,0.0976 ± 0.0233,0.3882 ± 0.0392,none@1.0+none@1.5,nms,+0.0209,+0.0219,5
ms_1.0+2.0,0.2268 ± 0.0390,0.0917 ± 0.0192,0.3918 ± 0.0393,none@1.0+none@2.0,nms,+0.0158,+0.0160,5
ms_1.0+1.25+1.5,0.2343 ± 0.0368,0.0997 ± 0.0230,0.3961 ± 0.0390,none@1.0+none@1.25+none@1.5,nms,+0.0233,+0.0240,5


Two best additional scales: [1.25, 1.5] | best multi-scale: ms_1.0+1.25+1.5 | second: ms_1.0+1.5


### Experiment flips + scales
In this experiment, we combine the best flip and scale configurations in two ways: A+B, where flips are applied only at 1.0× scale, and A@B, where each flip is tested at every selected scale.

We also test a combination with the second-best multi-scale set if it improves the baseline. If none of the flip configurations improves the baseline, we skip this experiment.

In [ ]:
STAGE_C = []
if BEST_FLIP == 'baseline':
    print('No flip configuration improved the baseline; experiment C is skipped.')
else:
    union = lambda multiscale_config: list(dict.fromkeys([
        view_name(flip, 1.0) for flip in FLIP_SET] + CONFIGS[multiscale_config][0]))
    STAGE_C.append(add_config(f'{BEST_FLIP}+{BEST_MULTISCALE}', union(BEST_MULTISCALE),
                              stage='C_flip_multiscale'))
    STAGE_C.append(add_config(
        f'{BEST_FLIP}@{BEST_MULTISCALE}',
        [view_name(flip, parse_view(view)[1]) for view in CONFIGS[BEST_MULTISCALE][0] for flip in FLIP_SET],
                              stage='C_flip_multiscale'))
    if summary.loc[SECOND_MULTISCALE, 'delta_mAP_mean'] > 0:
        STAGE_C.append(add_config(f'{BEST_FLIP}+{SECOND_MULTISCALE}', union(SECOND_MULTISCALE),
                                  stage='C_flip_multiscale'))
    run_stage(STAGE_C)
    summary = tta_summary(tta_results())
    tta_show(summary, ['baseline', BEST_FLIP, BEST_MULTISCALE, SECOND_MULTISCALE] + STAGE_C)

,mAP,APs,AR,views,merge,delta_mAP,delta_APs,folds_improved
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,none@1.0,none,+0.0000,+0.0000,0
flip_h,0.2157 ± 0.0340,0.0804 ± 0.0286,0.3631 ± 0.0425,none@1.0+h@1.0,nms,+0.0047,+0.0048,5
ms_1.0+1.25+1.5,0.2343 ± 0.0368,0.0997 ± 0.0230,0.3961 ± 0.0390,none@1.0+none@1.25+none@1.5,nms,+0.0233,+0.0240,5
ms_1.0+1.5,0.2319 ± 0.0368,0.0976 ± 0.0233,0.3882 ± 0.0392,none@1.0+none@1.5,nms,+0.0209,+0.0219,5
flip_h+ms_1.0+1.25+1.5,0.2348 ± 0.0368,0.1008 ± 0.0236,0.4010 ± 0.0391,none@1.0+h@1.0+none@1.25+none@1.5,nms,+0.0238,+0.0251,5
flip_h@ms_1.0+1.25+1.5,0.2377 ± 0.0370,0.1041 ± 0.0210,0.4101 ± 0.0393,none@1.0+h@1.0+none@1.25+h@1.25+none@1.5+h@1.5,nms,+0.0267,+0.0284,5
flip_h+ms_1.0+1.5,0.2329 ± 0.0368,0.0994 ± 0.0242,0.3954 ± 0.0394,none@1.0+h@1.0+none@1.5,nms,+0.0219,+0.0238,5


### Experiment NMS vs WBF

The three multi-view configurations with the highest mean CV mAP so far are merged again with WBF, from the same cached view predictions.

In [ ]:
NMS_TOP = summary[(summary.n_views > 1)
                  & (summary['merge'] == 'nms')].mAP_mean.sort_values(ascending=False).index[:3]
WBF_CONFIGS = [add_config(name + '_wbf', CONFIGS[name][0], 'wbf', stage='D_merging') for name in NMS_TOP]
run_stage(WBF_CONFIGS)
summary = tta_summary(tta_results())
tta_show(summary, ['baseline'] + [x for pair in zip(NMS_TOP, WBF_CONFIGS) for x in pair])

,mAP,APs,AR,views,merge,delta_mAP,delta_APs,folds_improved
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,none@1.0,none,+0.0000,+0.0000,0
flip_h@ms_1.0+1.25+1.5,0.2377 ± 0.0370,0.1041 ± 0.0210,0.4101 ± 0.0393,none@1.0+h@1.0+none@1.25+h@1.25+none@1.5+h@1.5,nms,+0.0267,+0.0284,5
flip_h@ms_1.0+1.25+1.5_wbf,0.2495 ± 0.0382,0.1131 ± 0.0236,0.4214 ± 0.0400,none@1.0+h@1.0+none@1.25+h@1.25+none@1.5+h@1.5,wbf,+0.0385,+0.0374,5
flip_h+ms_1.0+1.25+1.5,0.2348 ± 0.0368,0.1008 ± 0.0236,0.4010 ± 0.0391,none@1.0+h@1.0+none@1.25+none@1.5,nms,+0.0238,+0.0251,5
flip_h+ms_1.0+1.25+1.5_wbf,0.2452 ± 0.0372,0.1069 ± 0.0245,0.4112 ± 0.0395,none@1.0+h@1.0+none@1.25+none@1.5,wbf,+0.0342,+0.0313,5
ms_1.0+1.25+1.5,0.2343 ± 0.0368,0.0997 ± 0.0230,0.3961 ± 0.0390,none@1.0+none@1.25+none@1.5,nms,+0.0233,+0.0240,5
ms_1.0+1.25+1.5_wbf,0.2435 ± 0.0376,0.1061 ± 0.0243,0.4045 ± 0.0394,none@1.0+none@1.25+none@1.5,wbf,+0.0325,+0.0304,5


### Speed

In [ ]:
TIMED = [name for name in CONFIGS if CONFIGS[name][2] != 'check' and name in summary.index]

def gpu_utilization():
    try:
        process = subprocess.run(
            ['nvidia-smi', '--query-gpu=utilization.gpu', '--format=csv,noheader,nounits'],
            capture_output=True, text=True)
        return int(process.stdout.split()[0])
    except Exception:
        return None

In [ ]:
@torch.inference_mode()
def tta_frame(model, image, views, method):
    h, w = image.shape[-2:]
    by_scale = {}
    for view in views:
        flip, scale = parse_view(view)
        by_scale.setdefault(scale, []).append(flip)
    parts = []
    for scale, flips in by_scale.items():
        set_scale(model, scale)
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            preds = model([flip_image(image, flip) for flip in flips])
        parts += [(unflip(pred['boxes'].float(), flip, h, w).cpu().numpy(),
                   pred['scores'].float().cpu().numpy(), pred['labels'].cpu().numpy())
                  for flip, pred in zip(flips, preds)]
    set_scale(model, 1.0)
    return tta_merge(parts, method, h, w)

In [ ]:
def run_tta_speed(fold):
    path = TTA_OUT / 'speed' / f'fold{fold}.json'
    if path.exists():
        old = json.loads(path.read_text())
        if (old['settings_id'] == TTA_ID and old['checkpoint_sha256'] == CHECKPOINT_SHA[fold]
            and set(TIMED) <= set(old['ms_per_img'])):
            return old
    utilization = gpu_utilization()
    if utilization and utilization > 10:
        print(f'WARNING: GPU already {utilization}% busy before timing fold {fold}')
    model, run_config, checkpoint = load_model(fold)
    images = fold_images(fold)
    rng = np.random.default_rng(SEED + fold)
    picks = [images[i] for i in rng.choice(len(images), SPEED_IMAGES + SPEED_WARMUP, replace=False)]
    frames = [to_tensor(array).to(DEVICE) for array in read_images(picks)]
    times = {name: [] for name in TIMED}
    for i, image in enumerate(frames):
        for name in TIMED:
            views, method, _ = CONFIGS[name]
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            start = time.perf_counter()
            tta_frame(model, image, views, method)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            if i >= SPEED_WARMUP:
                times[name].append(time.perf_counter() - start)
    result = dict(settings_id=TTA_ID, checkpoint_sha256=CHECKPOINT_SHA[fold], fold=fold, frames=SPEED_IMAGES,
                  warmup=SPEED_WARMUP,
                  gpu_utilization_before=utilization,
                  device=torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu',
                  ms_per_img={name: float(np.mean(durations) * 1000) for name, durations in times.items()})
    write_json(result, path)
    del model, frames
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()
    return result

In [ ]:
speed_table = pd.DataFrame({k: run_tta_speed(k)['ms_per_img'] for k in FOLDS}).loc[TIMED]
speed_table['mean'], speed_table['std'] = speed_table[FOLDS].mean(axis=1), speed_table[FOLDS].std(axis=1)
display(speed_table.round(1).rename(columns=lambda col: f'fold {col}' if isinstance(col, int) else col))

E:\ucu_hw2\venv\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:51: UserWarning: X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.
  warnings.warn('X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')
E:\ucu_hw2\venv\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:42: UserWarning: X1 < 0 in box. Set it to 0.
  warnings.warn('X1 < 0 in box. Set it to 0.')


E:\ucu_hw2\venv\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:42: UserWarning: X1 < 0 in box. Set it to 0.
  warnings.warn('X1 < 0 in box. Set it to 0.')


E:\ucu_hw2\venv\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:51: UserWarning: X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.
  warnings.warn('X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')


E:\ucu_hw2\venv\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:51: UserWarning: X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.
  warnings.warn('X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')
E:\ucu_hw2\venv\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:42: UserWarning: X1 < 0 in box. Set it to 0.
  warnings.warn('X1 < 0 in box. Set it to 0.')


E:\ucu_hw2\venv\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:51: UserWarning: X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.
  warnings.warn('X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')


E:\ucu_hw2\venv\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:42: UserWarning: X1 < 0 in box. Set it to 0.
  warnings.warn('X1 < 0 in box. Set it to 0.')


E:\ucu_hw2\venv\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:42: UserWarning: X1 < 0 in box. Set it to 0.
  warnings.warn('X1 < 0 in box. Set it to 0.')


E:\ucu_hw2\venv\Lib\site-packages\ensemble_boxes\ensemble_boxes_wbf.py:51: UserWarning: X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.
  warnings.warn('X2 > 1 in box. Set it to 1. Check that you normalize boxes in [0, 1] range.')


,fold 0,fold 1,fold 2,fold 3,fold 4,mean,std
baseline,26.9,28.3,26.9,27.3,29.4,27.8,1.1
flip_h,53.4,55.6,53.2,53.5,57.4,54.6,1.8
flip_v,53.5,55.9,53.4,53.6,57.6,54.8,1.9
flip_hv,80.3,84.7,80.7,81.3,87.2,82.8,3.0
scale_0.5,19.1,19.9,19.3,19.4,20.6,19.7,0.6
scale_0.75,19.9,20.7,19.8,19.7,21.1,20.2,0.6
scale_1.25,38.9,40.9,38.7,39.4,42.2,40.0,1.5
scale_1.5,52.4,55.0,52.5,53.3,57.3,54.1,2.1
scale_2.0,91.4,95.7,91.9,92.3,99.4,94.1,3.4
ms_1.0+0.75,47.0,49.2,46.9,47.2,50.9,48.2,1.8


### Results

We select the best TTA configuration based on the highest mean CV mAP@[.50:.95] across all tested configurations, excluding the single flipped views used only for checking. We compare the results with the baseline separately for each fold, since both use the same data.

In [ ]:
fold_df = tta_results()
summary = tta_summary(fold_df)
stage_order = ['baseline', 'check', 'A_flip', 'B_scale', 'B_multiscale', 'C_flip_multiscale', 'D_merging']
order = [name for stage_name in stage_order for name in CONFIGS
         if CONFIGS[name][2] == stage_name and name in summary.index]
summary = summary.loc[order]
summary.index.name = 'config'
fold_df = fold_df.set_index('config').loc[order].reset_index()
fold_df.to_csv(TTA_OUT / 'tta_fold_results.csv', index=False)
summary.to_csv(TTA_OUT / 'tta_summary.csv')
not_evaluated = [name for name in CONFIGS if name not in summary.index]
print('Configurations without complete results:', not_evaluated or 'none')
tta_show(summary, order)

candidates = summary[summary.stage != 'check']
BEST = candidates.mAP_mean.idxmax()
best = summary.loc[BEST]
BEST_SCALE_CONFIG = summary.loc[[
    name for name in order if summary.loc[name, 'stage'] in ('B_scale', 'B_multiscale')]].mAP_mean.idxmax()
merging = pd.DataFrame([dict(config=name, nms=summary.loc[name, 'mAP_mean'],
                             wbf=summary.loc[name + '_wbf', 'mAP_mean'])
                        for name in NMS_TOP if name + '_wbf' in summary.index])
BEST_MERGE = 'wbf' if (merging.wbf > merging.nms).sum() > len(merging) / 2 else 'nms'
print('Best flip configuration:', BEST_FLIP, '| best scale configuration:', BEST_SCALE_CONFIG,
      '| best combined:', (summary.loc[STAGE_C].mAP_mean.idxmax() if STAGE_C else 'not run'),
      '| merging better in most pairs:', BEST_MERGE.upper(), '| best overall:', BEST)

Configurations without complete results: none


,mAP,APs,AR,views,merge,ms_per_img,delta_mAP,delta_APs,folds_improved
config,,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,none@1.0,none,27.8,+0.0000,+0.0000,0
flip_h_only,0.2077 ± 0.0326,0.0765 ± 0.0280,0.3453 ± 0.0430,h@1.0,none,NaN,-0.0033,+0.0008,0
flip_v_only,0.0540 ± 0.0075,0.0238 ± 0.0063,0.1475 ± 0.0153,v@1.0,none,NaN,-0.1570,-0.0519,0
flip_h,0.2157 ± 0.0340,0.0804 ± 0.0286,0.3631 ± 0.0425,none@1.0+h@1.0,nms,54.6,+0.0047,+0.0048,5
flip_v,0.2014 ± 0.0298,0.0736 ± 0.0238,0.3511 ± 0.0345,none@1.0+v@1.0,nms,54.8,-0.0096,-0.0020,0
flip_hv,0.2085 ± 0.0308,0.0788 ± 0.0260,0.3678 ± 0.0371,none@1.0+h@1.0+v@1.0,nms,82.8,-0.0026,+0.0031,1
scale_0.5,0.0992 ± 0.0190,0.0168 ± 0.0091,0.1859 ± 0.0287,none@0.5,none,19.7,-0.1118,-0.0589,0
scale_0.75,0.1703 ± 0.0257,0.0486 ± 0.0204,0.2904 ± 0.0337,none@0.75,none,20.2,-0.0407,-0.0270,0
scale_1.25,0.2228 ± 0.0356,0.0911 ± 0.0262,0.3663 ± 0.0402,none@1.25,none,40.0,+0.0118,+0.0154,5


Best flip configuration: flip_h | best scale configuration: ms_1.0+1.25+1.5 | best combined: flip_h@ms_1.0+1.25+1.5 | merging better in most pairs: WBF | best overall: flip_h@ms_1.0+1.25+1.5_wbf


In [ ]:
write_json(dict(TTA_SETTINGS, settings_id=TTA_ID, configs={
    name: dict(views=views, merge=method, stage=stage) for name, (views, method, stage) in CONFIGS.items()},
                input_sizes=json.loads(input_sizes.to_json(orient='index')),
                resolution_control='model.transform.min_size = 720*s and max_size = 1280*s per view; '
                                   'restored after every view',
                box_restoration=dict(h='x1 = W - x2, x2 = W - x1', v='y1 = H - y2, y2 = H - y1',
                                     scale='model transform.postprocess'),
                merging=dict(nms=f'torchvision batched_nms, class-aware, IoU {NMS_IOU}, top {DET_PER_IMG}',
                             wbf=f'ensemble_boxes.weighted_boxes_fusion, class-aware, IoU {WBF_IOU}, equal '
                                 f'weights, conf_type avg '
                                 f'(score * min(views, cluster size) / views), skip_box_thr 0, top '
                                 f'{DET_PER_IMG}'),
                confidence_filter='model box_score_thresh 0.01 inside every view; no extra filter',
                evaluation='COCOeval bbox, original annotations incl. iscrowd, maxDets=[1,10,100], held-out '
                           'fold of each checkpoint',
                speed='batch 1 original frame, autocast, frames on GPU before timing; all views, restoration '
                      'and merging; configs interleaved',
                not_evaluated=not_evaluated,
                checkpoints={k: dict(path=f'faster_rcnn/lr_scheduler/fold{k}/best.pth',
                                     sha256=CHECKPOINT_SHA[k]) for k in FOLDS}),
           TTA_OUT / 'tta_config.json')
best_rows = fold_df[fold_df.config == BEST].set_index('fold')
write_json(dict(config=BEST, views=CONFIGS[BEST][0], merge=CONFIGS[BEST][1],
                merge_settings=dict(iou=NMS_IOU if CONFIGS[BEST][1] == 'nms' else WBF_IOU,
                                    det_per_img=DET_PER_IMG,
                                    wbf=dict(skip_box_thr=0.0, conf_type='avg', weights='equal')
                                    if CONFIGS[BEST][1] == 'wbf' else None),
                selected_by='highest mean 5-fold CV mAP@[.50:.95]', best_flip=BEST_FLIP,
                best_scale_config=BEST_SCALE_CONFIG,
                best_combined=summary.loc[STAGE_C].mAP_mean.idxmax() if STAGE_C else None,
                better_merging=BEST_MERGE,
                cv_mean={col: float(best[col + '_mean']) for col in KEYS + ['ms_per_img', 'fps']},
                cv_std={col: float(best[col + '_std']) for col in KEYS + ['ms_per_img', 'fps']},
                baseline_cv_mean={col: float(summary.loc['baseline', col + '_mean'])
                                  for col in KEYS + ['ms_per_img', 'fps']},
                delta_vs_baseline=dict(mAP_mean=float(best.delta_mAP_mean), mAP_std=float(best.delta_mAP_std),
                                       APs_mean=float(best.delta_APs_mean), APs_std=float(best.delta_APs_std),
                                       folds_mAP_improved=int(best.folds_mAP_improved)),
                folds={int(k): {col: float(best_rows.loc[k, col]) for col in KEYS + ['ms_per_img']}
                       for k in best_rows.index}),
           TTA_OUT / 'tta_best_config.json')

In [ ]:
TTA_PLOTS = TTA_OUT / 'plots'

def save_tta_plot(fig, name):
    fig.update_layout(template='plotly_white', font=dict(size=13))
    fig.write_html(TTA_PLOTS / f'{name}.html', include_plotlyjs='directory', div_id=name)
    fig.show()

def tta_baseline_line(fig, key, **kwargs):
    fig.add_hline(y=summary.loc['baseline', key + '_mean'], line_dash='dash', line_color='#4c78a8',
                  annotation_text=f"baseline {summary.loc['baseline', key + '_mean']:.4f}", **kwargs)

def metric_bars(names, title, plot_name, labels=None):
    selected = summary.loc[names]
    fig = make_subplots(1, 2, subplot_titles=['mAP@[.50:.95]', 'AP_small'])
    for col, key in enumerate(['mAP', 'APs'], 1):
        fig.add_bar(x=labels or names, y=selected[key + '_mean'], error_y=dict(array=selected[key + '_std']),
                    showlegend=False,
                    marker_color=['#4c78a8' if name == 'baseline' else '#f58518' for name in names],
                    text=[f'{value:.4f}' for value in selected[key + '_mean']], hovertext=selected.views,
                    row=1, col=col)
        tta_baseline_line(fig, key, row=1, col=col)
    fig.update_layout(title=title + '; mean ± std over 5 CV folds')
    save_tta_plot(fig, plot_name)

In [ ]:
metric_bars(['baseline', *FLIPS, 'flip_h_only', 'flip_v_only'],
            'Experiment A: flip TTA (NMS); *_only = single flipped view (check)', 'flip_configurations')

scale_names = [name
               for name in ['scale_0.5', 'scale_0.75', 'baseline', 'scale_1.25', 'scale_1.5', 'scale_2.0']
               if name in summary.index]
subset = summary.loc[scale_names]
xs = [parse_view(view)[1] for view in subset.views]
fig = make_subplots(1, 2, subplot_titles=['mAP@[.50:.95]', 'AP_small'])
for col, key in enumerate(['mAP', 'APs'], 1):
    fig.add_scatter(x=xs, y=subset[key + '_mean'], error_y=dict(array=subset[key + '_std']),
                    mode='lines+markers', line_color='#e45756', showlegend=False,
                    text=[input_sizes.loc['1904×1071', f'{scale}×'] for scale in xs],
                    hovertemplate='scale %{x}×: %{y:.4f}<br>1904×1071 frame → %{text}<extra></extra>', row=1,
                    col=col)
fig.update_xaxes(title='scale (1.0 = original 1280×720 input)', tickvals=xs)
fig.update_layout(title='Experiment B: single-scale inference; mean ± std over 5 CV folds')
save_tta_plot(fig, 'map_aps_by_scale')

metric_bars(['baseline'] + MULTISCALE, 'Experiment B: 1.0× plus other scales (NMS)',
            'multiscale_combinations')

In [ ]:
pairs = [(name, name + '_wbf') for name in NMS_TOP if name + '_wbf' in summary.index]
fig = make_subplots(1, 2, subplot_titles=['mAP@[.50:.95]', 'AP_small'])
for method, color, pick in [('NMS', '#4c78a8', 0), ('WBF', '#e45756', 1)]:
    names = [pair[pick] for pair in pairs]
    for col, key in enumerate(['mAP', 'APs'], 1):
        fig.add_bar(x=[pair[0] for pair in pairs], y=summary.loc[names, key + '_mean'],
                    error_y=dict(array=summary.loc[names, key + '_std']),
                    name=method, marker_color=color, showlegend=col == 1,
                    text=[f'{value:.4f}' for value in summary.loc[names, key + '_mean']], row=1, col=col)
fig.update_layout(title='Experiment D: NMS vs WBF on the same view predictions; mean ± std over 5 CV folds',
                  barmode='group')
save_tta_plot(fig, 'nms_vs_wbf')

stage_colors = {'baseline': '#4c78a8', 'A_flip': '#f58518', 'B_scale': '#b279a2', 'B_multiscale': '#e45756',
                'C_flip_multiscale': '#54a24b', 'D_merging': '#9d755d'}
fig = go.Figure()
for stage, subset in summary[summary.stage != 'check'].groupby('stage', sort=False):
    fig.add_scatter(x=subset.ms_per_img_mean, y=subset.mAP_mean, mode='markers', name=stage,
                    text=list(subset.index),
                    marker=dict(size=11, color=stage_colors[stage]),
                    error_y=dict(array=subset.mAP_std, thickness=1),
                    hovertemplate='%{text}<br>%{x:.0f} ms/frame<br>mAP %{y:.4f}<extra></extra>')
fig.update_layout(title='Time per original frame vs mean CV mAP (all configurations)',
                  xaxis_title='ms / frame (batch 1, AMP)',
                  yaxis_title='mAP@[.50:.95]')
save_tta_plot(fig, 'speed_vs_map')

fig = make_subplots(1, 2, subplot_titles=['mAP@[.50:.95]', 'AP_small'])
for config, name, color in [('baseline', 'baseline', '#4c78a8'), (BEST, f'TTA {BEST}', '#e45756')]:
    subset = fold_df[fold_df.config == config]
    for col, key in enumerate(['mAP', 'APs'], 1):
        fig.add_bar(x=[f'fold {k}' for k in subset.fold] + ['mean'],
                    y=list(subset[key]) + [subset[key].mean()], name=name, marker_color=color,
                    error_y=dict(array=[0] * len(subset) + [subset[key].std()]), showlegend=col == 1, row=1,
                    col=col)
fig.update_layout(title='Baseline vs best TTA configuration per fold (CV held-out folds)', barmode='group')
save_tta_plot(fig, 'baseline_vs_best_tta')

In [ ]:
def tta_figure(rgb, image_info, panels, title):
    h, w = rgb.shape[:2]
    anns = [annotation for annotation in coco_train['annotations']
            if annotation['image_id'] == image_info['id']]
    xyxy = lambda annotation: [annotation['bbox'][0], annotation['bbox'][1],
                               annotation['bbox'][0] + annotation['bbox'][2],
                               annotation['bbox'][1] + annotation['bbox'][3]]
    small = Image.fromarray(rgb)
    small.thumbnail((1280, 1280))
    buffer = io.BytesIO()
    small.save(buffer, 'JPEG', quality=85)
    source = 'data:image/jpeg;base64,' + base64.b64encode(buffer.getvalue()).decode()
    fig = make_subplots(1, len(panels), horizontal_spacing=.01,
                        subplot_titles=[f"{name}: {int((pred[1] >= .3).sum())} boxes with score ≥ 0.3"
                                        for name, pred in panels.items()])
    def outline(boxes, color, text=None, name=''):
        xs, ys, hover = [], [], []
        for i, (x1, y1, x2, y2) in enumerate(boxes):
            xs += [x1, x2, x2, x1, x1, None]
            ys += [y1, y1, y2, y2, y1, None]
            hover += [text[i] if text else name] * 5 + [None]
        return go.Scatter(x=xs, y=ys, mode='lines', line=dict(color=color, width=1.2), text=hover,
                          hoverinfo='text', showlegend=False)
    for col, (name, (boxes, scores, labels)) in enumerate(panels.items(), 1):
        keep = scores >= .3
        fig.add_trace(outline([xyxy(annotation) for annotation in anns if not annotation.get('iscrowd', 0)],
                              'lime', name='ground truth'), 1, col)
        fig.add_trace(outline([xyxy(annotation) for annotation in anns if annotation.get('iscrowd', 0)],
                              'yellow', name='ignored region'), 1, col)
        fig.add_trace(outline(
            boxes[keep].tolist(), 'red',
            [f"{CATS[LABEL_TO_CAT[int(x)]]} {y:.2f}" for x, y in zip(labels[keep], scores[keep])]), 1, col)
        fig.add_layout_image(source=source, xref=f'x{col}', yref=f'y{col}', x=0, y=0, sizex=w, sizey=h,
                             xanchor='left', yanchor='top', sizing='stretch', layer='below')
    fig.update_xaxes(range=[0, w], visible=False)
    fig.update_yaxes(range=[h, 0], visible=False)
    fig.update_layout(title=title
                      + ' · green = ground truth, red = prediction (score ≥ 0.3), yellow = ignored',
                      width=1500, height=int(750 * h / w) + 120, margin=dict(l=10, r=10, t=90, b=10),
                      template='plotly_white')
    return fig

In [ ]:
base_folds = fold_df[fold_df.config == 'baseline'].set_index('fold').mAP
rep_fold = int(base_folds.sort_values().index[len(base_folds) // 2])
images = fold_images(rep_fold)
n_objects = (pd.Series([annotation['image_id'] for annotation in coco_train['annotations']
                        if not annotation.get('iscrowd', 0)]).value_counts())
examples = [('most_objects', max(images, key=lambda image_info: n_objects.get(image_info['id'], 0))),
            ('random', images[int(np.random.default_rng(SEED).integers(len(images)))])]
model, run_config, checkpoint = load_model(rep_fold)
saved = []
for label, image_info in examples:
    rgb = read_images([image_info])[0]
    image = to_tensor(rgb).to(DEVICE)
    panels = {'Baseline (none@1.0)': tta_frame(model, image, ['none@1.0'], 'nms'),
              f'TTA {BEST}': tta_frame(model, image, *CONFIGS[BEST][:2])}
    fig = tta_figure(rgb, image_info, panels,
                     f"Fold {rep_fold} held-out frame {image_info['file_name']} "
                     f"({n_objects.get(image_info['id'], 0)} objects)")
    path = TTA_OUT / 'visualizations' / f"fold{rep_fold}_{label}_{image_info['id']}.html"
    fig.write_html(path, include_plotlyjs='directory')
    saved.append(path.name)
del model
print('Saved to', TTA_OUT / 'visualizations', ':', saved)

Saved to E:\ucu_hw2\faster_rcnn\inference_experiments\tta\visualizations : ['fold1_most_objects_1128.html', 'fold1_random_18517.html']


## Experiment №4 Combined Inference Methods

In this experiment, we check whether combining methods from previous experiments can improve Faster R-CNN performance without retraining the model, using the same 5 validation folds.

We test three combinations:

1. **TTA + SAHI**
2. **TTA + Soft-NMS** with σ = 0.2 and 0.3
3. **TTA + SAHI + Soft-NMS** with σ = 0.2 and 0.3

For each method, we use the best settings from previous experiments: **TTA** with scales of 1.0×, 1.25×, and 1.5× and horizontal flips (6 views) with WBF, **SAHI** with 1024 px slices, 0.3 overlap, and an additional full-image pass with NMS (IoU 0.5), and **Gaussian Soft-NMS** with σ = 0.2 and 0.3.

We use **TTA + WBF** as the main benchmark and also compare each new combination with the corresponding version without the added method to see how it affects performance.

In [ ]:
%pip -q install sahi ensemble-boxes

In [ ]:
import warnings
COMBO_OUT = ROOT / 'faster_rcnn/inference_experiments/combinations'
for subfolder in ['views', 'folds', 'speed', 'plots', 'visualizations']:
    (COMBO_OUT / subfolder).mkdir(parents=True, exist_ok=True)
EXPERIMENTS_DIR = ROOT / 'faster_rcnn/inference_experiments'
tta_best = json.loads((EXPERIMENTS_DIR / 'tta/tta_best_config.json').read_text())
sahi_best = json.loads((EXPERIMENTS_DIR / 'sahi/sahi_best_config.json').read_text())
soft_config = json.loads((EXPERIMENTS_DIR / 'soft_nms/soft_nms_config.json').read_text())

In [ ]:
TTA_VIEWS = tta_best['views']
SLICE, OVERLAP = sahi_best['slice_size'], sahi_best['overlap']
SOFT_SIGMAS = [0.2, 0.3]
NMS_VARIANTS = ['std'] + [f's{sigma}' for sigma in SOFT_SIGMAS]
assert (TTA_VIEWS == ['none@1.0', 'h@1.0', 'none@1.25', 'h@1.25', 'none@1.5', 'h@1.5']
        and tta_best['merge'] == 'wbf')
assert ((SLICE, OVERLAP, sahi_best['merge'], sahi_best['merge_threshold'],
         sahi_best['full_image_pass']) == (1024, 0.3, 'nms', 0.5, True))
assert soft_config['soft_min_score'] == SOFT_MIN_SCORE and set(SOFT_SIGMAS) <= set(soft_config['sigmas'])
assert tta_best['merge_settings']['iou'] == WBF_IOU and MATCH['nms'] == ('IOU', 0.5) and SLICE_BATCH == 8

In [ ]:
COMBO_SETTINGS = dict(tta_views=TTA_VIEWS, tta_batch={str(k): batch for k, batch in TTA_BATCH.items()},
                      wbf=dict(iou_thr=WBF_IOU, skip_box_thr=0.0, conf_type='avg', weights='equal'),
                      sahi=dict(slice=SLICE, overlap=OVERLAP, slice_batch=SLICE_BATCH, merge='nms', iou=0.5,
                                full_image=True),
                      soft_sigmas=SOFT_SIGMAS, soft_min_score=SOFT_MIN_SCORE, det_per_img=DET_PER_IMG,
                      amp=AMP, coco_max_dets=MAX_DETS,
                      inference=INFERENCE, folds_sha=FOLDS_SHA, speed_images=SPEED_IMAGES,
                      speed_warmup=SPEED_WARMUP, seed=SEED,
                      versions=dict(torch=torch.__version__, torchvision=torchvision.__version__,
                                    sahi=sahi.__version__,
                                    ensemble_boxes=version('ensemble-boxes')))
COMBO_ID = hashlib.sha256(json.dumps(COMBO_SETTINGS, sort_keys=True).encode()).hexdigest()[:16]
warnings.filterwarnings('ignore', module='ensemble_boxes')

def set_scale(model, scale):
    model.transform.min_size = (round(INFERENCE['min_size'] * scale),)
    model.transform.max_size = round(INFERENCE['max_size'] * scale)

In [ ]:
def fold_rows(folder, name):
    return {int(row['fold']): row for path in sorted((EXPERIMENTS_DIR / folder).glob(f'{name}_fold*.json'))
            for row in [json.loads(path.read_text())['row']]}
soft_saved = {}
for k in FOLDS:
    for p in [EXPERIMENTS_DIR / f'soft_nms/folds/fold{k}.json',
              EXPERIMENTS_DIR / f'soft_nms/folds/fold{k}_extra_sigmas.json']:
        for record in json.loads(p.read_text())['rows']:
            soft_saved[record['variant'], k] = record
REFERENCE = {'baseline': {k: stored[k]['metrics'] for k in FOLDS},
             **{f'soft_s{sigma}': {k: soft_saved[f'soft_nms_s{sigma}', k] for k in FOLDS}
                for sigma in SOFT_SIGMAS},
             'sahi': fold_rows('sahi/folds', sahi_best['config']),
             'tta_wbf': fold_rows('tta/folds', tta_best['config'])}
assert all(len(rows_by_fold) == len(FOLDS) for rows_by_fold in REFERENCE.values())
ref_table = pd.DataFrame({
    name: {col: np.mean([REFERENCE[name][k][col] for k in FOLDS]) for col in ['mAP', 'APs']}
    for name in REFERENCE}).T
ref_table['ms_per_img (earlier sessions)'] = [tta_best['baseline_cv_mean']['ms_per_img'],
                                              *[np.mean([soft_saved[f'soft_nms_s{sigma}', k]['ms_per_img']
                                                         for k in FOLDS]) for sigma in SOFT_SIGMAS],
                                              sahi_best['cv_mean']['ms_per_img'],
                                              tta_best['cv_mean']['ms_per_img']]
display(ref_table.round(4))
print('settings id', COMBO_ID, '| sahi', sahi.__version__, '| ensemble-boxes', version('ensemble-boxes'), '|',
      torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu')

,mAP,APs,ms_per_img (earlier sessions)
baseline,0.2110,0.0757,27.7821
soft_s0.2,0.2144,0.0784,91.3553
soft_s0.3,0.2144,0.0789,86.8125
sahi,0.2232,0.0934,117.6369
tta_wbf,0.2495,0.1131,267.9821


settings id e206de278b60c9d8 | sahi 0.14.0 | ensemble-boxes 1.0.10 | NVIDIA GeForce RTX 5070


### Inspection of the earlier implementations


### Combined pipeline
In this experiment, we combine TTA, SAHI, and Soft-NMS in one pipeline, using 6 views of each image: scales of 1.0×, 1.25×, and 1.5×, with and without horizontal flips.

For each view, we first process the full image, and for configurations with SAHI, we also split it into 1024 px slices with an overlap of 0.3. All slices use the same scale and flip as the corresponding TTA view, and then we map the detected boxes back to the original image coordinates.

Inside Faster R-CNN, we use either standard NMS or Gaussian Soft-NMS with σ = 0.2 and 0.3, without changing the RPN. When SAHI is used, we first merge predictions from the slices and the full image using NMS with IoU 0.5, and then combine the results from all 6 TTA views using WBF.

This approach keeps the settings of each method from previous experiments and allows us to test how well they work together without unnecessary repeated processing.

To check that everything works correctly, we reproduce the baseline, Soft-NMS, SAHI, and TTA + WBF results separately and compare them with the previous experiments.

In [ ]:
def soft_nms_fast(boxes, scores, labels, sigma, max_dets, iou=None):
    if iou is None:
        iou = class_iou(boxes, labels)
    decay = torch.exp(-iou.square() / sigma).cpu().numpy()
    current = scores.float().cpu().numpy().copy()
    n_steps = min(len(current), max_dets)
    keep, final = np.empty(n_steps, dtype=np.int64), np.empty(n_steps, dtype=np.float32)
    for i in range(n_steps):
        j = current.argmax()
        keep[i], final[i] = j, current[j]
        current *= decay[j]
        current[j] = -1.
    valid = final >= SOFT_MIN_SCORE
    return torch.from_numpy(keep[valid]).to(boxes.device), torch.from_numpy(final[valid]).to(boxes.device)

def soft_fast(sigma):
    return lambda boxes, scores, labels, heads: soft_nms_fast(boxes, scores, labels, sigma,
                                                              heads.detections_per_img)

def set_internal_nms(model, variant):
    use_final_nms(model, None if variant == 'std' else soft_fast(float(variant[1:])))

In [ ]:
rng = np.random.default_rng(SEED)
H, W = 1071, 1904
top_left = rng.uniform(0, [W - 200, H - 200], (50, 2))
boxes = torch.tensor(np.c_[top_left, top_left + rng.uniform(1, 200, (50, 2))], dtype=torch.float32)
for f in ['h', 'v']:
    assert torch.allclose(unflip(unflip(boxes, f, H, W), f, H, W), boxes)
for x0, y0, x1, y1 in get_slice_bboxes(H, W, SLICE, SLICE, False, OVERLAP, OVERLAP):
    inside = boxes[(boxes[:, 0] >= x0) & (boxes[:, 1] >= y0) & (boxes[:, 2] <= x1) & (boxes[:, 3] <= y1)]
    in_flipped_crop = unflip(inside - torch.tensor([x0, y0, x0, y0]), 'h', y1 - y0, x1 - x0)
    assert torch.allclose(unflip(in_flipped_crop, 'h', y1 - y0, x1 - x0) + torch.tensor([x0, y0, x0, y0]),
                          inside)
model, run_config, checkpoint = load_model(FOLDS[0])
with torch.inference_mode():
    for s in [1.0, 1.25, 1.5]:
        set_scale(model, s)
        transformed, targets = model.transform(
            [torch.zeros(3, H, W, device=DEVICE)],
            [dict(boxes=boxes.to(DEVICE), labels=torch.ones(50, dtype=torch.int64, device=DEVICE))])
        restored = model.transform.postprocess([dict(boxes=targets[0]['boxes'])], transformed.image_sizes,
                                               [(H, W)])[0]['boxes']
        assert torch.allclose(restored.cpu(), boxes, atol=1e-3), s
set_scale(model, 1.0)

In [ ]:
captured = []
use_final_nms(model, capture(captured))
predict(model, fold_images(FOLDS[0])[:8])
use_final_nms(model)
with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=AMP):
    for b, s, l in captured:
        b, s, l = b.to(DEVICE), s.to(DEVICE), l.to(DEVICE)
        for sigma in SOFT_SIGMAS:
            keep_gpu, scores_gpu = soft_nms(b, s, l, sigma, DET_PER_IMG)
            keep_cpu, scores_cpu = soft_nms_fast(b, s, l, sigma, DET_PER_IMG)
            assert torch.equal(keep_gpu, keep_cpu) and torch.equal(scores_gpu, scores_cpu)
del model, captured
print('Round-trip checks (flip, crop + flip, scale) and Soft-NMS equality checks passed.')

Round-trip checks (flip, crop + flip, scale) and Soft-NMS equality checks passed.


### Cached views and evaluation


In [ ]:
COMBOS = {
    'baseline':  (['none@1.0'], False, 'std', 'check'),
    'soft_s0.2': (['none@1.0'], False, 's0.2', 'check'),
    'soft_s0.3': (['none@1.0'], False, 's0.3', 'check'),
    'sahi':      (['none@1.0'], True, 'std', 'check'),
    'tta_wbf':   (TTA_VIEWS, False, 'std', 'reference'),
    'tta_sahi_wbf': (TTA_VIEWS, True, 'std', '4.1'),
    **{f'tta_soft_s{sigma}_wbf': (TTA_VIEWS, False, f's{sigma}', '4.2') for sigma in SOFT_SIGMAS},
    **{f'tta_sahi_soft_s{sigma}_wbf': (TTA_VIEWS, True, f's{sigma}', '4.3') for sigma in SOFT_SIGMAS},
}
CONTROL = {'tta_sahi_wbf': 'tta_wbf', **{f'tta_soft_s{sigma}_wbf': 'tta_wbf' for sigma in SOFT_SIGMAS},
           **{f'tta_sahi_soft_s{sigma}_wbf': 'tta_sahi_wbf' for sigma in SOFT_SIGMAS}}
SETS = [f'{kind}_{variant}' for kind in ['full', 'sahi'] for variant in NMS_VARIANTS]

In [ ]:
def load_sahi_slices(fold):
    data = np.load(SAHI_OUT / 'cache' / f'slices{SLICE}_overlap{OVERLAP}_fold{fold}.npz')
    ends = np.cumsum(data['counts'])
    return [(int(i), data['boxes'][start:end], data['scores'][start:end], data['labels'][start:end])
            for i, start, end in zip(data['ids'], ends - data['counts'], ends)]

def combo_view_path(fold, view, suffix='.npz'):
    return COMBO_OUT / 'views' / f'fold{fold}_{view}{suffix}'

In [ ]:
@torch.inference_mode()
def predict_variants(model, inputs):
    candidates = []
    use_final_nms(model, capture(candidates))
    try:
        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
            outputs = model(inputs)
    finally:
        use_final_nms(model)
    results = []
    with torch.autocast(device_type=DEVICE.type, enabled=AMP):
        for output, (boxes, scores, labels) in zip(outputs, candidates):
            boxes, scores, labels = boxes.to(DEVICE), scores.to(DEVICE), labels.to(DEVICE)
            iou = class_iou(boxes, labels)
            kept = {'std': standard_nms(boxes, scores, labels, model.roi_heads)}
            kept.update({f's{sigma}': soft_nms_fast(boxes, scores, labels, sigma, DET_PER_IMG, iou)
                         for sigma in SOFT_SIGMAS})
            results.append({
                variant: (output['boxes'][keep].float(), new_scores.float(), output['labels'][keep])
                for variant, (keep, new_scores) in kept.items()})
    return results

def to_numpy(boxes, scores, labels):
    return boxes.cpu().numpy(), scores.cpu().numpy(), labels.cpu().numpy()

In [ ]:
def full_view(model, images, flip, scale):
    rows = []
    for chunk, image_tensors in batches(images, TTA_BATCH[scale]):
        image_tensors = [image.to(DEVICE, non_blocking=True) for image in image_tensors]
        for image, per_variant in zip(
                image_tensors, predict_variants(model, [flip_image(image, flip) for image in image_tensors])):
            h, w = image.shape[-2:]
            rows.append({variant: to_numpy(unflip(b, flip, h, w), s, l)
                         for variant, (b, s, l) in per_variant.items()})
    return rows

def slice_view(model, image, flip):
    h, w = image.shape[-2:]
    windows = get_slice_bboxes(h, w, SLICE, SLICE, False, OVERLAP, OVERLAP)
    parts = {variant: [] for variant in NMS_VARIANTS}
    for i in range(0, len(windows), SLICE_BATCH):
        chunk = windows[i:i + SLICE_BATCH]
        slice_results = predict_variants(
            model, [flip_image(image[:, y0:y1, x0:x1], flip) for x0, y0, x1, y1 in chunk])
        for (x0, y0, x1, y1), per_variant in zip(chunk, slice_results):
            offset = torch.tensor([x0, y0, x0, y0], dtype=torch.float32, device=DEVICE)
            for variant, (boxes, scores, labels) in per_variant.items():
                parts[variant].append((unflip(boxes, flip, y1 - y0, x1 - x0) + offset, scores, labels))
    return {variant: to_numpy(*(torch.cat(x) for x in zip(*slice_parts)))
            for variant, slice_parts in parts.items()}, len(windows)

In [ ]:
def boxes_valid(boxes, labels, h, w):
    return len(boxes) == 0 or (np.isfinite(boxes).all() and boxes.min() >= -.5
                               and boxes[:, [0, 2]].max() <= w + .5 and boxes[:, [1, 3]].max() <= h + .5
                           and labels.min() >= 1 and labels.max() <= N_CLASSES - 1)

def same_arrays(arrays_a, arrays_b):
    return all(np.array_equal(x, y) for x, y in zip(arrays_a, arrays_b))

In [ ]:
def compute_view(fold, view, model, images):
    flip, scale = parse_view(view)
    set_scale(model, scale)
    try:
        full = full_view(model, images, flip, scale)
        tta_cache = view_path(fold, view)
        tta_same = (all(same_arrays(full_row['std'], cached_row[1:])
                        for full_row, cached_row in zip(full, load_view(tta_cache)))
                    if tta_cache.exists() else None)
        sahi_cache = load_sahi_slices(fold) if view == 'none@1.0' else None
        sets = {set_name: [] for set_name in SETS}
        before, slices, slices_same = {variant: [] for variant in NMS_VARIANTS}, [], True
        for k, (chunk, image_tensors) in enumerate(batches(images, 1)):
            image = image_tensors[0].to(DEVICE)
            h, w = image.shape[-2:]
            parts, n_slices = slice_view(model, image, flip)
            slices.append(n_slices)
            if sahi_cache is not None:
                slices_same &= (sahi_cache[k][0] == int(chunk[0]['id'])
                                and same_arrays(parts['std'], sahi_cache[k][1:4]))
            for variant in NMS_VARIANTS:
                b, s, l = (np.concatenate([x, y]) for x, y in zip(parts[variant], full[k][variant]))
                before[variant].append(len(s))
                sets[f'full_{variant}'].append(full[k][variant])
                sets[f'sahi_{variant}'].append(merge(b, s, l, 'nms', h, w))
        image_tensors = [to_tensor(array).to(DEVICE) for array in read_images(images[:TTA_BATCH[scale]])]
        in_model_same = True
        for variant in NMS_VARIANTS[1:]:
            set_internal_nms(model, variant)
            with torch.inference_mode(), torch.autocast(device_type=DEVICE.type, enabled=AMP):
                preds = model([flip_image(batch_image, flip) for batch_image in image_tensors])
            for k, (batch_image, pred) in enumerate(zip(image_tensors, preds)):
                in_model_same &= same_arrays(to_numpy(
                    unflip(pred['boxes'].float(), flip, *batch_image.shape[-2:]), pred['scores'].float(),
                    pred['labels']), full[k][variant])
            set_internal_nms(model, 'std')
    except torch.cuda.OutOfMemoryError:
        write_json(dict(fold=fold, view=view, error='CUDA out of memory'),
                   combo_view_path(fold, view, '.oom.json'))
        print(f'  fold {fold} {view}: CUDA out of memory, not evaluated')
        return
    finally:
        set_scale(model, 1.0)
        use_final_nms(model)
        torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
    dims = [(image_info['height'], image_info['width']) for image_info in images]
    checks = dict(settings_id=COMBO_ID, checkpoint_sha256=CHECKPOINT_SHA[fold], fold=fold, view=view,
                  frames=len(images),
                  ids_equal_fold=True, standard_equals_tta_cache=tta_same,
                  slices_equal_sahi_cache=slices_same if sahi_cache is not None else None,
                  in_model_soft_equals_captured_first_batch=bool(in_model_same),
                  boxes_and_labels_valid=all(boxes_valid(row[0], row[2], h, w) for set_name in SETS
                                             for row, (h, w) in zip(sets[set_name], dims)),
                  slices_per_frame=float(np.mean(slices)),
                  boxes_per_frame={set_name: float(np.mean([len(row[1]) for row in sets[set_name]]))
                                   for set_name in SETS},
                  boxes_before_sahi_merge={variant: float(np.mean(before[variant]))
                                           for variant in NMS_VARIANTS})
    assert (checks['boxes_and_labels_valid'] and in_model_same and tta_same is not False
            and slices_same), checks
    arrays = dict(ids=np.array([int(image_info['id']) for image_info in images]))
    for s in SETS:
        arrays[s + '_counts'] = np.array([len(row[1]) for row in sets[s]])
        arrays[s + '_boxes'] = np.concatenate([row[0] for row in sets[s]]).reshape(-1, 4).astype(np.float32)
        arrays[s + '_scores'] = np.concatenate([row[1] for row in sets[s]]).astype(np.float32)
        arrays[s + '_labels'] = np.concatenate([row[2] for row in sets[s]]).astype(np.int64)
    tmp_path = combo_view_path(fold, view, '.tmp.npz')
    np.savez(tmp_path, **arrays)
    os.replace(tmp_path, combo_view_path(fold, view))
    write_json(checks, combo_view_path(fold, view, '.json'))
    print(f"  fold {fold} {view}: {len(images)} frames, {checks['slices_per_frame']:.1f} slices/frame, "
          f"standard = TTA cache: {tta_same}, slices = SAHI cache: {checks['slices_equal_sahi_cache']}")

In [ ]:
def view_ready(fold, view):
    path = combo_view_path(fold, view, '.json')
    if not path.exists() or not combo_view_path(fold, view).exists():
        return False
    checks = json.loads(path.read_text())
    return (checks['settings_id'] == COMBO_ID and checks['checkpoint_sha256'] == CHECKPOINT_SHA[fold]
            and checks['frames'] == len(fold_images(fold)))

def compute_combo_views(fold):
    todo = [view for view in TTA_VIEWS
            if not view_ready(fold, view) and not combo_view_path(fold, view, '.oom.json').exists()]
    if not todo:
        return
    started = time.time()
    model, run_config, checkpoint = load_model(fold)
    images = fold_images(fold)
    assert len(images) == stored[fold]['val_frames']
    for view in todo:
        compute_view(fold, view, model, images)
    del model
    print(f'  fold {fold}: views computed in {(time.time() - started) / 60:.1f} min')

In [ ]:
def load_set(fold, view, name):
    data = np.load(combo_view_path(fold, view))
    ends = np.cumsum(data[name + '_counts'])
    starts = ends - data[name + '_counts']
    boxes, scores, labels = data[name + '_boxes'], data[name + '_scores'], data[name + '_labels']
    return data['ids'], [(boxes[i:j], scores[i:j], labels[i:j]) for i, j in zip(starts, ends)]

def combo_predictions(name, fold):
    views, slices, variant, _ = COMBOS[name]
    data = [load_set(fold, view, f"{'sahi' if slices else 'full'}_{variant}") for view in views]
    images = fold_images(fold)
    assert all(list(ids) == [int(image_info['id']) for image_info in images] for ids, _ in data)
    preds = []
    for k, image_info in enumerate(images):
        parts = [view_set[1][k] for view_set in data]
        assert len(parts) == len(views)
        boxes, scores, labels = tta_merge(parts, 'wbf', image_info['height'], image_info['width'])
        assert boxes_valid(boxes, labels, image_info['height'], image_info['width'])
        preds.append((int(image_info['id']), dict(boxes=boxes, scores=scores, labels=labels)))
    return preds

In [ ]:
def evaluate_combo(name, fold):
    path = COMBO_OUT / 'folds' / f'{name}_fold{fold}.json'
    if path.exists():
        old = json.loads(path.read_text())
        if old['settings_id'] == COMBO_ID and old['checkpoint_sha256'] == CHECKPOINT_SHA[fold]:
            return old
    views, slices, variant, role = COMBOS[name]
    if not all(view_ready(fold, view) for view in views):
        return None
    preds = combo_predictions(name, fold)
    metrics = coco_metrics(preds, fold_gt(fold))
    row = dict(config=name, role=role, views='+'.join(views), n_views=len(views), sahi=slices,
               internal_nms=variant,
               cross_slice='sahi nms 0.5' if slices else 'none',
               cross_view='wbf' if len(views) > 1 else 'none', fold=fold, **metrics,
               detections_per_img=float(np.mean([len(pred['scores']) for _, pred in preds])))
    result = dict(settings_id=COMBO_ID, checkpoint_sha256=CHECKPOINT_SHA[fold], row=row)
    if name in REFERENCE:
        result['max_abs_diff_vs_earlier_result'] = max(abs(metrics[metric] - REFERENCE[name][fold][metric])
                                                       for metric in KEYS)
    write_json(result, path)
    return result

In [ ]:
def combo_results():
    rows = pd.DataFrame([json.loads(path.read_text())['row']
                         for path in sorted((COMBO_OUT / 'folds').glob('*_fold*.json'))])
    speed = [dict(config=name, fold=record['fold'], ms_per_img=ms)
             for path in sorted((COMBO_OUT / 'speed').glob('fold*.json'))
             for record in [json.loads(path.read_text())] for name, ms in record['ms_per_img'].items()]
    if speed:
        rows = rows.merge(pd.DataFrame(speed), on=['config', 'fold'], how='left')
        rows['fps'] = 1000 / rows.ms_per_img
    return rows

In [ ]:
started = time.time()
for fold in FOLDS:
    compute_combo_views(fold)
    for name in COMBOS:
        record = evaluate_combo(name, fold)
        if record is None:
            print(f'NOT EVALUATED: {name} fold {fold} (missing views)')
    print(f'fold {fold} done after {(time.time() - started) / 60:.1f} min')

reproduction = pd.DataFrame([dict(config=name, fold=k, mAP=result['row']['mAP'],
                                  earlier=REFERENCE[name][k]['mAP'],
                                  max_abs_diff=result['max_abs_diff_vs_earlier_result'])
                             for name in REFERENCE for k in FOLDS for result in [evaluate_combo(name, k)]
                             if result])
display(reproduction.pivot(index='fold', columns='config', values='max_abs_diff'))
view_checks = pd.DataFrame([json.loads(path.read_text())
                            for path in sorted((COMBO_OUT / 'views').glob('fold*.json'))
                            if not path.name.endswith('.oom.json')])
display(view_checks[['fold', 'view', 'frames', 'standard_equals_tta_cache', 'slices_equal_sahi_cache',
                     'in_model_soft_equals_captured_first_batch',
                     'boxes_and_labels_valid', 'slices_per_frame']])

fold 0 done after 0.0 min
fold 1 done after 0.0 min
fold 2 done after 0.0 min
fold 3 done after 0.0 min
fold 4 done after 0.0 min


config,baseline,sahi,soft_s0.2,soft_s0.3,tta_wbf
fold,,,,,
0,0.0,0.0,0.0,0.0,0.0
1,0.0,0.0,0.0,0.0,0.0
2,0.0,0.0,0.0,0.0,0.0
3,0.0,0.0,0.0,0.0,0.0
4,0.0,0.0,0.0,0.0,0.0


,fold,view,frames,standard_equals_tta_cache,slices_equal_sahi_cache,in_model_soft_equals_captured_first_batch,boxes_and_labels_valid,slices_per_frame
0,0,h@1.0,981,True,None,True,True,6.024465
1,0,h@1.25,981,True,None,True,True,6.024465
2,0,h@1.5,981,True,None,True,True,6.024465
3,0,none@1.0,981,True,True,True,True,6.024465
4,0,none@1.25,981,True,None,True,True,6.024465
5,0,none@1.5,981,True,None,True,True,6.024465
6,1,h@1.0,979,True,None,True,True,4.561798
7,1,h@1.25,979,True,None,True,True,4.561798
8,1,h@1.5,979,True,None,True,True,4.561798
9,1,none@1.0,979,True,True,True,True,4.561798


### Speed

In [ ]:
COMBO_SPEED_CONFIGS = list(COMBOS)

def gpu_busy():
    if DEVICE.type != 'cuda':
        return None
    torch.cuda.synchronize()
    time.sleep(2)
    process = subprocess.run(['nvidia-smi', '--query-gpu=utilization.gpu', '--format=csv,noheader,nounits'],
                             capture_output=True, text=True)
    return int(process.stdout.split()[0]) if process.returncode == 0 else None

In [ ]:
@torch.inference_mode()
def combo_frame(model, image, name):
    views, slices, variant, _ = COMBOS[name]
    h, w = image.shape[-2:]
    set_internal_nms(model, variant)
    by_scale = {}
    for view in views:
        flip, scale = parse_view(view)
        by_scale.setdefault(scale, []).append(flip)
    parts = []
    try:
        for scale, flips in by_scale.items():
            set_scale(model, scale)
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                preds = model([flip_image(image, view_flip) for view_flip in flips])
            for view_flip, pred in zip(flips, preds):
                full = to_numpy(unflip(pred['boxes'].float(), view_flip, h, w), pred['scores'].float(),
                                pred['labels'])
                if slices:
                    windows = get_slice_bboxes(h, w, SLICE, SLICE, False, OVERLAP, OVERLAP)
                    slice_boxes, slice_scores, slice_labels = [], [], []
                    for i in range(0, len(windows), SLICE_BATCH):
                        chunk = windows[i:i + SLICE_BATCH]
                        with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                            slice_preds = model([flip_image(image[:, y0:y1, x0:x1], view_flip)
                                                 for x0, y0, x1, y1 in chunk])
                        for (x0, y0, x1, y1), slice_pred in zip(chunk, slice_preds):
                            slice_boxes.append(unflip(
                                slice_pred['boxes'].float(), view_flip, y1 - y0,
                                x1 - x0) + torch.tensor([x0, y0, x0, y0], dtype=torch.float32, device=DEVICE))
                            slice_scores.append(slice_pred['scores'].float())
                            slice_labels.append(slice_pred['labels'])
                    boxes, scores, labels = to_numpy(torch.cat(slice_boxes), torch.cat(slice_scores),
                                                     torch.cat(slice_labels))
                    full = merge(np.concatenate([boxes, full[0]]), np.concatenate([scores, full[1]]),
                                 np.concatenate([labels, full[2]]), 'nms', h, w)
                parts.append(full)
    finally:
        set_scale(model, 1.0)
        set_internal_nms(model, 'std')
    return tta_merge(parts, 'wbf', h, w)

In [ ]:
def run_combo_speed(fold):
    path = COMBO_OUT / 'speed' / f'fold{fold}.json'
    if path.exists():
        old = json.loads(path.read_text())
        if (old['settings_id'] == COMBO_ID and old['checkpoint_sha256'] == CHECKPOINT_SHA[fold]
            and set(COMBO_SPEED_CONFIGS) <= set(old['ms_per_img'])):
            return old
    busy = gpu_busy()
    if busy is not None and busy > 10:
        print(f'WARNING: GPU {busy}% busy before timing fold {fold} (another process)')
    model, run_config, checkpoint = load_model(fold)
    images = fold_images(fold)
    rng = np.random.default_rng(SEED + fold)
    picks = [images[i] for i in rng.choice(len(images), SPEED_IMAGES + SPEED_WARMUP, replace=False)]
    frames = [to_tensor(array).to(DEVICE) for array in read_images(picks)]
    times = {name: [] for name in COMBO_SPEED_CONFIGS}
    for i, image in enumerate(frames):
        for name in COMBO_SPEED_CONFIGS:
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            start = time.perf_counter()
            combo_frame(model, image, name)
            if DEVICE.type == 'cuda':
                torch.cuda.synchronize()
            if i >= SPEED_WARMUP:
                times[name].append(time.perf_counter() - start)
    result = dict(settings_id=COMBO_ID, checkpoint_sha256=CHECKPOINT_SHA[fold], fold=fold,
                  frames=SPEED_IMAGES, warmup=SPEED_WARMUP,
                  gpu_utilization_before=busy,
                  device=torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu',
                  ms_per_img={name: float(np.mean(durations) * 1000) for name, durations in times.items()})
    write_json(result, path)
    del model, frames
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
    return result

In [ ]:
speed_table = pd.DataFrame({k: run_combo_speed(k)['ms_per_img'] for k in FOLDS}).loc[COMBO_SPEED_CONFIGS]
speed_table['mean'], speed_table['std'] = speed_table[FOLDS].mean(axis=1), speed_table[FOLDS].std(axis=1)
display(speed_table.round(1).rename(columns=lambda col: f'fold {col}' if isinstance(col, int) else col))

,fold 0,fold 1,fold 2,fold 3,fold 4,mean,std
baseline,26.7,25.6,25.9,25.9,25.5,25.9,0.5
soft_s0.2,27.5,26.4,26.6,26.3,26.1,26.6,0.5
soft_s0.3,27.5,26.3,26.3,26.2,26.3,26.5,0.5
sahi,132.2,114.7,119.3,128.4,100.5,119.0,12.5
tta_wbf,250.0,241.5,244.4,243.8,245.4,245.0,3.1
tta_sahi_wbf,1181.6,1030.7,1077.3,1154.1,918.2,1072.4,105.0
tta_soft_s0.2_wbf,265.0,256.3,265.0,260.8,264.7,262.4,3.8
tta_soft_s0.3_wbf,273.3,266.0,277.1,271.2,277.4,273.0,4.7
tta_sahi_soft_s0.2_wbf,1209.5,1056.6,1106.4,1184.0,941.4,1099.6,107.4
tta_sahi_soft_s0.3_wbf,1218.8,1055.0,1107.9,1184.2,941.4,1101.5,110.0


### Results

In [ ]:
fold_df = combo_results()
order = list(COMBOS)
complete = fold_df.groupby('config').fold.nunique() == len(FOLDS)
not_evaluated = [name for name in order if not complete.get(name, False)]
print('Configurations without complete results:', not_evaluated or 'none')
fold_df = fold_df[fold_df.config.isin([name for name in order if name not in not_evaluated])]
fold_df = fold_df.set_index('config').loc[[name for name in order if name not in not_evaluated]].reset_index()
cols = [col for col in KEYS + ['ms_per_img', 'fps', 'detections_per_img'] if col in fold_df]
stats = fold_df.groupby('config')[cols].agg(['mean', 'std'])
summary = fold_df.groupby(
    'config',
    sort=False)[['role', 'views', 'n_views', 'sahi', 'internal_nms', 'cross_slice', 'cross_view']].first()
for col in cols:
    summary[col + '_mean'], summary[col + '_std'] = stats[col, 'mean'], stats[col, 'std']

In [ ]:
T95 = 2.776
by_config = {name: config_rows.set_index('fold') for name, config_rows in fold_df.groupby('config')}
def paired(name, ref, key):
    diff = by_config[name][key] - by_config[ref][key]
    return diff.mean(), diff.std(), T95 * diff.std() / np.sqrt(len(diff)), int((diff > 0).sum())
for name in summary.index:
    for tag, ref in [('vs_tta', 'tta_wbf'), ('vs_control', CONTROL.get(name))]:
        if ref is None or ref not in by_config or ref == name:
            continue
        for key in ['mAP', 'APs']:
            m, s, ci, improved_folds = paired(name, ref, key)
            summary.loc[name, f'{tag}_d{key}_mean'], summary.loc[
                name, f'{tag}_d{key}_std'], summary.loc[name, f'{tag}_d{key}_ci95'] = m, s, ci
            summary.loc[name, f'{tag}_folds_{key}_improved'] = improved_folds
        summary.loc[name, f'{tag}_ref'] = ref
        if 'ms_per_img' in by_config[name]:
            summary.loc[
                name,
                f'{tag}_time_ratio'] = (by_config[name].ms_per_img.mean() / by_config[ref].ms_per_img.mean())
summary.index.name = 'config'
fold_df.to_csv(COMBO_OUT / 'combinations_fold_results.csv', index=False)
summary.to_csv(COMBO_OUT / 'combinations_summary.csv')

In [ ]:
format_mean_std = lambda col: summary[
    col + '_mean'].map('{:.4f}'.format) + ' ± ' + summary[col + '_std'].map('{:.4f}'.format)
table = pd.DataFrame({col: format_mean_std(col) for col in ['mAP', 'APs', 'AR']})
table['ms_per_img'] = summary.ms_per_img_mean.round(1)
for tag in ['vs_tta', 'vs_control']:
    table[f'{tag} ΔmAP (95% CI)'] = [f"{mean:+.4f} ± {ci:.4f} ({int(n_improved)}/5 ↑)" if pd.notna(mean)
                                     else '' for mean, ci, n_improved in
                                    zip(summary[f'{tag}_dmAP_mean'], summary[f'{tag}_dmAP_ci95'],
                                        summary[f'{tag}_folds_mAP_improved'])]
    table[f'{tag} ΔAP_small'] = summary[f'{tag}_dAPs_mean'].map(lambda value: f'{value:+.4f}'
                                                                if pd.notna(value) else '')
display(table)
display(fold_df.pivot(index='fold', columns='config', values='mAP')[list(summary.index)].round(4))

NEW_COMBOS = [name for name in summary.index if summary.loc[name, 'role'] in ('4.1', '4.2', '4.3')]
BEST_COMBO = summary.loc[NEW_COMBOS].mAP_mean.idxmax()
BEST_ALL = summary.loc[NEW_COMBOS + ['tta_wbf']].mAP_mean.idxmax()
b = summary.loc[BEST_COMBO]
print('Best new combination:', BEST_COMBO,
      f"| ΔmAP vs TTA + WBF {b.vs_tta_dmAP_mean:+.4f} ± {b.vs_tta_dmAP_ci95:.4f}, "
      f"{int(b.vs_tta_folds_mAP_improved)}/5 folds",
      '| best overall (incl. TTA + WBF):', BEST_ALL)

In [ ]:
checks = view_checks.groupby('fold')[['standard_equals_tta_cache',
                                      'in_model_soft_equals_captured_first_batch',
                                      'boxes_and_labels_valid']].all()
checks[
    'slices_equal_sahi_cache'] = view_checks.dropna(
        subset=['slices_equal_sahi_cache']).groupby('fold').slices_equal_sahi_cache.all()
checks = checks.join(reproduction.pivot(index='fold', columns='config',
                                        values='max_abs_diff').add_prefix('max_abs_diff_'))
checks.to_csv(COMBO_OUT / 'combinations_checks.csv')
display(checks)

surviving = view_checks.assign(before=view_checks.boxes_before_sahi_merge, after=view_checks.boxes_per_frame)
merge_counts = pd.DataFrame([dict(view=row.view, fold=row.fold, variant=variant,
                                  boxes_full_view=row.after[f'full_{variant}'],
                                  boxes_before_sahi_merge=row.before[variant],
                                  boxes_after_sahi_merge=row.after[f'sahi_{variant}'])
                             for row in surviving.itertuples() for variant in NMS_VARIANTS])
merge_counts = merge_counts.groupby('variant')[['boxes_full_view', 'boxes_before_sahi_merge',
                                                'boxes_after_sahi_merge']].mean()
merge_counts.to_csv(COMBO_OUT / 'combinations_box_counts.csv')
display(merge_counts.round(1))

Configurations without complete results: none


,mAP,APs,AR,ms_per_img,vs_tta ΔmAP (95% CI),vs_tta ΔAP_small,vs_control ΔmAP (95% CI),vs_control ΔAP_small
config,,,,,,,,
baseline,0.2110 ± 0.0337,0.0757 ± 0.0276,0.3450 ± 0.0404,25.9,-0.0385 ± 0.0083 (0/5 ↑),-0.0374,,
soft_s0.2,0.2144 ± 0.0344,0.0784 ± 0.0282,0.3709 ± 0.0415,26.6,-0.0351 ± 0.0082 (0/5 ↑),-0.0347,,
soft_s0.3,0.2144 ± 0.0342,0.0789 ± 0.0287,0.3802 ± 0.0421,26.5,-0.0352 ± 0.0081 (0/5 ↑),-0.0342,,
sahi,0.2232 ± 0.0340,0.0934 ± 0.0219,0.3868 ± 0.0445,119.0,-0.0263 ± 0.0089 (0/5 ↑),-0.0197,,
tta_wbf,0.2495 ± 0.0382,0.1131 ± 0.0236,0.4214 ± 0.0400,245.0,,,,
tta_sahi_wbf,0.2496 ± 0.0357,0.1224 ± 0.0228,0.4322 ± 0.0428,1072.4,+0.0001 ± 0.0079 (3/5 ↑),+0.0093,+0.0001 ± 0.0079 (3/5 ↑),+0.0093
tta_soft_s0.2_wbf,0.2356 ± 0.0373,0.1083 ± 0.0221,0.4226 ± 0.0399,262.4,-0.0139 ± 0.0026 (0/5 ↑),-0.0048,-0.0139 ± 0.0026 (0/5 ↑),-0.0048
tta_soft_s0.3_wbf,0.2270 ± 0.0390,0.1044 ± 0.0206,0.4229 ± 0.0402,273.0,-0.0226 ± 0.0058 (0/5 ↑),-0.0086,-0.0226 ± 0.0058 (0/5 ↑),-0.0086
tta_sahi_soft_s0.2_wbf,0.2488 ± 0.0356,0.1228 ± 0.0223,0.4334 ± 0.0412,1099.6,-0.0008 ± 0.0080 (3/5 ↑),+0.0097,-0.0009 ± 0.0005 (0/5 ↑),+0.0004


config,baseline,soft_s0.2,soft_s0.3,sahi,tta_wbf,tta_sahi_wbf,tta_soft_s0.2_wbf,tta_soft_s0.3_wbf,tta_sahi_soft_s0.2_wbf,tta_sahi_soft_s0.3_wbf
fold,,,,,,,,,,
0,0.2377,0.2408,0.2407,0.2567,0.2780,0.2822,0.2657,0.2626,0.2817,0.2815
1,0.2208,0.2252,0.2246,0.2283,0.2586,0.2518,0.2451,0.2326,0.2503,0.2507
2,0.2380,0.2426,0.2425,0.2418,0.2777,0.2709,0.2625,0.2548,0.2703,0.2705
3,0.2015,0.2040,0.2044,0.2223,0.2481,0.2533,0.2313,0.2209,0.2520,0.2524
4,0.1571,0.1595,0.1596,0.1672,0.1853,0.1900,0.1737,0.1639,0.1894,0.1899


Best new combination: tta_sahi_wbf | ΔmAP vs TTA + WBF +0.0001 ± 0.0079, 3/5 folds | best overall (incl. TTA + WBF): tta_sahi_wbf


,standard_equals_tta_cache,in_model_soft_equals_captured_first_batch,boxes_and_labels_valid,slices_equal_sahi_cache,max_abs_diff_baseline,max_abs_diff_sahi,max_abs_diff_soft_s0.2,max_abs_diff_soft_s0.3,max_abs_diff_tta_wbf
fold,,,,,,,,,
0,True,True,True,True,0.0,0.0,0.0,0.0,0.0
1,True,True,True,True,0.0,0.0,0.0,0.0,0.0
2,True,True,True,True,0.0,0.0,0.0,0.0,0.0
3,True,True,True,True,0.0,0.0,0.0,0.0,0.0
4,True,True,True,True,0.0,0.0,0.0,0.0,0.0


,boxes_full_view,boxes_before_sahi_merge,boxes_after_sahi_merge
variant,,,
s0.2,214.6,1060.5,241.3
s0.3,274.3,1376.9,246.2
std,132.1,669.1,250.5


In [ ]:
write_json(dict(COMBO_SETTINGS, settings_id=COMBO_ID, configs={name: dict(views=views, sahi=with_sahi,
                                                                          internal_nms=internal_nms,
                                                                          role=role,
                                                                          matched_control=CONTROL.get(name))
                                                             for name, (views, with_sahi, internal_nms,
                                                                        role) in COMBOS.items()},
                pipeline=['per view: flip + scale (transform min_size/max_size)',
                          'SAHI slices on the original grid, each crop flipped like the view',
                          'internal RoIHeads NMS or Soft-NMS in every pass',
                          'cross-slice SAHI NMS IoU 0.5 within a view (slices + full view), top 500',
                          'WBF over the six views in original coordinates, top 500', 'COCOeval maxDets 100'],
                soft_nms_implementation='section 4 algorithm; selection loop on CPU, bit-identical to '
                                        'soft_nms (checked)',
                not_evaluated=not_evaluated,
                checkpoints={k: dict(path=f'faster_rcnn/lr_scheduler/fold{k}/best.pth',
                                     sha256=CHECKPOINT_SHA[k]) for k in FOLDS}),
           COMBO_OUT / 'combinations_config.json')
def config_record(name):
    summary_row, rows = summary.loc[name], by_config[name]
    return dict(config=name, views=COMBOS[name][0], sahi=COMBOS[name][1], internal_nms=COMBOS[name][2],
                cv_mean={col: float(summary_row[col + '_mean']) for col in KEYS + ['ms_per_img', 'fps']},
                cv_std={col: float(summary_row[col + '_std']) for col in KEYS + ['ms_per_img', 'fps']},
                vs_tta_wbf=dict(dmAP_mean=float(summary_row.vs_tta_dmAP_mean),
                                dmAP_ci95=float(summary_row.vs_tta_dmAP_ci95),
                                dAPs_mean=float(summary_row.vs_tta_dAPs_mean),
                                folds_mAP_improved=int(summary_row.vs_tta_folds_mAP_improved),
                                time_ratio=float(summary_row
                                                 .vs_tta_time_ratio)) if name != 'tta_wbf' else None,
                folds={int(k): {col: float(rows.loc[k, col]) for col in KEYS + ['ms_per_img']}
                       for k in rows.index})
write_json(dict(selected_by='highest mean 5-fold CV mAP@[.50:.95] among the new combinations',
                best_combination=config_record(BEST_COMBO),
                best_overall_including_tta_wbf=BEST_ALL, tta_wbf=config_record('tta_wbf'),
                settings=dict(sahi=COMBO_SETTINGS['sahi'], wbf=COMBO_SETTINGS['wbf'],
                              soft_min_score=SOFT_MIN_SCORE, det_per_img=DET_PER_IMG)),
           COMBO_OUT / 'combinations_best_config.json')

In [ ]:
COMBO_PLOTS = COMBO_OUT / 'plots'
LABELS = {'baseline': 'Baseline', 'soft_s0.2': 'Soft-NMS σ=0.2', 'soft_s0.3': 'Soft-NMS σ=0.3',
          'sahi': 'SAHI', 'tta_wbf': 'TTA + WBF',
          'tta_sahi_wbf': 'TTA + SAHI', 'tta_soft_s0.2_wbf': 'TTA + Soft σ=0.2',
          'tta_soft_s0.3_wbf': 'TTA + Soft σ=0.3',
          'tta_sahi_soft_s0.2_wbf': 'TTA + SAHI + Soft σ=0.2',
          'tta_sahi_soft_s0.3_wbf': 'TTA + SAHI + Soft σ=0.3'}
ROLE_COLORS = {'check': '#9d9d9d', 'reference': '#4c78a8', '4.1': '#f58518', '4.2': '#54a24b',
               '4.3': '#e45756'}

def save_combo_plot(fig, name):
    fig.update_layout(template='plotly_white', font=dict(size=13))
    fig.write_html(COMBO_PLOTS / f'{name}.html', include_plotlyjs='directory', div_id=name)
    fig.show()

In [ ]:
names = list(summary.index)
fig = make_subplots(1, 2, subplot_titles=['mAP@[.50:.95]', 'AP_small'])
for col, key in enumerate(['mAP', 'APs'], 1):
    fig.add_bar(x=[LABELS[name] for name in names], y=summary[key + '_mean'],
                error_y=dict(array=summary[key + '_std']), showlegend=False,
                marker_color=[ROLE_COLORS[summary.loc[name, 'role']] for name in names],
                text=[f'{value:.4f}' for value in summary[key + '_mean']], row=1, col=col)
fig.update_layout(title='Baseline, single methods and combinations; mean ± std over 5 CV folds (grey = '
                        'single methods, blue = TTA + WBF)',
                  height=550)
save_combo_plot(fig, 'methods_overview')

def paired_plot(key, title, plot_name):
    fig = make_subplots(1, 2, subplot_titles=['vs TTA + WBF', 'vs matched control'])
    for col, tag in enumerate(['vs_tta', 'vs_control'], 1):
        for name in NEW_COMBOS:
            ref = 'tta_wbf' if tag == 'vs_tta' else CONTROL[name]
            diff = by_config[name][key] - by_config[ref][key]
            fig.add_scatter(x=[f'fold {k}' for k in diff.index], y=diff, mode='lines+markers',
                            name=LABELS[name], legendgroup=name, showlegend=col == 1,
                            line=dict(color=ROLE_COLORS[summary.loc[name, 'role']],
                                      dash='dot' if name.endswith('0.3_wbf') else 'solid'),
                            hovertemplate=f'{LABELS[name]} − {LABELS[ref]}<br>%{{x}}: '
                                          f'%{{y:+.4f}}<extra></extra>',
                            row=1, col=col)
        fig.add_hline(y=0, line_color='black', line_width=1, row=1, col=col)
    fig.update_layout(title=title, yaxis_title='paired difference')
    save_combo_plot(fig, plot_name)

paired_plot('mAP', 'Paired mAP difference per fold (CV held-out folds)', 'paired_map_differences')
paired_plot('APs', 'Paired AP_small difference per fold (CV held-out folds)', 'ap_small_differences')

In [ ]:
fig = go.Figure()
for role, subset in summary.groupby('role', sort=False):
    fig.add_scatter(x=subset.ms_per_img_mean, y=subset.mAP_mean, mode='markers+text', name=role,
                    text=[LABELS[name] for name in subset.index], textposition='top center',
                    marker=dict(size=11, color=ROLE_COLORS[role]),
                    error_x=dict(array=subset.ms_per_img_std, thickness=1),
                    error_y=dict(array=subset.mAP_std, thickness=1),
                    hovertemplate='%{text}<br>%{x:.0f} ms/frame<br>mAP %{y:.4f}<extra></extra>')
fig.update_layout(title='Time per original frame vs mean CV mAP (all timed in one session)',
                  xaxis=dict(title='ms / frame (batch 1, AMP, log scale)', type='log'),
                  yaxis_title='mAP@[.50:.95]', height=600)
save_combo_plot(fig, 'accuracy_vs_latency')

groups = [('TTA + Soft-NMS', 'tta_soft_s{}_wbf', 'tta_wbf'),
          ('TTA + SAHI + Soft-NMS', 'tta_sahi_soft_s{}_wbf', 'tta_sahi_wbf')]
fig = make_subplots(1, 2, subplot_titles=['mAP@[.50:.95]', 'AP_small'])
for col, key in enumerate(['mAP', 'APs'], 1):
    for sigma, color in [('control', '#4c78a8'), (0.2, '#54a24b'), (0.3, '#e45756')]:
        config_names = [control if sigma == 'control' else pattern.format(sigma)
                        for _, pattern, control in groups]
        fig.add_bar(x=[group[0] for group in groups], y=summary.loc[config_names, key + '_mean'],
                    error_y=dict(array=summary.loc[config_names, key + '_std']),
                    name='standard NMS (matched control)' if sigma == 'control' else f'σ = {sigma}',
                    marker_color=color, showlegend=col == 1,
                    text=[f'{value:.4f}' for value in summary.loc[config_names, key + '_mean']], row=1,
                    col=col)
fig.update_layout(title='Soft-NMS σ = 0.2 vs σ = 0.3 inside the combinations; mean ± std over 5 CV folds',
                  barmode='group')
save_combo_plot(fig, 'soft_nms_sigma_in_combinations')

In [ ]:
def combo_figure(rgb, image_info, panels, title):
    h, w = rgb.shape[:2]
    anns = [annotation for annotation in coco_train['annotations']
            if annotation['image_id'] == image_info['id']]
    xyxy = lambda annotation: [annotation['bbox'][0], annotation['bbox'][1],
                               annotation['bbox'][0] + annotation['bbox'][2],
                               annotation['bbox'][1] + annotation['bbox'][3]]
    small = Image.fromarray(rgb)
    small.thumbnail((1280, 1280))
    buffer = io.BytesIO()
    small.save(buffer, 'JPEG', quality=85)
    source = 'data:image/jpeg;base64,' + base64.b64encode(buffer.getvalue()).decode()
    fig = make_subplots(1, len(panels), horizontal_spacing=.01,
                        subplot_titles=[f"{name}: {int((pred['scores'] >= .3).sum())} boxes ≥ 0.3"
                                        for name, pred in panels.items()])
    def outline(boxes, color, text=None, name=''):
        xs, ys, hover = [], [], []
        for i, (x1, y1, x2, y2) in enumerate(boxes):
            xs += [x1, x2, x2, x1, x1, None]
            ys += [y1, y1, y2, y2, y1, None]
            hover += [text[i] if text else name] * 5 + [None]
        return go.Scatter(x=xs, y=ys, mode='lines', line=dict(color=color, width=1.2), text=hover,
                          hoverinfo='text', showlegend=False)
    for col, (name, pred) in enumerate(panels.items(), 1):
        keep = pred['scores'] >= .3
        fig.add_trace(outline([xyxy(annotation) for annotation in anns if not annotation.get('iscrowd', 0)],
                              'lime', name='ground truth'), 1, col)
        fig.add_trace(outline([xyxy(annotation) for annotation in anns if annotation.get('iscrowd', 0)],
                              'yellow', name='ignored region'), 1, col)
        fig.add_trace(outline(pred['boxes'][keep].tolist(), 'red', [
            f"{CATS[LABEL_TO_CAT[int(x)]]} {y:.2f}"
            for x, y in zip(pred['labels'][keep], pred['scores'][keep])]), 1, col)
        fig.add_layout_image(source=source, xref=f'x{col}', yref=f'y{col}', x=0, y=0, sizex=w, sizey=h,
                             xanchor='left', yanchor='top', sizing='stretch', layer='below')
    fig.update_xaxes(range=[0, w], visible=False)
    fig.update_yaxes(range=[h, 0], visible=False)
    fig.update_layout(title=title
                      + ' · green = ground truth, red = prediction (score ≥ 0.3), yellow = ignored',
                      width=1800, height=int(600 * h / w) + 120, margin=dict(l=10, r=10, t=90, b=10),
                      template='plotly_white')
    return fig

In [ ]:
base_folds = by_config['baseline'].mAP
rep_fold = int(base_folds.sort_values().index[len(base_folds) // 2])
images = fold_images(rep_fold)
n_objects = (pd.Series([annotation['image_id'] for annotation in coco_train['annotations']
                        if not annotation.get('iscrowd', 0)]).value_counts())
examples = [('most_objects', max(images, key=lambda image_info: n_objects.get(image_info['id'], 0))),
            ('random', images[int(np.random.default_rng(SEED).integers(len(images)))])]
shown = list(dict.fromkeys(['tta_wbf', 'tta_sahi_wbf', BEST_COMBO]))
cached = {name: dict(combo_predictions(name, rep_fold)) for name in shown}
saved = []
for label, image_info in examples:
    rgb = read_images([image_info])[0]
    fig = combo_figure(rgb, image_info, {LABELS[name]: cached[name][int(image_info['id'])] for name in shown},
                       f"Fold {rep_fold} held-out frame {image_info['file_name']} "
                       f"({n_objects.get(image_info['id'], 0)} objects)")
    path = COMBO_OUT / 'visualizations' / f"fold{rep_fold}_{label}_{image_info['id']}.html"
    fig.write_html(path, include_plotlyjs='directory')
    saved.append(path.name)
print('Saved to', COMBO_OUT / 'visualizations', ':', saved)

Saved to E:\ucu_hw2\faster_rcnn\inference_experiments\combinations\visualizations : ['fold1_most_objects_1128.html', 'fold1_random_18517.html']



## Final Evaluation Official Validation

We compare two configurations selected based on the results of previous experiments:

Baseline: Faster R-CNN lr_scheduler with standard NMS.

TTA + WBF: the best configuration from Experiment 3, using scales of 1.0×, 1.25×, and 1.5× with and without horizontal flips (6 views), merging predictions with WBF.

We do not evaluate TTA + SAHI on the official validation set because it showed almost no improvement in CV mAP (+0.0001) in the previous experiment, while being 4.4 times slower than TTA + WBF.

In [ ]:
VAL_OUT = ROOT / 'faster_rcnn/official_validation'
for subfolder in ['cache', 'runs', 'speed', 'plots', 'visualizations']:
    (VAL_OUT / subfolder).mkdir(parents=True, exist_ok=True)
with open(DATA / 'val/annotations_coco.json') as f:
    coco_val = json.load(f)
val_images = [dict(image_info, split='val') for image_info in coco_val['images']]
VAL_IDS = [int(image_info['id']) for image_info in val_images]
VAL_DIMS = {int(image_info['id']): (image_info['height'], image_info['width']) for image_info in val_images}

In [ ]:
def read_images(images):
    arrays = []
    for image in images:
        with Image.open(DATA / image.get('split', 'train') / image['file_name']) as picture:
            arrays.append(np.array(picture.convert('RGB')))
    return arrays

In [ ]:
combo_config = json.loads((COMBO_OUT / 'combinations_config.json').read_text())
assert (TTA_VIEWS == tta_best['views'] == combo_config['configs']['tta_wbf']['views']
        and tta_best['merge'] == 'wbf')
assert combo_config['wbf'] == dict(iou_thr=WBF_IOU, skip_box_thr=0.0, conf_type='avg', weights='equal')
CONFIGS_VAL = {'baseline': 'A. Baseline (standard NMS)', 'tta_wbf': 'B. TTA + WBF'}
CV_NAME = {'baseline': 'baseline', 'tta_wbf': 'tta_wbf'}
VAL_SETTINGS = dict(split='val', frames=len(val_images), tta_views=TTA_VIEWS,
                    tta_batch=COMBO_SETTINGS['tta_batch'], wbf=combo_config['wbf'], det_per_img=DET_PER_IMG,
                    inference=INFERENCE, amp=AMP, coco_max_dets=MAX_DETS,
                    eval_batch=EVAL_BATCH, versions=COMBO_SETTINGS['versions'])
VAL_ID = hashlib.sha256(json.dumps(VAL_SETTINGS, sort_keys=True).encode()).hexdigest()[:16]

history = (pd.read_csv(ROOT / 'faster_rcnn/official_val_folds.csv')
           .query("run == 'lr_scheduler'").set_index('fold'))
cv_results = pd.read_csv(COMBO_OUT / 'combinations_fold_results.csv').set_index(['config', 'fold'])
print('official validation:', len(val_images), 'frames |', 'recorded baseline mAP',
      round(history.mAP.mean(), 4), 'AP50', round(history.AP50.mean(), 4),
      'AP_small', round(history.APs.mean(), 4), '| settings id', VAL_ID)

official validation: 2846 frames | recorded baseline mAP 0.1909 AP50 0.3937 AP_small 0.0879 | settings id bd2dc3593ce71bc6


### Dataset and protocol checks

In [ ]:
train_videos = {image_info['video'] for image_info in coco_train['images']}
val_videos = {image_info['video'] for image_info in coco_val['images']}
flight = lambda video: video.split('_')[0]
assert (not train_videos & val_videos
        and not {flight(video) for video in train_videos} & {flight(video) for video in val_videos})
assert set(folds.video) <= train_videos and not set(folds.video) & val_videos
assert coco_val['categories'] == coco_train['categories']
audit_ids = (pd.read_csv(ROOT / 'model_comparison/audit/official_val_image_ids.csv').image_id
             .astype(int).tolist())
assert VAL_IDS == audit_ids and len(set(VAL_IDS)) == len(VAL_IDS)

def file_md5(path):
    return hashlib.md5(path.read_bytes()).hexdigest()
cv_frames = [image_info for k in FOLDS for image_info in fold_images(k)]
with ThreadPoolExecutor(8) as pool:
    val_md5 = set(pool.map(file_md5, [DATA / 'val' / image_info['file_name'] for image_info in val_images]))
    cv_md5 = set(pool.map(file_md5, [DATA / 'train' / image_info['file_name'] for image_info in cv_frames]))
assert not val_md5 & cv_md5
audit = json.loads((ROOT / 'model_comparison/audit/overlap_check.json').read_text())
val_gt = make_gt(coco_val, VAL_IDS)
n_crowd = sum(annotation.get('iscrowd', 0) for annotation in coco_val['annotations'])
print(f'{len(val_videos)} validation videos, {len(val_images)} frames, '
      f'{len(coco_val["annotations"]) - n_crowd} objects + {n_crowd} ignore boxes | '
      f'shared videos/flights with train: 0 | identical JPEGs val vs {len(cv_frames)} CV frames: 0 | earlier '
      f'audit (all files): '
      f"train/val {audit['identical_files_train_val']}, val/test {audit['identical_files_val_test']}")

resolutions = pd.DataFrame(val_images).groupby(['width', 'height']).size().rename('val_frames').reset_index()
resolutions['cv_frames'] = [
    sum((image_info['width'], image_info['height']) == (w, h) for image_info in cv_frames)
    for w, h in zip(resolutions.width, resolutions.height)]
input_scale = lambda h, w: min(INFERENCE['min_size'] / min(h, w), INFERENCE['max_size'] / max(h, w))
for s in [1.0, 1.25, 1.5]:
    resolutions[f'model_input_{s}x'] = [
        f"{round(w * input_scale(h, w) * s)}×{round(h * input_scale(h, w) * s)}"
        for w, h in zip(resolutions.width, resolutions.height)]
display(resolutions)

7 validation videos, 2846 frames, 114132 objects + 31960 ignore boxes | shared videos/flights with train: 0 | identical JPEGs val vs 4868 CV frames: 0 | earlier audit (all files): train/val 0, val/test 0


,width,height,val_frames,cv_frames,model_input_1.0x,model_input_1.25x,model_input_1.5x
0,1344,756,827,1134,1280×720,1600×900,1920×1080
1,1904,1071,459,2593,1280×720,1600×900,1920×1080
2,2688,1512,233,707,1280×720,1600×900,1920×1080
3,2720,1530,349,96,1280×720,1600×900,1920×1080
4,3840,2160,978,0,1280×720,1600×900,1920×1080


### Inference

In [ ]:
@torch.inference_mode()
def val_view(model, images, view):
    flip, scale = parse_view(view)
    set_scale(model, scale)
    full, seconds = [], 0.
    try:
        for chunk, image_tensors in batches(images, TTA_BATCH[scale]):
            image_tensors = [image.to(DEVICE, non_blocking=True) for image in image_tensors]
            torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
            start = time.perf_counter()
            with torch.autocast(device_type=DEVICE.type, enabled=AMP):
                preds = model([flip_image(image, flip) for image in image_tensors])
            full += [to_numpy(unflip(pred['boxes'].float(), flip, *image.shape[-2:]), pred['scores'].float(),
                              pred['labels']) for image, pred in zip(image_tensors, preds)]
            torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
            seconds += time.perf_counter() - start
    finally:
        set_scale(model, 1.0)
    return full, seconds

In [ ]:
def pack(prefix, rows):
    return {prefix + '_counts': np.array([len(row[1]) for row in rows]),
            prefix + '_boxes': np.concatenate([row[0] for row in rows]).reshape(-1, 4).astype(np.float32),
            prefix + '_scores': np.concatenate([row[1] for row in rows]).astype(np.float32),
            prefix + '_labels': np.concatenate([row[2] for row in rows]).astype(np.int64)}

def unpack(data, prefix):
    ends = np.cumsum(data[prefix + '_counts'])
    boxes, scores, labels = data[prefix + '_boxes'], data[prefix + '_scores'], data[prefix + '_labels']
    return [(boxes[i:j], scores[i:j], labels[i:j]) for i, j in zip(ends - data[prefix + '_counts'], ends)]

def cache_path(fold, view, suffix='.npz'):
    return VAL_OUT / 'cache' / f'fold{fold}_{view}{suffix}'

def cached(path, fold):
    if not path.with_suffix('.json').exists() or not path.exists():
        return False
    saved = json.loads(path.with_suffix('.json').read_text())
    return saved['settings_id'] == VAL_ID and saved['checkpoint_sha256'] == CHECKPOINT_SHA[fold]

def run_json(name, fold):
    return VAL_OUT / 'runs' / f'{name}_fold{fold}.json'

In [ ]:
model, run_config, checkpoint = load_model(FOLDS[0])
first_cv_frames = fold_images(FOLDS[0])[:8]
for view in TTA_VIEWS:
    full, _ = val_view(model, first_cv_frames, view)
    assert all(same_arrays(new_row, cached_row[1:])
               for new_row, cached_row in zip(full, load_view(view_path(FOLDS[0], view))[:8])), view
del model
print('val_view reproduces the cached Experiment 3 views exactly (CV fold 0, first 8 frames, all six views).')

val_view reproduces the cached Experiment 3 views exactly (CV fold 0, first 8 frames, all six views).


### Evaluation runs

In [ ]:
def valid_scores(scores):
    return len(scores) == 0 or (np.isfinite(scores).all() and scores.min() > 0 and scores.max() <= 1)

def check_rows(rows):
    return all(boxes_valid(boxes, labels, *VAL_DIMS[i]) and valid_scores(scores)
               for i, (boxes, scores, labels) in zip(VAL_IDS, rows))

def evaluate_val(name, fold, preds, seconds, extra=None):
    start = time.perf_counter()
    metrics = coco_metrics([(i, dict(boxes=boxes, scores=scores, labels=labels))
                            for i, (boxes, scores, labels) in zip(VAL_IDS, preds)], val_gt)
    seconds = dict(seconds, coco_eval=time.perf_counter() - start)
    row = dict(config=name, label=CONFIGS_VAL[name], checkpoint=f'fold{fold}', fold=fold, frames=len(preds),
               **metrics,
               detections_per_img=float(np.mean([len(pred[1]) for pred in preds])),
               eval_time_s=float(sum(seconds.values())))
    write_json(dict(settings_id=VAL_ID, checkpoint_sha256=CHECKPOINT_SHA[fold], row=row, time_s=seconds,
                    **(extra or {})), run_json(name, fold))
    return row

In [ ]:
def run_baseline_val(fold):
    path = cache_path(fold, 'baseline')
    if cached(path, fold) and run_json('baseline', fold).exists():
        return
    model, run_config, checkpoint = load_model(fold)
    torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
    start = time.perf_counter()
    preds = [(pred['boxes'].float().numpy(), pred['scores'].float().numpy(), pred['labels'].numpy())
             for _, pred in predict(model, val_images)]
    seconds = dict(inference=time.perf_counter() - start)
    assert len(preds) == len(VAL_IDS) and check_rows(preds)
    row = evaluate_val('baseline', fold, preds, seconds)
    diff = max(abs(row[k] - history.loc[fold, k]) for k in KEYS)
    write_json(dict(json.loads(run_json('baseline', fold).read_text()), max_abs_diff_vs_recorded=diff),
               run_json('baseline', fold))
    if diff > 1e-9:
        raise RuntimeError(f'fold {fold}: baseline differs from the recorded official-validation result by '
                           f'{diff}; investigate before continuing')
    np.savez(path, ids=np.array(VAL_IDS), **pack('base', preds))
    write_json(dict(settings_id=VAL_ID, checkpoint_sha256=CHECKPOINT_SHA[fold]), path.with_suffix('.json'))
    del model
    print(f"  fold {fold}: baseline mAP {row['mAP']:.4f} (recorded {history.loc[fold, 'mAP']:.4f}, max abs "
          f"diff {diff:.1e}) in {seconds['inference'] / 60:.1f} min")

In [ ]:
def run_views_val(fold):
    todo = [view for view in TTA_VIEWS if not cached(cache_path(fold, view), fold)]
    if not todo:
        return
    model, run_config, checkpoint = load_model(fold)
    for view in todo:
        full, seconds = val_view(model, val_images, view)
        assert len(full) == len(VAL_IDS) and check_rows(full)
        info = dict(settings_id=VAL_ID, checkpoint_sha256=CHECKPOINT_SHA[fold], view=view, frames=len(full),
                    seconds=seconds)
        if view == 'none@1.0':
            baseline_rows = unpack(np.load(cache_path(fold, 'baseline')), 'base')
            info['full_equals_baseline'] = all(same_arrays(view_row, baseline_row)
                                               for view_row, baseline_row in zip(full, baseline_rows))
            assert info['full_equals_baseline'], 'none@1.0 full view differs from the baseline predictions'
        np.savez(cache_path(fold, view), ids=np.array(VAL_IDS), **pack('full', full))
        write_json(info, cache_path(fold, view, '.json'))
        print(f"  fold {fold} {view}: {seconds / 60:.1f} min")
    del model
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None

In [ ]:
def run_fused_val(fold):
    if (run_json('tta_wbf', fold).exists()
        and json.loads(run_json('tta_wbf', fold).read_text())['settings_id'] == VAL_ID):
        return
    data = {view: np.load(cache_path(fold, view)) for view in TTA_VIEWS}
    info = {view: json.loads(cache_path(fold, view, '.json').read_text()) for view in TTA_VIEWS}
    assert all(list(view_data['ids']) == VAL_IDS for view_data in data.values())
    views = [unpack(data[view], 'full') for view in TTA_VIEWS]
    start = time.perf_counter()
    preds = []
    for k, i in enumerate(VAL_IDS):
        parts = [views[j][k] for j in range(len(TTA_VIEWS))]
        assert len(parts) == 6
        preds.append(tta_merge(parts, 'wbf', *VAL_DIMS[i]))
    seconds = dict(view_passes=sum(info[view]['seconds'] for view in TTA_VIEWS),
                   wbf=time.perf_counter() - start)
    assert check_rows(preds)
    row = evaluate_val('tta_wbf', fold, preds, seconds)
    print(f"  fold {fold} tta_wbf: mAP {row['mAP']:.4f} | AP_small {row['APs']:.4f}")

In [ ]:
started = time.time()
for fold in FOLDS:
    run_baseline_val(fold)
print(f'baseline verified for all checkpoints ({(time.time() - started) / 60:.1f} min)')
for fold in FOLDS:
    run_views_val(fold)
    run_fused_val(fold)
    print(f'fold {fold} done after {(time.time() - started) / 60:.1f} min')

baseline verified for all checkpoints (0.0 min)
fold 0 done after 0.0 min
fold 1 done after 0.0 min
fold 2 done after 0.0 min
fold 3 done after 0.0 min
fold 4 done after 0.0 min


### Speed

In [ ]:
speed_ids = (pd.read_csv(ROOT / 'model_comparison/audit/speed_image_ids.csv').id.astype(int)
             .tolist())
assert len(speed_ids) == 200 and set(speed_ids) <= set(VAL_IDS)
SPEED_WARMUP_VAL = 10

In [ ]:
def run_val_speed(fold):
    path = VAL_OUT / 'speed' / f'fold{fold}.json'
    if path.exists():
        old = json.loads(path.read_text())
        if old['settings_id'] == VAL_ID and old['checkpoint_sha256'] == CHECKPOINT_SHA[fold]:
            return old
    busy = gpu_busy()
    if busy is not None and busy > 10:
        print(f'WARNING: GPU {busy}% busy before timing fold {fold} (another process)')
    model, run_config, checkpoint = load_model(fold)
    by_id = {int(image_info['id']): image_info for image_info in val_images}
    frames = [torch.from_numpy(array).permute(2, 0, 1).contiguous().to(DEVICE)
              for array in read_images([by_id[i] for i in speed_ids])]
    order = frames[:SPEED_WARMUP_VAL] + frames
    times = {name: [] for name in CONFIGS_VAL}
    for round_index, frame in enumerate(order):
        for name in CONFIGS_VAL:
            torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
            start = time.perf_counter()
            combo_frame(model, frame.float().div_(255), CV_NAME[name])
            torch.cuda.synchronize() if DEVICE.type == 'cuda' else None
            if round_index >= SPEED_WARMUP_VAL:
                times[name].append(time.perf_counter() - start)
    result = dict(settings_id=VAL_ID, checkpoint_sha256=CHECKPOINT_SHA[fold], fold=fold, frames=len(frames),
                  warmup=SPEED_WARMUP_VAL,
                  gpu_utilization_before=busy,
                  device=torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu',
                  ms_per_img={name: float(np.mean(durations) * 1000) for name, durations in times.items()},
                  ms_per_img_by_width={name: {int(w): float(np.mean([x for x, frame in zip(durations, frames)
                                                                     if frame.shape[-1] == w]) * 1000)
                                           for w in sorted({
                                               frame.shape[-1]
                                               for frame in frames})} for name, durations in times.items()})
    write_json(result, path)
    del model, frames, order
    torch.cuda.empty_cache() if DEVICE.type == 'cuda' else None
    return result

In [ ]:
speed_val = {k: run_val_speed(k) for k in FOLDS}
speed_table = pd.DataFrame({k: result['ms_per_img']
                            for k, result in speed_val.items()}).loc[list(CONFIGS_VAL)]
speed_table['mean'], speed_table['std'] = speed_table[FOLDS].mean(axis=1), speed_table[FOLDS].std(axis=1)
display(speed_table.round(1).rename(columns=lambda col: f'fold {col}' if isinstance(col, int) else col))
display(pd.DataFrame({
    name: pd.DataFrame([result['ms_per_img_by_width'][name] for result in speed_val.values()]).mean()
    for name in CONFIGS_VAL}).round(1)
        .rename_axis('frame width'))

,fold 0,fold 1,fold 2,fold 3,fold 4,mean,std
baseline,28.7,28.2,27.6,28.3,28.0,28.2,0.4
tta_wbf,275.8,269.2,265.9,268.1,268.6,269.5,3.7


,baseline,tta_wbf
frame width,,
1344,28.0,267.1
1904,27.9,267.6
2688,27.8,276.9
2720,28.0,271.7
3840,29.2,264.4


### Results

In [ ]:
rows = [json.loads(run_json(name, k).read_text()) for name in CONFIGS_VAL for k in FOLDS
        if run_json(name, k).exists()]
val_df = pd.DataFrame([record['row'] for record in rows])
val_df['ms_per_img'] = [speed_val[record.fold]['ms_per_img'][record.config] for record in val_df.itertuples()]
val_df['fps'] = 1000 / val_df.ms_per_img
missing = [(name, k) for name in CONFIGS_VAL for k in FOLDS
           if not ((val_df.config == name) & (val_df.fold == k)).any()]
print('Unfinished runs:', missing or 'none')
assert not missing and (val_df.frames == len(VAL_IDS)).all()
val_df.to_csv(VAL_OUT / 'official_val_checkpoint_results.csv', index=False)

In [ ]:
cols = KEYS + ['ms_per_img', 'fps', 'detections_per_img', 'eval_time_s']
stats = val_df.groupby('config')[cols].agg(['mean', 'std']).loc[list(CONFIGS_VAL)]
val_summary = pd.DataFrame(index=list(CONFIGS_VAL))
val_summary['label'] = list(CONFIGS_VAL.values())
for col in cols:
    val_summary[col + '_mean'], val_summary[col + '_std'] = stats[col, 'mean'], stats[col, 'std']
for name in CONFIGS_VAL:
    cv_rows = cv_results.loc[CV_NAME[name]]
    val_summary.loc[
        name, 'cv_mAP_mean'], val_summary.loc[name, 'cv_mAP_std'] = cv_rows.mAP.mean(), cv_rows.mAP.std()
    val_summary.loc[name, 'cv_APs_mean'], val_summary.loc[
        name, 'cv_ms_per_img_mean'] = cv_rows.APs.mean(), cv_rows.ms_per_img.mean()
val_summary.index.name = 'config'
val_summary.to_csv(VAL_OUT / 'official_val_summary.csv')

In [ ]:
val_by_config = {name: config_rows.set_index('fold') for name, config_rows in val_df.groupby('config')}
PAIRS = [('tta_wbf', 'baseline')]
paired = []
for a, b in PAIRS:
    for k in FOLDS:
        record = dict(comparison=f'{a} - {b}', fold=k)
        for col in KEYS + ['ms_per_img']:
            record[f'd{col}'] = val_by_config[a].loc[k, col] - val_by_config[b].loc[k, col]
            record[f'rel_{col}_pct'] = 100 * record[f'd{col}'] / val_by_config[b].loc[k, col]
        record['cv_dmAP'] = cv_results.loc[(CV_NAME[a], k), 'mAP'] - cv_results.loc[(CV_NAME[b], k), 'mAP']
        paired.append(record)
paired = pd.DataFrame(paired)
paired.to_csv(VAL_OUT / 'official_val_paired_differences.csv', index=False)
paired_summary = paired.groupby('comparison', sort=False).agg(dmAP_mean=('dmAP', 'mean'),
                                                              dmAP_std=('dmAP', 'std'),
                                                              rel_mAP_pct=('rel_mAP_pct', 'mean'),
                                                              dAPs_mean=('dAPs', 'mean'),
                                                              rel_APs_pct=('rel_APs_pct', 'mean'),
                                                              checkpoints_improved=(
                                                                  'dmAP',
                                                                  lambda diffs: int((diffs > 0).sum())),
                                                              cv_dmAP_mean=('cv_dmAP', 'mean'))

In [ ]:
format_mean_std = lambda col: val_summary[
    col + '_mean'].map('{:.4f}'.format) + ' ± ' + val_summary[col + '_std'].map('{:.4f}'.format)
table = pd.DataFrame({col: format_mean_std(col) for col in KEYS}, index=val_summary.index)
table['ms_per_img'] = (val_summary.ms_per_img_mean.map('{:.1f}'.format) + ' ± '
                       + val_summary.ms_per_img_std.map('{:.1f}'.format))
table['eval time (min)'] = (val_summary.eval_time_s_mean / 60).round(1)
table['CV mAP'] = val_summary.cv_mAP_mean.map('{:.4f}'.format)
display(table.rename(index=CONFIGS_VAL))
display(paired_summary.round(4))
display(val_df.pivot(index='fold', columns='config', values='mAP')[list(CONFIGS_VAL)].round(4))

base_checks = pd.DataFrame([dict(fold=k, **{
    field: value for field, value in json.loads(run_json('baseline', k).read_text()).items()
    if field == 'max_abs_diff_vs_recorded'},
                                 none_full_equals_baseline=json.loads(cache_path(
                                     k, 'none@1.0', '.json').read_text())['full_equals_baseline'])
                            for k in FOLDS]).set_index('fold')
display(base_checks)

Unfinished runs: none


,mAP,AP50,AP75,APs,APm,APl,AR,ms_per_img,eval time (min),CV mAP
config,,,,,,,,,,
A. Baseline (standard NMS),0.1909 ± 0.0062,0.3937 ± 0.0074,0.1629 ± 0.0090,0.0879 ± 0.0074,0.1716 ± 0.0023,0.3612 ± 0.0121,0.3232 ± 0.0099,28.2 ± 0.4,3.9,0.2110
B. TTA + WBF,0.2184 ± 0.0056,0.4459 ± 0.0062,0.1887 ± 0.0079,0.1139 ± 0.0059,0.2047 ± 0.0019,0.3809 ± 0.0147,0.3921 ± 0.0266,269.5 ± 3.7,19.1,0.2495


,dmAP_mean,dmAP_std,rel_mAP_pct,dAPs_mean,rel_APs_pct,checkpoints_improved,cv_dmAP_mean
comparison,,,,,,,
tta_wbf - baseline,0.0275,0.0016,14.4374,0.026,29.9418,5,0.0385


config,baseline,tta_wbf
fold,,
0,0.1960,0.2211
1,0.1858,0.2140
2,0.1962,0.2252
3,0.1828,0.2113
4,0.1935,0.2202


,max_abs_diff_vs_recorded,none_full_equals_baseline
fold,,
0,0.0,True
1,0.0,True
2,0.0,True
3,0.0,True
4,0.0,True


In [ ]:
write_json(dict(VAL_SETTINGS, settings_id=VAL_ID, configurations={
                    'baseline': 'torchvision Faster R-CNN lr_scheduler, standard RoIHeads NMS (IoU 0.5, '
                                'score > 0.01, top 500), batch 4, AMP',
                    'tta_wbf': dict(views=TTA_VIEWS,
                                    scale_control='model.transform min_size=720*s, max_size=1280*s',
                                    flip_restoration='x1=W-x2, x2=W-x1',
                                    fusion=combo_config['wbf'], top=DET_PER_IMG,
                                    source='Experiment 3 best (tta_best_config.json)')},
                not_evaluated=dict(tta_sahi='dropped before official validation: same CV mAP as TTA + WBF '
                                            '(+0.0001) at 4.4x latency (Experiment 4)'),
                selection='all three fixed by internal 5-fold CV before this evaluation; nothing tuned on '
                          'official validation',
                evaluation='COCOeval bbox on all 2,846 official validation frames, val/annotations_coco.json '
                           'incl. iscrowd ignore regions, '
                           'default IoU thresholds and area ranges, maxDets=[1,10,100]; each checkpoint '
                           'evaluates the full set; no ensembling',
                speed=dict(frames='audit/speed_image_ids.csv (200 frames, 40 per resolution), uint8 on GPU',
                           warmup_rounds=SPEED_WARMUP_VAL,
                           batch='1 original frame', precision='AMP fp16',
                           timing='torch.cuda.synchronize before/after each frame, configurations '
                                  'interleaved'),
                checkpoints={k: dict(path=f'faster_rcnn/lr_scheduler/fold{k}/best.pth',
                                     sha256=CHECKPOINT_SHA[k]) for k in FOLDS},
                environment=dict(COMBO_SETTINGS['versions'], python=sys.version.split()[0],
                                 device=torch.cuda.get_device_name() if DEVICE.type == 'cuda' else 'cpu')),
           VAL_OUT / 'official_val_config.json')

In [ ]:
VAL_PLOTS = VAL_OUT / 'plots'
VAL_COLORS = {'baseline': '#4c78a8', 'tta_wbf': '#f58518'}

def save_val_plot(fig, name):
    fig.update_layout(template='plotly_white', font=dict(size=13))
    fig.write_html(VAL_PLOTS / f'{name}.html', include_plotlyjs='directory', div_id=name)
    fig.show()

def config_bars(key, title, name):
    fig = go.Figure()
    for config in CONFIGS_VAL:
        checkpoint_rows = val_by_config[config]
        fig.add_bar(x=[CONFIGS_VAL[config]], y=[checkpoint_rows[key].mean()],
                    error_y=dict(array=[checkpoint_rows[key].std()]), marker_color=VAL_COLORS[config],
                    name=CONFIGS_VAL[config],
                    text=[f'{checkpoint_rows[key].mean():.4f}'], textposition='outside')
        fig.add_scatter(x=[CONFIGS_VAL[config]] * len(checkpoint_rows), y=checkpoint_rows[key],
                        mode='markers', marker=dict(color='black', size=7), showlegend=False,
                        text=[f'fold {k}' for k in checkpoint_rows.index],
                        hovertemplate='%{text}: %{y:.4f}<extra></extra>')
    fig.update_layout(title=title, yaxis_title=key, showlegend=False)
    save_val_plot(fig, name)

In [ ]:
config_bars(
    'mAP', 'Official validation: COCO mAP@[.50:.95] (bars: mean ± std over 5 checkpoints, dots: checkpoints)',
    'official_val_map_by_config')
config_bars('APs', 'Official validation: AP_small (bars: mean ± std over 5 checkpoints, dots: checkpoints)',
            'official_val_ap_small_by_config')

fig = make_subplots(1, 2, subplot_titles=['ΔmAP@[.50:.95]', 'ΔAP_small'])
for col, key in enumerate(['dmAP', 'dAPs'], 1):
    for comparison, subset in paired.groupby('comparison', sort=False):
        fig.add_scatter(x=[f'fold {k}' for k in subset.fold], y=subset[key], mode='lines+markers',
                        name=comparison, legendgroup=comparison, showlegend=col == 1,
                        hovertemplate=comparison + '<br>%{x}: %{y:+.4f}<extra></extra>', row=1, col=col)
    fig.add_hline(y=0, line_color='black', line_width=1, row=1, col=col)
fig.update_layout(title='Official validation: paired differences per checkpoint (same frames)')
save_val_plot(fig, 'official_val_paired_differences')

In [ ]:
fig = make_subplots(1, 2, subplot_titles=['mAP@[.50:.95]', 'AP_small'])
for col, key in enumerate(['mAP', 'APs'], 1):
    for split, color in [('internal 5-fold CV (held-out training videos)', '#9d9d9d'),
                         ('official validation (separate videos)', '#e45756')]:
        is_cv = split.startswith('internal')
        y = [cv_results.loc[CV_NAME[name]][key].mean() if is_cv else val_by_config[name][key].mean()
             for name in CONFIGS_VAL]
        error_values = [cv_results.loc[CV_NAME[name]][key].std() if is_cv else val_by_config[name][key].std()
                        for name in CONFIGS_VAL]
        fig.add_bar(x=list(CONFIGS_VAL.values()), y=y, error_y=dict(array=error_values), name=split,
                    marker_color=color, showlegend=col == 1,
                    text=[f'{value:.4f}' for value in y], row=1, col=col)
fig.update_layout(
    title='CV vs official validation (mean ± std over 5 checkpoints; CV: each checkpoint on its own fold)',
    barmode='group')
save_val_plot(fig, 'cv_vs_official_val')

fig = go.Figure()
for n in CONFIGS_VAL:
    subset = val_by_config[n]
    fig.add_scatter(x=[subset.ms_per_img.mean()], y=[subset.mAP.mean()],
                    error_x=dict(array=[subset.ms_per_img.std()]), error_y=dict(array=[subset.mAP.std()]),
                    mode='markers+text', text=[CONFIGS_VAL[n]], textposition='top center',
                    marker=dict(size=13, color=VAL_COLORS[n]), name=CONFIGS_VAL[n],
                    hovertemplate='%{text}<br>%{x:.0f} ms/frame<br>mAP %{y:.4f}<extra></extra>')
fig.update_layout(title='Official validation: mAP vs time per original frame (batch 1, AMP, 200 speed '
                        'frames)',
                  xaxis=dict(title='ms / frame (log scale)', type='log'), yaxis_title='mAP@[.50:.95]',
                  showlegend=False)
save_val_plot(fig, 'official_val_accuracy_vs_latency')

In [ ]:
SHOW = 0.3
rep_fold = int(val_by_config['baseline'].mAP.sort_values().index[len(FOLDS) // 2])
preds_rep = {'baseline': unpack(np.load(cache_path(rep_fold, 'baseline')), 'base')}
views = [unpack(np.load(cache_path(rep_fold, view)), 'full') for view in TTA_VIEWS]
preds_rep['tta_wbf'] = [tta_merge([view_preds[k] for view_preds in views], 'wbf', *VAL_DIMS[i])
                        for k, i in enumerate(VAL_IDS)]
anns_by_image = {}
for a in coco_val['annotations']:
    anns_by_image.setdefault(int(a['image_id']), []).append(a)
xyxy = lambda annotation: [annotation['bbox'][0], annotation['bbox'][1],
                           annotation['bbox'][0] + annotation['bbox'][2],
                           annotation['bbox'][1] + annotation['bbox'][3]]

In [ ]:
def frame_errors(i, pred):
    anns = anns_by_image.get(i, [])
    gt_boxes = (torch.tensor([xyxy(annotation) for annotation in anns if not annotation.get('iscrowd', 0)],
                             dtype=torch.float32).reshape(-1, 4))
    gt_labels = torch.tensor([CAT_IDS.index(annotation['category_id']) + 1 for annotation in anns
                              if not annotation.get('iscrowd', 0)])
    ignore_boxes = (torch.tensor([xyxy(annotation) for annotation in anns if annotation.get('iscrowd', 0)],
                                 dtype=torch.float32).reshape(-1, 4))
    boxes, scores, labels = pred
    keep = scores >= SHOW
    boxes, labels = torch.from_numpy(boxes[keep]), torch.from_numpy(labels[keep])
    matched, true_positives, false_positives = torch.zeros(len(gt_boxes), dtype=torch.bool), 0, 0
    iou = box_iou(boxes, gt_boxes) if len(gt_boxes) and len(boxes) else torch.zeros(len(boxes), len(gt_boxes))
    for j in range(len(boxes)):
        match_mask = (iou[j] >= .5) & ~matched & (gt_labels == labels[j])
        if match_mask.any():
            matched[torch.where(match_mask)[0][iou[j][match_mask].argmax()]] = True
            true_positives += 1
        elif len(ignore_boxes):
            intersection = ((torch.min(boxes[j, 2:], ignore_boxes[:, 2:]) - torch.max(
                boxes[j, :2], ignore_boxes[:, :2])).clamp(min=0).prod(1))
            false_positives += int((intersection / (boxes[
                j, 2:] - boxes[j, :2]).prod().clamp(min=1e-6)).max() < .5)
        else:
            false_positives += 1
    return dict(gt=len(gt_boxes), tp=true_positives, fp=false_positives, fn=len(gt_boxes) - true_positives)

In [ ]:
errors = pd.DataFrame([dict(
    image_id=i, file_name=image_info['file_name'], width=image_info['width'],
    **{f'{name}_{k}': value for name in preds_rep
                                                               for k, value in frame_errors(
                                                                   i,
                                                                   preds_rep[name][idx]).items()})
                       for idx, (i, image_info) in enumerate(zip(VAL_IDS, val_images))])
errors.to_csv(VAL_OUT / f'per_frame_errors_fold{rep_fold}.csv', index=False)
display(errors[[
    col for col in errors
    if col.endswith(('_tp', '_fp', '_fn'))]].sum().to_frame(f'fold {rep_fold}, all frames, score ≥ {SHOW}').T)

In [ ]:
def val_figure(image_info, panels, title):
    rgb = read_images([image_info])[0]
    h, w = rgb.shape[:2]
    anns = anns_by_image.get(int(image_info['id']), [])
    small = Image.fromarray(rgb)
    small.thumbnail((1280, 1280))
    buffer = io.BytesIO()
    small.save(buffer, 'JPEG', quality=85)
    source = 'data:image/jpeg;base64,' + base64.b64encode(buffer.getvalue()).decode()
    fig = make_subplots(1, len(panels), horizontal_spacing=.01, subplot_titles=[
        f"{name}: {int((pred[1] >= SHOW).sum())} boxes ≥ {SHOW}" for name, pred in panels.items()])
    def outline(boxes, color, text=None, name=''):
        xs, ys, hover = [], [], []
        for k, (x1, y1, x2, y2) in enumerate(boxes):
            xs += [x1, x2, x2, x1, x1, None]
            ys += [y1, y1, y2, y2, y1, None]
            hover += [text[k] if text else name] * 5 + [None]
        return go.Scatter(x=xs, y=ys, mode='lines', line=dict(color=color, width=1.1), text=hover,
                          hoverinfo='text', showlegend=False)
    for col, (name, (boxes, scores, labels)) in enumerate(panels.items(), 1):
        keep = scores >= SHOW
        fig.add_trace(outline([xyxy(annotation) for annotation in anns if not annotation.get('iscrowd', 0)],
                              'lime', name='ground truth'), 1, col)
        fig.add_trace(outline([xyxy(annotation) for annotation in anns if annotation.get('iscrowd', 0)],
                              'yellow', name='ignored region'), 1, col)
        fig.add_trace(outline(
            boxes[keep].tolist(), 'red',
            [f"{CATS[LABEL_TO_CAT[int(x)]]} {y:.2f}" for x, y in zip(labels[keep], scores[keep])]), 1, col)
        fig.add_layout_image(source=source, xref=f'x{col}', yref=f'y{col}', x=0, y=0, sizex=w, sizey=h,
                             xanchor='left', yanchor='top', sizing='stretch', layer='below')
    fig.update_xaxes(range=[0, w], visible=False)
    fig.update_yaxes(range=[h, 0], visible=False)
    fig.update_layout(title=title
                      + ' · green = ground truth, red = prediction (score ≥ 0.3), yellow = ignored',
                      width=1800, height=int(600 * h / w) + 130, margin=dict(l=10, r=10, t=100, b=10),
                      template='plotly_white')
    return fig

In [ ]:
error_values = errors.assign(tta_minus_base_tp=errors.tta_wbf_tp - errors.baseline_tp)
cases = [('representative_random', int(np.random.default_rng(SEED).integers(len(VAL_IDS))), 'random frame'),
         ('most_objects', int(error_values.baseline_gt.idxmax()), 'most annotated objects'),
         ('failure_most_missed', int(error_values.tta_wbf_fn.idxmax()),
          'failure: most missed objects with TTA + WBF'),
         ('failure_most_false_positives', int(error_values.tta_wbf_fp.idxmax()),
          'failure: most false positives with TTA + WBF'),
         ('failure_tta_worse_than_baseline', int(error_values.tta_minus_base_tp.idxmin()),
          'failure: TTA + WBF finds fewer objects than the baseline')]
saved = []
for label, idx, why in cases:
    image_info, record = val_images[idx], error_values.loc[idx]
    stats_text = ' | '.join(f"{CONFIGS_VAL[name].split('. ')[1]}: TP {record[name + '_tp']}, FP "
                            f"{record[name + '_fp']}, FN {record[name + '_fn']}"
                            for name in preds_rep)
    fig = val_figure(image_info, {CONFIGS_VAL[name]: preds_rep[name][idx] for name in preds_rep},
                     f"Official val, checkpoint fold {rep_fold}, {image_info['file_name']} "
                     f"({image_info['width']}×{image_info['height']}, {record.baseline_gt} objects) — "
                     f"{why}<br>{stats_text}")
    path = VAL_OUT / 'visualizations' / f"{label}_{image_info['id']}.html"
    fig.write_html(path, include_plotlyjs='directory')
    saved.append(path.name)
print('Saved to', VAL_OUT / 'visualizations', ':', saved)

,baseline_tp,baseline_fp,baseline_fn,tta_wbf_tp,tta_wbf_fp,tta_wbf_fn
"fold 4, all frames, score ≥ 0.3",65963,43745,48169,70904,46221,43228


Saved to E:\ucu_hw2\faster_rcnn\official_validation\visualizations : ['representative_random_2421.html', 'most_objects_955.html', 'failure_most_missed_768.html', 'failure_most_false_positives_1272.html', 'failure_tta_worse_than_baseline_321.html']
